# EVAC-TWIN
## Complete Project Architecture, Connectivity, Training and Execution Notebook

**Project title:** Digital Twin-Based Intelligent Emergency Response and Evacuation System

**Purpose of this notebook:** explain the complete project structure, data flow, model-training workflow, deployment flow, API connectivity, route planning and frontend integration. It also embeds the current source files found in the project folder for review.

> **Important:** This notebook is a technical walkthrough and source-code companion. It does not automatically start MongoDB, Mosquitto, FastAPI, the simulator or the React server. It does not retrain models automatically. Long-running services and training jobs must be started deliberately in separate terminals or explicitly selected cells.

## 1. How to use this notebook

1. Read the architecture and execution-order sections first.
2. Confirm the project root and environment paths.
3. Start MongoDB and Mosquitto outside Jupyter.
4. Start the FastAPI backend, then the IoT simulator, then the React frontend in separate terminals.
5. Use the optional diagnostic cells only when the relevant service is already running.
6. Read the source-reference sections to inspect the actual implementation.
7. Training scripts are included as source references. Run them from the backend package context only when intentionally retraining.

Cells marked **SOURCE REFERENCE** contain original Python or JavaScript source. They are not ordinary notebook exercises and should not be run as Python cells. JavaScript/JSX is shown as fenced text in Markdown cells.

## 2. Project architecture at a glance

```text
                         BUILDING ALPHA DIGITAL TWIN
                 Static topology + dynamic virtual sensor state
                                  |
                                  v
                      Virtual IoT Sensor Simulator
                         (12 zones / 3 floors)
                                  |
                       MQTT publish: JSON messages
                                  |
                                  v
                    Mosquitto Broker (localhost:1883)
                     Topic: evac-twin/sensors
                                  |
                                  v
                    FastAPI MQTT Subscriber
                    (backend/app/mqtt_subscriber.py)
                       |          |          |
                       |          |          +--> Emergency alert / SMS
                       |          |
                       |          +--> FireModel (6 sensor features)
                       |
                       +--> RiskNet (4 sensor features)
                                  |
                                  v
                       MongoDB: evac_twin
                  sensors / events / buildings / contacts
                                  |
                    +-------------+--------------+
                    |                            |
                    v                            v
             FastAPI REST API              Routing services
             (port 8001)                   Dijkstra / BFS / DFS / A*
                    |                            |
                    +-------------+--------------+
                                  |
                                  v
                         React + Vite Frontend
                            (port 5173)
                    Dashboard / Monitoring / Evacuation
                    Simulation / AI Insights / Analytics
```

### Main connectivity chain

**Sensor → MQTT → Subscriber → AI prediction → MongoDB → FastAPI endpoints → frontend API service → React pages.**

For evacuation, the backend combines the static building graph with current zone-risk data. The production route calculation uses Dijkstra with risk-aware edge costs. BFS, DFS and A* are available for algorithm comparison.

## 3. Main project folders

The generator discovers files from the current project folder. The expected layout is broadly:

```text
EVAC-TWIN/
├── backend/
│   ├── app/
│   │   ├── main.py
│   │   ├── database.py
│   │   ├── building_layout.py
│   │   ├── building_structure.py
│   │   ├── iot_simulator.py
│   │   ├── mqtt_subscriber.py
│   │   ├── risk_engine.py
│   │   ├── ml/
│   │   │   ├── predict.py
│   │   │   ├── fire_predict.py
│   │   │   ├── train_risk_model.py
│   │   │   ├── train_fire_model.py
│   │   │   ├── compare_models.py
│   │   │   └── compare_dl_models.py
│   │   ├── routing/
│   │   └── services/
│   └── requirements.txt
├── frontend/
│   ├── index.html
│   ├── package.json
│   └── src/
│       ├── main.jsx
│       ├── App.jsx
│       ├── services/api.js
│       └── pages/
├── datasets/
├── models/
├── run_evactwin.bat
└── create_evactwin_flow_notebook.py
```

Actual filenames can differ. The source inventory near the end lists the files that were found when this notebook was generated.

## 4. Component responsibilities

| Component | Responsibility | Connects to |
|---|---|---|
| `building_layout.py` | Building Alpha rooms, corridors, stairs, exits and graph edges | Routing service |
| `iot_simulator.py` | Generates virtual sensor readings for zones and publishes JSON | Mosquitto |
| Mosquitto | MQTT message broker | Simulator and subscriber |
| `mqtt_subscriber.py` | Receives readings, invokes predictions, handles events/alerts and stores readings | MQTT, models, MongoDB, routing, SMS |
| `predict.py` | Four-feature multi-class building risk prediction | Risk model artifacts |
| `fire_predict.py` | Six-feature binary fire-alarm prediction | Fire model artifacts |
| `database.py` | MongoDB client, collections and indexes | MongoDB |
| `sensor_service.py` | Retrieves/enriches latest zone readings and history | MongoDB, risk prediction |
| `routing_service.py` | Risk-aware graph route calculation and algorithm comparisons | Building topology, sensor risk |
| `evacuation_router.py` | REST endpoints for routes, exits, summaries and simulations | Routing and sensor services |
| `main.py` | FastAPI app, API endpoints and MQTT subscriber lifecycle | All backend services |
| `frontend/src/services/api.js` | HTTP communication from React to FastAPI | FastAPI |
| React pages | Present monitoring, insights, evacuation, simulation and analytics | API service |

`building_structure.py` is a compatibility module that imports the topology from `building_layout.py`; it intentionally does not define a second building graph.

## 5. Digital Twin representation

The Building Alpha representation has two complementary parts:

- **Static building model:** room/zone identifiers, floors, corridors, stairs, exits and graph connectivity are defined in `building_layout.py`.
- **Dynamic building state:** virtual IoT readings (temperature, humidity, smoke, occupancy and fire-related air-quality features) update over MQTT and are stored in MongoDB.

The routing engine combines the static graph with current sensor-derived risk values. Therefore, the route can change as zone conditions change. The simulator is a virtual sensor source; it does not mean physical sensors are installed in the building.

## 6. Risk and fire prediction models

### 6.1 Overall zone-risk model

The deployed risk model uses four inputs:

1. `temperature`
2. `humidity`
3. `smoke`
4. `occupancy`

The supplied `risk_config.json` describes an MLP with input size 4, hidden sizes `[64, 32]`, and four output classes: `CRITICAL`, `HIGH`, `LOW`, `MODERATE`.

The deployed prediction code loads the trained model/scaler artifacts, predicts class probabilities and combines the model result with a rule-based safety check. If model artifacts cannot be loaded, the code has a fallback prediction path. The exact runtime result depends on the artifacts present in the local project.

### 6.2 Fire-alarm model

The separate fire model uses six features:

- `Temperature[C]`
- `Humidity[%]`
- `TVOC[ppb]`
- `eCO2[ppm]`
- `PM1.0`
- `PM2.5`

The supplied fire configuration contains calibration temperature `0.8120764493942261` and threshold `0.5`. The subscriber also applies its own final probability gate (80% in the reviewed implementation) before treating a prediction as a final fire alarm. This is distinct from the four-feature zone-risk classifier.

### 6.3 Model training, comparison and deployment are different

| Workflow | Purpose | Main files |
|---|---|---|
| Train risk model | Train the multi-class risk classifier | `train_risk_model.py` (and any older/alternate `train_model.py`) |
| Train fire model | Train the binary fire classifier | `train_fire_model.py` |
| Compare models | Compare candidate ML/DL models on the fire dataset | `compare_models.py`, `compare_dl_models.py` |
| Deploy predictions | Load saved model artifacts for live readings | `predict.py`, `fire_predict.py` |

A model with the best comparison score is **not automatically deployed**. Deployment changes only when the training/deployment code and compatible saved artifacts are deliberately updated.

`compare_models.py` compares Random Forest, Logistic Regression, KNN, SVM, MLP, TabNet, FT-Transformer and SAINT-style Transformer. `compare_dl_models.py` trains/evaluates TabNet separately. These comparison scripts are offline experiments, not part of the per-message MQTT prediction path.

## 7. Sensor message flow in detail

1. The simulator creates a payload for a zone. The reviewed simulator generates readings for 12 zones across three floors and publishes repeatedly (default interval: 3 seconds).
2. The payload is published to the MQTT topic `evac-twin/sensors` on `localhost:1883`.
3. FastAPI startup starts the MQTT subscriber. The subscriber receives and decodes each JSON message.
4. It normalizes sensor fields and invokes the four-feature risk predictor.
5. When all six fire-model features are available, it invokes the separate fire predictor.
6. It creates or updates emergency/risk events and applies duplicate-alert protection.
7. For qualifying emergencies, it can calculate evacuation routes and attempt to send an SMS alert.
8. The processed sensor record is inserted into MongoDB.
9. REST endpoints expose the latest sensor intelligence, history, events, building summary and evacuation information.
10. React pages poll the backend APIs and render the current state.

## 8. MongoDB data layer

Reviewed default settings:

- MongoDB URI: `mongodb://127.0.0.1:27017`
- Database: `evac_twin`
- Collections: `sensors`, `events`, `buildings`, `emergency_contacts`

The application code creates indexes when the database module is imported. A MongoDB client object being constructed does not, by itself, prove that the server is reachable. Use the explicit ping/health diagnostic after MongoDB is running.

The notebook does not create or delete database data automatically.

## 9. Risk-aware evacuation routing

The building graph is constructed from the topology in `building_layout.py`. The routing service applies risk-dependent costs to graph nodes/edges. Critical-risk rooms can be blocked as route destinations (except the starting room handling), while high/moderate/unknown-risk areas receive penalties or caution treatment.

- **Production route:** `shortest_path()` uses Dijkstra with risk-aware costs.
- **BFS / DFS / A*:** used for comparison and analysis.
- **Route status:** AVAILABLE, CAUTION or BLOCKED.
- **Exit selection:** routes to the available exits are evaluated and a recommended exit is selected using route/risk scores.

The React frontend visualizes route data returned by the backend; it is not the source of the Dijkstra calculation.

## 10. Frontend connectivity

```text
frontend/index.html
        |
        v
frontend/src/main.jsx
        |
        v
frontend/src/App.jsx
        |
        +--> Dashboard
        +--> Monitoring
        +--> Evacuation
        +--> Simulation
        +--> AI Insights
        +--> Analytics
                    |
                    v
          frontend/src/services/api.js
                    |
          HTTP requests to localhost:8001
                    |
                    v
              FastAPI backend
```

`vite.config.js` enables the React plugin. The reviewed configuration does not define a development proxy; `api.js` uses the backend base URL directly. `index.html` provides the `root` element and loads `src/main.jsx`.

## 11. API endpoints to know

The reviewed backend exposes these main routes (confirm the current local code if it has changed):

| Method | Endpoint | Purpose |
|---|---|---|
| GET | `/` | Backend service information |
| GET | `/api/health` | Backend, database and subsystem health |
| GET | `/api/sensors` | Latest zone sensor intelligence |
| GET | `/api/sensors/history` | Sensor history |
| GET | `/api/sensors/{zone_id}` | Latest reading for a zone |
| POST | `/api/ai/predict-risk` | On-demand overall risk prediction |
| GET | `/api/events` | Recent/active events |
| GET | `/api/buildings` | Building topology/data |
| GET | `/api/buildings/summary` | Building-level risk summary |
| GET | `/api/evacuation/topology` | Building route topology |
| GET | `/api/evacuation/routes` | Route data |
| POST | `/api/evacuation/route/{device_id}` | Calculate a zone route |
| POST | `/api/evacuation/compare/{device_id}` | Compare route algorithms |
| GET | `/api/evacuation/summary` | Evacuation summary |
| GET | `/api/evacuation/exits` | Exit information |
| POST | `/api/evacuation/simulation/{device_id}` | Scenario simulation / alert workflow |

FastAPI Swagger documentation is normally available at `http://localhost:8001/docs` while the backend is running.

## 12. Environment and dependency setup

### 12.1 Find the existing virtual environment

The `.venv` location was reported as **Other / not one of the two standard locations**, so do not assume it is in the root or backend folder. In the next cell, inspect candidate locations and the active Python executable. This cell does not activate or modify an environment.

In [1]:
from pathlib import Path
import sys

root = Path.cwd()
print("Notebook working directory:", root)
print("Active Python:", sys.executable)

candidates = [
    root / ".venv",
    root / "backend" / ".venv",
    root.parent / ".venv",
]
print("\nCommon .venv candidates:")
for path in candidates:
    print(path, "->", "FOUND" if path.exists() else "not found")

print("\nIf your environment is elsewhere, use its actual Python executable when launching Jupyter/backend.")

Notebook working directory: C:\EVAC-TWIN
Active Python: C:\EVAC-TWIN\backend\.venv\Scripts\python.exe

Common .venv candidates:
C:\EVAC-TWIN\.venv -> not found
C:\EVAC-TWIN\backend\.venv -> FOUND
C:\.venv -> not found

If your environment is elsewhere, use its actual Python executable when launching Jupyter/backend.


### 12.2 Backend packages

The supplied `backend/requirements.txt` lists FastAPI, Uvicorn, PyMongo, dotenv, Paho MQTT, scikit-learn, NumPy, pandas, joblib and PyTorch.

Additional imports observed in the source need to be checked against the actual environment:
- `pytorch-tabnet` is required by the model-comparison scripts.
- `requests` may be required by the TextBee SMS service if it is imported by the current implementation.
- Plotting packages such as `matplotlib` (and `seaborn` only if imported) may be required by training/evaluation scripts.

Do not install blindly into a different Python environment. Install missing packages into the environment actually used by the project.

### 12.3 Frontend packages

The supplied `frontend/package.json` uses React, React DOM, React Router, Vite, the React Vite plugin, Recharts, Three.js-related packages, Lucide React and Oxlint. Install frontend packages from the frontend folder with `npm install` if `node_modules` is absent or dependencies need restoring.

## 13. Correct service startup order

Start these services in separate terminals. Keep each terminal open while the service is running.

### Step 1 — MongoDB

Start the MongoDB service using the installation/service configuration on the machine. The supplied `run_evactwin.bat` does not start MongoDB.

### Step 2 — Mosquitto broker

```powershell
mosquitto -v
```

Expected broker address: `localhost:1883`.

### Step 3 — FastAPI backend

Open a new terminal. Change to the backend folder and activate the **actual** virtual environment path. Example only, if `.venv` is in the project root:

```powershell
cd C:\EVAC-TWIN\backend
..\.venv\Scripts\Activate.ps1
python -m uvicorn app.main:app --reload --port 8001
```

If the environment is elsewhere, replace the activation path with the correct one. The backend startup imports the app and starts the MQTT subscriber through its FastAPI startup lifecycle.

### Step 4 — IoT simulator

Open another terminal:

```powershell
cd C:\EVAC-TWIN\backend
..\.venv\Scripts\Activate.ps1
python -m app.iot_simulator
```

The simulator is a separate process. Starting the backend alone does not publish simulated readings.

### Step 5 — React frontend

Open another terminal:

```powershell
cd C:\EVAC-TWIN\frontend
npm install
npm run dev
```

Open `http://localhost:5173`.

### Step 6 — Verify

- Backend root: `http://localhost:8001/`
- Backend health: `http://localhost:8001/api/health`
- Swagger UI: `http://localhost:8001/docs`
- Frontend: `http://localhost:5173`

The Vite port can change if 5173 is already occupied; use the URL printed by Vite in that case.

## 14. About the supplied `run_evactwin.bat`

The reviewed batch file starts Mosquitto, FastAPI, Vite and the simulator in separate command windows, then opens the frontend. It does **not** start MongoDB.

One path needs verification: the batch file changes directory to `C:\EVAC-TWIN\backend` and then runs `.\.venv\Scripts\activate`. If the environment is in the project root, that relative path is incorrect; from the backend folder the expected relative path is `..\.venv\Scripts\activate`. If the environment is elsewhere, use its actual path. The generator does not edit the batch file.

The launcher contains machine-specific absolute paths (`C:\EVAC-TWIN` and the Mosquitto installation path). Update those manually if the project is located elsewhere.

## 15. Safe diagnostic cells

These cells only inspect connectivity. They do not start or stop services and do not write project files.

In [2]:
# Check whether the backend HTTP service is already running.
# Run only after starting FastAPI in a separate terminal.
from urllib.request import urlopen
from urllib.error import URLError

url = "http://localhost:8001/api/health"
try:
    with urlopen(url, timeout=3) as response:
        print("HTTP status:", response.status)
        print(response.read().decode("utf-8"))
except (URLError, TimeoutError, OSError) as exc:
    print("Backend health endpoint is not reachable:", exc)
    print("Start the FastAPI backend in a separate terminal, then retry.")

Backend health endpoint is not reachable: <urlopen error [WinError 10061] No connection could be made because the target machine actively refused it>
Start the FastAPI backend in a separate terminal, then retry.


In [3]:
# Optional MongoDB ping. Requires pymongo in the active Jupyter environment
# and MongoDB already running locally.
try:
    from pymongo import MongoClient
    client = MongoClient("mongodb://127.0.0.1:27017", serverSelectionTimeoutMS=2500)
    client.admin.command("ping")
    print("MongoDB ping successful.")
    print("Databases:", client.list_database_names())
    client.close()
except Exception as exc:
    print("MongoDB ping failed or pymongo is unavailable:", repr(exc))

MongoDB ping successful.
Databases: ['admin', 'config', 'digital_twin_db', 'employee_db', 'evac_twin', 'local', 'product', 'studentdb']


In [4]:
# Optional MQTT broker TCP check. This checks the port only;
# it does not publish or subscribe to messages.
import socket

host, port = "127.0.0.1", 1883
sock = socket.socket()
sock.settimeout(2)
try:
    sock.connect((host, port))
    print(f"MQTT broker TCP port is reachable at {host}:{port}")
except OSError as exc:
    print("MQTT broker is not reachable:", exc)
finally:
    sock.close()

MQTT broker TCP port is reachable at 127.0.0.1:1883


## 16. Training workflow — run deliberately, not during normal startup

The project has separate scripts for risk-model training, fire-model training and model comparison. Training may take time and may overwrite model artifacts used by live prediction. Before training:

1. Confirm the correct dataset exists.
2. Confirm the Python environment has all required packages.
3. Back up existing model/scaler/config artifacts.
4. Confirm which risk training script is the current intended one (`train_risk_model.py` versus any older/alternate `train_model.py`).
5. Run one training script at a time from the correct project context.
6. Check its output metrics and generated artifacts.
7. Verify that deployment code expects the same model architecture, feature order, class order and artifact filenames.

### Example commands (PowerShell)

```powershell
cd C:\EVAC-TWIN\backend
# Activate the actual virtual environment first.

python -m app.ml.train_risk_model
python -m app.ml.train_fire_model
python -m app.ml.compare_models
python -m app.ml.compare_dl_models
```

These are examples based on module paths. Check each file's package location and `if __name__ == "__main__"` entry point before running. The comparison scripts are experiments; they are not required to start the live dashboard.

## 17. Important implementation notes and limitations

- The simulator publishes virtual readings; it is not a physical sensor network.
- MongoDB must be running separately; the provided batch launcher does not start it.
- MQTT broker must be running before the subscriber can receive simulator messages.
- The backend starts the MQTT subscriber during FastAPI startup; the simulator is started separately.
- RiskNet and FireModel are separate models with different feature sets and targets.
- The fire subscriber's final alarm decision includes an additional probability gate.
- The frontend displays backend predictions; frontend rule-based calculations are fallback/preview logic and should not be confused with the deployed PyTorch model.
- The frontend simulation preview, backend on-demand prediction and backend scenario-alert calculation are distinct paths.
- Analytics that fetches `/api/sensors` works on the latest per-zone readings returned by that endpoint; it is not automatically the full historical MongoDB dataset.
- The building topology is defined in `building_layout.py`; `building_structure.py` re-exports it.
- A successful SMS API request does not independently prove that the recipient's phone received the message.
- Model comparison results depend on the dataset, preprocessing, random split, environment and package versions. Do not state performance values unless they are taken from the actual run output.

## 18. Source code reference library

The following sections embed project source files that exist at notebook-generation time. Python files are included as unexecuted reference cells. JavaScript/JSX, JSON, CSS, HTML and batch files are shown as fenced text in Markdown so they are not mistakenly executed by the Python kernel.

**Long files are split into sequential chunks; all lines are retained.** Re-run the generator after source-code changes to refresh the notebook.

In [5]:
import os
import sys

PROJECT_PATH = r"C:\EVAC-TWIN"
BACKEND_PATH = os.path.join(PROJECT_PATH, "backend")

if BACKEND_PATH not in sys.path:
    sys.path.insert(0, BACKEND_PATH)

os.chdir(PROJECT_PATH)

print("Project path:", PROJECT_PATH)
print("Backend path:", BACKEND_PATH)
print("App folder exists:", os.path.isdir(os.path.join(BACKEND_PATH, "app")))

Project path: C:\EVAC-TWIN
Backend path: C:\EVAC-TWIN\backend
App folder exists: True


### backend/app/main.py

**File:** `backend/app/main.py`

This is the actual project source copied into the notebook for inspection.
It is a reference section, not an instruction to execute the entire file in a notebook cell.

In [6]:
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from app.ml.predict import predict_risk
from pydantic import BaseModel

from app.database import (
    buildings_collection,
    events_collection,
    check_database_connection,
)
from app.building_layout import BUILDING_ID, ROOMS, CORRIDORS, STAIRS, EXITS
from app.mqtt_subscriber import start_mqtt_subscriber
from app.routing.evacuation_router import router as evacuation_router
from app.services.sensor_service import latest_zone_intelligence, sensor_history

app = FastAPI(
    title="EVAC-TWIN",
    description="Emergency Intelligence Digital Twin",
    version="2.0.0",
)
class RiskPredictionInput(BaseModel):
    temperature: float
    humidity: float
    smoke: float
    occupancy: int
app.add_middleware(
    CORSMiddleware,
    allow_origins=["http://localhost:5173", "http://127.0.0.1:5173"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

app.include_router(evacuation_router)


@app.get("/")
def root():
    return {
        "project": "EVAC-TWIN",
        "service": "Emergency Intelligence Digital Twin",
        "motto": "Sense → Predict → Understand → Simulate → Route → Respond",
        "status": "online",
        "backend": "FastAPI",
        "routing": "dynamic",
        "deep_learning": "PyTorch MLP",
        "mqtt": "enabled",
    }


@app.get("/api/health")
def health():
    return {
        "status": "healthy",
        "service": "EVAC-TWIN Backend",
        "database": "connected" if check_database_connection() else "disconnected",
        "mqtt": "enabled",
        "deep_learning": "enabled",
        "routing": "dynamic",
        "topology": "Building Alpha",
    }


@app.get("/api/sensors")
def get_sensors():
    return latest_zone_intelligence()


@app.get("/api/sensors/history")
def get_sensor_history(zone_id: str | None = None, limit: int = 100):
    return sensor_history(zone_id, limit)


@app.get("/api/sensors/{zone_id}")
def get_sensor(zone_id: str):
    sensors = latest_zone_intelligence()
    sensor = next((s for s in sensors if str(s["zone_id"]) == str(zone_id).removeprefix("sensor-")), None)
    if sensor is None:
        from fastapi import HTTPException
        raise HTTPException(status_code=404, detail="Zone sensor not found")
    return sensor
@app.post("/api/ai/predict-risk")
def predict_risk_api(data: RiskPredictionInput):
    sensor_data = {
        "temperature": data.temperature,
        "humidity": data.humidity,
        "smoke": data.smoke,
        "occupancy": data.occupancy,
    }

    result = predict_risk(sensor_data)

    return {
        "model": "PyTorch Deep Learning",
        "prediction": result.get("level"),
        "risk_score": result.get("score"),
        "confidence": result.get("confidence"),
        "probabilities": result.get("probabilities"),
        "recommendation": result.get("recommendation"),
    }


@app.get("/api/events")
def get_events():
    active_events = list(
        events_collection.find(
            {"status": "ACTIVE"},
            {"_id": 0}
        ).sort("timestamp", -1).limit(100)
    )

    return active_events


@app.get("/api/buildings")
def get_buildings():
    buildings = list(buildings_collection.find({}, {"_id": 0}))
    if not buildings:
        return [{
            "building_id": BUILDING_ID,
            "name": "Building Alpha",
            "floors": 3,
            "zones": len(ROOMS),
            "staircases": len(STAIRS),
            "exits": len(EXITS),
        }]
    return buildings


@app.get("/api/buildings/summary")
def get_building_summary():
    sensors = latest_zone_intelligence()
    distribution = {"low": 0, "moderate": 0, "high": 0, "critical": 0}
    total_occupancy = 0
    for sensor in sensors:
        level = str(sensor.get("risk_level", "LOW")).lower()
        if level in distribution:
            distribution[level] += 1
        total_occupancy += int(sensor.get("occupancy", 0) or 0)

    if distribution["critical"]:
        overall = "CRITICAL"
    elif distribution["high"]:
        overall = "HIGH"
    elif distribution["moderate"]:
        overall = "MODERATE"
    else:
        overall = "LOW"

    return {
        "building_id": BUILDING_ID,
        "total_zones": len(sensors),
        "overall_risk": overall,
        "risk_distribution": distribution,
        "total_occupancy": total_occupancy,
    }


mqtt_client = None


@app.on_event("startup")
def startup_event():
    global mqtt_client
    print("=" * 60)
    print("EVAC-TWIN BACKEND — ONLINE")
    print("=" * 60)
    try:
        mqtt_client = start_mqtt_subscriber()
        print("MQTT Subscriber : STARTED")
    except Exception as error:
        print(f"MQTT Subscriber : FAILED - {error}")


@app.on_event("shutdown")
def shutdown_event():
    global mqtt_client
    if mqtt_client:
        try:
            mqtt_client.loop_stop()
            mqtt_client.disconnect()
        except Exception:
            pass


[DL] Risk model loaded successfully
[DL] Device: cpu
[DL] Classes: ['CRITICAL', 'HIGH', 'LOW', 'MODERATE']
[DL] Fire Detection model loaded successfully
[DL] Device: cpu
[DL] Threshold: 0.5
[DL] Calibration temperature: 0.8120764493942261


C:\Users\bharg\AppData\Local\Temp\ipykernel_20272\2239097797.py:162: DeprecationWarning: 
        on_event is deprecated, use lifespan event handlers instead.

        Read more about it in the
        [FastAPI docs for Lifespan Events](https://fastapi.tiangolo.com/advanced/events/).
        
  @app.on_event("startup")
C:\Users\bharg\AppData\Local\Temp\ipykernel_20272\2239097797.py:175: DeprecationWarning: 
        on_event is deprecated, use lifespan event handlers instead.

        Read more about it in the
        [FastAPI docs for Lifespan Events](https://fastapi.tiangolo.com/advanced/events/).
        
  @app.on_event("shutdown")


### backend/app/database.py

**File:** `backend/app/database.py`

This is the actual project source copied into the notebook for inspection.
It is a reference section, not an instruction to execute the entire file in a notebook cell.

In [7]:
from pymongo import MongoClient, ASCENDING, DESCENDING
import os

from dotenv import load_dotenv


# ============================================================
# EVAC-TWIN — DATABASE CONFIGURATION
# ============================================================

load_dotenv()


MONGO_URL = os.getenv(
    "MONGO_URL",
    "mongodb://127.0.0.1:27017"
)

DATABASE_NAME = os.getenv(
    "MONGO_DATABASE",
    "evac_twin"
)


# ============================================================
# MONGODB CONNECTION
# ============================================================

client = MongoClient(
    MONGO_URL,
    serverSelectionTimeoutMS=5000
)

db = client[DATABASE_NAME]


# ============================================================
# COLLECTIONS
# ============================================================

sensors_collection = db["sensors"]
events_collection = db["events"]
buildings_collection = db["buildings"]
emergency_contacts_collection = db["emergency_contacts"]


# ============================================================
# INDEXES
# ============================================================

def initialize_indexes():

    # --------------------------------------------------------
    # SENSOR HISTORY
    # --------------------------------------------------------

    sensors_collection.create_index(
        [
            ("device_id", ASCENDING),
            ("timestamp", DESCENDING)
        ],
        name="sensor_history"
    )

    sensors_collection.create_index(
        [
            ("building_id", ASCENDING),
            ("zone_id", ASCENDING),
            ("timestamp", DESCENDING)
        ],
        name="zone_sensor_history"
    )
        # --------------------------------------------------------
    # EMERGENCY CONTACTS
    # --------------------------------------------------------

    emergency_contacts_collection.create_index(
        [
            ("building_id", ASCENDING),
            ("phone", ASCENDING)
        ],
        unique=True,
        name="emergency_contact_unique"
    )

    emergency_contacts_collection.create_index(
        [
            ("building_id", ASCENDING),
            ("floor", ASCENDING),
            ("zone_id", ASCENDING)
        ],
        name="contact_location"
    )

    # --------------------------------------------------------
    # EVENTS
    # --------------------------------------------------------

    events_collection.create_index(
        [
            ("building_id", ASCENDING),
            ("zone_id", ASCENDING),
            ("timestamp", DESCENDING)
        ],
        name="event_history"
    )

    events_collection.create_index(
        [
            ("risk_level", ASCENDING),
            ("timestamp", DESCENDING)
        ],
        name="risk_event_history"
    )

    # --------------------------------------------------------
    # BUILDINGS
    # --------------------------------------------------------

    buildings_collection.create_index(
        [
            ("building_id", ASCENDING)
        ],
        unique=True,
        name="building_id_unique"
    )


# ============================================================
# DATABASE HEALTH CHECK
# ============================================================

def check_database_connection():

    try:

        client.admin.command(
            "ping"
        )

        return True

    except Exception as error:

        print(
            f"MongoDB connection failed: {error}"
        )

        return False


# ============================================================
# INITIALIZE DATABASE
# ============================================================

initialize_indexes()

### backend/app/building_layout.py

**File:** `backend/app/building_layout.py`

This is the actual project source copied into the notebook for inspection.
It is a reference section, not an instruction to execute the entire file in a notebook cell.

> This file has 334 lines and is split into numbered reference sections of at most 220 lines. All lines are included.

#### backend/app/building_layout.py — lines 1–220

In [8]:
# ============================================================
# EVAC-TWIN — BUILDING ALPHA TOPOLOGY
# ============================================================

BUILDING_ID = "building-alpha"


# ============================================================
# ROOMS
# ============================================================

ROOMS = {
    "101": {"floor": 1, "name": "Room 101"},
    "102": {"floor": 1, "name": "Room 102"},
    "103": {"floor": 1, "name": "Room 103"},
    "104": {"floor": 1, "name": "Room 104"},

    "201": {"floor": 2, "name": "Room 201"},
    "202": {"floor": 2, "name": "Room 202"},
    "203": {"floor": 2, "name": "Room 203"},
    "204": {"floor": 2, "name": "Room 204"},

    "301": {"floor": 3, "name": "Room 301"},
    "302": {"floor": 3, "name": "Room 302"},
    "303": {"floor": 3, "name": "Room 303"},
    "304": {"floor": 3, "name": "Room 304"},
}


# ============================================================
# FLOOR CORRIDOR SECTIONS
#
# The corridor is divided into three sections so that
# different hazards can affect different parts of the route.
#
# LEFT  -> rooms 101/201/301
# CENTER -> rooms 102/202/302 and 103/203/303
# RIGHT -> rooms 104/204/304
# ============================================================

CORRIDORS = {

    "CORRIDOR_1_LEFT": {
        "floor": 1,
        "adjacent_zones": ["101", "102"],
    },

    "CORRIDOR_1_CENTER": {
        "floor": 1,
        "adjacent_zones": ["102", "103"],
    },

    "CORRIDOR_1_RIGHT": {
        "floor": 1,
        "adjacent_zones": ["103", "104"],
    },

    "CORRIDOR_2_LEFT": {
        "floor": 2,
        "adjacent_zones": ["201", "202"],
    },

    "CORRIDOR_2_CENTER": {
        "floor": 2,
        "adjacent_zones": ["202", "203"],
    },

    "CORRIDOR_2_RIGHT": {
        "floor": 2,
        "adjacent_zones": ["203", "204"],
    },

    "CORRIDOR_3_LEFT": {
        "floor": 3,
        "adjacent_zones": ["301", "302"],
    },

    "CORRIDOR_3_CENTER": {
        "floor": 3,
        "adjacent_zones": ["302", "303"],
    },

    "CORRIDOR_3_RIGHT": {
        "floor": 3,
        "adjacent_zones": ["303", "304"],
    },
}


# ============================================================
# STAIRS
# ============================================================

STAIRS = {

    "STAIR_A": {
        "name": "Stair A",
        "exit": "EXIT A",

        "floor_connections": {
            1: "STAIR_A_1",
            2: "STAIR_A_2",
            3: "STAIR_A_3",
        },

        "connected_zones": {
            1: ["101", "102"],
            2: ["201", "202"],
            3: ["301", "302"],
        },
    },

    "STAIR_B": {
        "name": "Stair B",
        "exit": "EXIT B",

        "floor_connections": {
            1: "STAIR_B_1",
            2: "STAIR_B_2",
            3: "STAIR_B_3",
        },

        "connected_zones": {
            1: ["102", "103"],
            2: ["202", "203"],
            3: ["302", "303"],
        },
    },

    "STAIR_C": {
        "name": "Stair C",
        "exit": "EXIT C",

        "floor_connections": {
            1: "STAIR_C_1",
            2: "STAIR_C_2",
            3: "STAIR_C_3",
        },

        "connected_zones": {
            1: ["103", "104"],
            2: ["203", "204"],
            3: ["303", "304"],
        },
    },
}


# ============================================================
# EXITS
# ============================================================

EXITS = {

    "EXIT A": {
        "name": "Primary Emergency Exit",
        "stair": "STAIR_A",
        "base_distance": 1.0,
    },

    "EXIT B": {
        "name": "Secondary Emergency Exit",
        "stair": "STAIR_B",
        "base_distance": 1.2,
    },

    "EXIT C": {
        "name": "Rear Emergency Exit",
        "stair": "STAIR_C",
        "base_distance": 1.5,
    },
}


# ============================================================
# GRAPH EDGES
#
# Each edge represents a physically possible movement.
# ============================================================

EDGES = [

    # --------------------------------------------------------
    # FLOOR 1
    # --------------------------------------------------------

    ("101", "CORRIDOR_1_LEFT"),
    ("102", "CORRIDOR_1_LEFT"),

    ("102", "CORRIDOR_1_CENTER"),
    ("103", "CORRIDOR_1_CENTER"),

    ("103", "CORRIDOR_1_RIGHT"),
    ("104", "CORRIDOR_1_RIGHT"),

    ("CORRIDOR_1_LEFT", "CORRIDOR_1_CENTER"),
    ("CORRIDOR_1_CENTER", "CORRIDOR_1_RIGHT"),

    # Stair access
    ("CORRIDOR_1_LEFT", "STAIR_A_1"),
    ("CORRIDOR_1_CENTER", "STAIR_B_1"),
    ("CORRIDOR_1_RIGHT", "STAIR_C_1"),

    # --------------------------------------------------------
    # FLOOR 2
    # --------------------------------------------------------

    ("201", "CORRIDOR_2_LEFT"),
    ("202", "CORRIDOR_2_LEFT"),

    ("202", "CORRIDOR_2_CENTER"),
    ("203", "CORRIDOR_2_CENTER"),

    ("203", "CORRIDOR_2_RIGHT"),
    ("204", "CORRIDOR_2_RIGHT"),

    ("CORRIDOR_2_LEFT", "CORRIDOR_2_CENTER"),
    ("CORRIDOR_2_CENTER", "CORRIDOR_2_RIGHT"),

    ("CORRIDOR_2_LEFT", "STAIR_A_2"),
        ("CORRIDOR_2_CENTER", "STAIR_B_2"),
    ("CORRIDOR_2_RIGHT", "STAIR_C_2"),

    # --------------------------------------------------------
    # FLOOR 3
    # --------------------------------------------------------

    ("301", "CORRIDOR_3_LEFT"),
    ("302", "CORRIDOR_3_LEFT"),

    ("302", "CORRIDOR_3_CENTER"),
    ("303", "CORRIDOR_3_CENTER"),

    ("303", "CORRIDOR_3_RIGHT"),
    ("304", "CORRIDOR_3_RIGHT"),

    ("CORRIDOR_3_LEFT", "CORRIDOR_3_CENTER"),
    ("CORRIDOR_3_CENTER", "CORRIDOR_3_RIGHT"),

    ("CORRIDOR_3_LEFT", "STAIR_A_3"),
    ("CORRIDOR_3_CENTER", "STAIR_B_3"),
    ("CORRIDOR_3_RIGHT", "STAIR_C_3"),

    # --------------------------------------------------------
    # VERTICAL STAIR CONNECTIONS
    # --------------------------------------------------------

    ("STAIR_A_1", "STAIR_A_2"),
    ("STAIR_A_2", "STAIR_A_3"),

    ("STAIR_B_1", "STAIR_B_2"),
    ("STAIR_B_2", "STAIR_B_3"),

    ("STAIR_C_1", "STAIR_C_2"),
    ("STAIR_C_2", "STAIR_C_3"),
]


# ============================================================
# EXIT NODES
# ============================================================

EXIT_NODES = {

    "EXIT A": "EXIT_A_NODE",
    "EXIT B": "EXIT_B_NODE",
    "EXIT C": "EXIT_C_NODE",
}


EDGES += [

    ("STAIR_A_1", "EXIT_A_NODE"),
    ("STAIR_B_1", "EXIT_B_NODE"),
    ("STAIR_C_1", "EXIT_C_NODE"),
]


# ============================================================
# HELPERS
# ============================================================

def get_floor(zone_id):

    zone_id = str(zone_id)

    if zone_id in ROOMS:
        return ROOMS[zone_id]["floor"]

    return 0


def get_exit_stair(exit_name):

    exit_data = EXITS.get(exit_name)

    if not exit_data:
        return None

    return exit_data["stair"]


def get_stair_floor_node(stair_name, floor):

    stair = STAIRS.get(stair_name)

    if not stair:
        return None

    return stair["floor_connections"].get(floor)


def get_room_stairs(zone_id):

    zone_id = str(zone_id)

    floor = get_floor(zone_id)

    if floor == 0:
        return []

    result = []

    for stair_name, stair in STAIRS.items():

        zones = stair["connected_zones"].get(
            floor,
            []
        )

        if zone_id in zones:
            result.append(stair_name)

    return result


### backend/app/building_structure.py

**File:** `backend/app/building_structure.py`

This is the actual project source copied into the notebook for inspection.
It is a reference section, not an instruction to execute the entire file in a notebook cell.

In [9]:
"""Compatibility module.

Building Alpha's single source of truth is app.building_layout.
This module intentionally contains no second topology definition.
"""

from app.building_layout import BUILDING_ID, ROOMS, CORRIDORS, STAIRS, EXITS, EDGES


### backend/app/iot_simulator.py

**File:** `backend/app/iot_simulator.py`

This is the actual project source copied into the notebook for inspection.
It is a reference section, not an instruction to execute the entire file in a notebook cell.

> This file has 699 lines and is split into numbered reference sections of at most 220 lines. All lines are included.

In [10]:
import argparse
import json
import os
import random
import time
from datetime import datetime, timezone

import paho.mqtt.client as mqtt


# ============================================================
# EVAC-TWIN — VIRTUAL IoT SENSOR SIMULATOR
# ============================================================

MQTT_BROKER = os.getenv("MQTT_BROKER", "localhost")
MQTT_PORT = int(os.getenv("MQTT_PORT", "1883"))
MQTT_TOPIC = os.getenv("MQTT_TOPIC", "evac-twin/sensors")
PUBLISH_INTERVAL = float(os.getenv("PUBLISH_INTERVAL", "3"))


# ============================================================
# BUILDING CONFIGURATION
# ============================================================

BUILDING_ID = "building-alpha"


ZONES = [
    # Floor 1
    {"zone_id": "101", "floor": 1, "base_occupancy": 6,
     "connected_to": ["102"]},
    {"zone_id": "102", "floor": 1, "base_occupancy": 7,
     "connected_to": ["101", "103"]},
    {"zone_id": "103", "floor": 1, "base_occupancy": 5,
     "connected_to": ["102", "104"]},
    {"zone_id": "104", "floor": 1, "base_occupancy": 4,
     "connected_to": ["103"]},

    # Floor 2
    {"zone_id": "201", "floor": 2, "base_occupancy": 8,
     "connected_to": ["202"]},
    {"zone_id": "202", "floor": 2, "base_occupancy": 7,
     "connected_to": ["201", "203"]},
    {"zone_id": "203", "floor": 2, "base_occupancy": 6,
     "connected_to": ["202", "204"]},
    {"zone_id": "204", "floor": 2, "base_occupancy": 5,
     "connected_to": ["203"]},

    # Floor 3
    {"zone_id": "301", "floor": 3, "base_occupancy": 7,
     "connected_to": ["302"]},
    {"zone_id": "302", "floor": 3, "base_occupancy": 6,
     "connected_to": ["301", "303"]},
    {"zone_id": "303", "floor": 3, "base_occupancy": 5,
     "connected_to": ["302", "304"]},
    {"zone_id": "304", "floor": 3, "base_occupancy": 4,
     "connected_to": ["303"]},
]



# ============================================================
# INITIAL SENSOR STATE
# ============================================================


sensor_state = {}

for zone in ZONES:
    sensor_state[zone["zone_id"]] = {
        "temperature": random.uniform(29.0, 31.0),
        "humidity": random.uniform(48.0, 58.0),
        "smoke": random.uniform(0.05, 0.15),
        "occupancy": zone["base_occupancy"],
        "tvoc": random.uniform(50, 300),
        "eco2": random.uniform(400, 600),
        "pm1": random.uniform(1.0, 3.0),
        "pm25": random.uniform(1.0, 4.0),
    }


# ============================================================
# SIMULATION MODE  — set via CLI args or defaults
# ============================================================
#
# Usage:
#   python -m app.iot_simulator                     # NORMAL mode, random zone
#   python -m app.iot_simulator --mode FIRE          # FIRE in random zone
#   python -m app.iot_simulator --mode FIRE --zone 304
#   python -m app.iot_simulator --mode SMOKE --zone 201
#

_parser = argparse.ArgumentParser(
    description="EVAC-TWIN Virtual IoT Sensor Simulator"
)
_parser.add_argument(
    "--mode",
    default=os.getenv("SIMULATION_MODE", "FIRE"),
    choices=["NORMAL", "FIRE", "SMOKE", "CROWD"],
    help="Simulation scenario mode",
)
_parser.add_argument(
    "--zone",
    default=os.getenv("EMERGENCY_ZONE", ""),
    help="Emergency zone ID (e.g. 304). Leave blank for random selection.",
)
_args, _unknown = _parser.parse_known_args()

SIMULATION_MODE: str = _args.mode.upper()

# Resolve emergency zone: use CLI arg, or pick a random zone.
_zone_arg = str(_args.zone).strip()
if _zone_arg and _zone_arg in [z["zone_id"] for z in ZONES]:
    EMERGENCY_ZONE: str = _zone_arg
elif SIMULATION_MODE != "NORMAL":
    EMERGENCY_ZONE = random.choice(
        [z["zone_id"] for z in ZONES]
    )
else:
    EMERGENCY_ZONE = ""


# ============================================================
# MQTT CONNECTION
# ============================================================

client = mqtt.Client(
    mqtt.CallbackAPIVersion.VERSION2
)


def on_connect(client, userdata, flags, reason_code, properties):

    if reason_code == 0:
        print()
        print("=" * 60)
        print("EVAC-TWIN IoT SIMULATOR")
        print("=" * 60)
        print("Connected to MQTT broker")
        print(f"Broker : {MQTT_BROKER}:{MQTT_PORT}")
        print(f"Topic  : {MQTT_TOPIC}")
        print(f"Mode   : {SIMULATION_MODE}")
        if EMERGENCY_ZONE:
            print(f"Zone   : {EMERGENCY_ZONE} (emergency origin)")
        print("=" * 60)
        print()

    else:
        print(
            f"MQTT connection failed: {reason_code}"
        )


client.on_connect = on_connect


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def clamp(value, minimum, maximum):
    return max(minimum, min(value, maximum))


def smooth_change(
    current,
    target,
    max_change
):
    difference = target - current

    change = clamp(
        difference,
        -max_change,
        max_change
    )

    return current + change


# ============================================================
# NORMAL ENVIRONMENT
# ============================================================

def update_normal(zone_id, zone):

    state = sensor_state[zone_id]

    # --------------------------------------------------------
    # Temperature
    # --------------------------------------------------------

    target_temperature = random.uniform(
        29.5,
        31.5
    )

    state["temperature"] = smooth_change(
        state["temperature"],
        target_temperature,
        0.35
    )

    # --------------------------------------------------------
    # Humidity
    # --------------------------------------------------------

    target_humidity = random.uniform(
        48,
        58
    )

    state["humidity"] = smooth_change(
        state["humidity"],
        target_humidity,
        1.0
    )

    # --------------------------------------------------------
    # Smoke
    # --------------------------------------------------------

    target_smoke = random.uniform(
        0.05,
        0.15
    )

    state["smoke"] = smooth_change(
        state["smoke"],
        target_smoke,
        0.03
    )

    # --------------------------------------------------------
    # Occupancy
    # --------------------------------------------------------

    occupancy_change = random.choice(
        [-1, 0, 0, 0, 1]
    )

    state["occupancy"] += occupancy_change

    state["occupancy"] = int(
        clamp(
            state["occupancy"],
            0,
            50
        )
    )


# ============================================================
# FIRE SIMULATION
# ============================================================



def update_fire(zone_id, zone):

    state = sensor_state[zone_id]

    # Find the fire-origin zone
    origin = next(
        (item for item in ZONES
         if item["zone_id"] == EMERGENCY_ZONE),
        None
    )

    if origin is None:
        update_normal(zone_id, zone)
        return

    # Determine whether this room is connected to the fire
    connected_zones = set(origin.get("connected_to", []))

    # Find rooms directly connected to the origin
    directly_affected = zone_id in connected_zones

    # Fire-origin room
    if zone_id == EMERGENCY_ZONE:

        state["temperature"] += random.uniform(0.20, 0.40)
        state["smoke"] += random.uniform(0.015, 0.035)

        state["tvoc"] = min(
            state["tvoc"] + random.uniform(10, 30), 60000
        )
        state["eco2"] = min(
            state["eco2"] + random.uniform(5, 15), 60000
        )
        state["pm1"] = min(
            state["pm1"] + random.uniform(0.1, 0.4), 14333.69
        )
        state["pm25"] = min(
            state["pm25"] + random.uniform(0.2, 0.6), 45432.26
        )

    # Directly connected rooms: gradual spread
    elif directly_affected:

        state["temperature"] += random.uniform(0.02, 0.07)
        state["smoke"] += random.uniform(0.002, 0.008)

        state["tvoc"] = min(
            state["tvoc"] + random.uniform(1, 5), 60000
        )
        state["eco2"] = min(
            state["eco2"] + random.uniform(1, 3), 60000
        )
        state["pm1"] = min(
            state["pm1"] + random.uniform(0.01, 0.05), 14333.69
        )
        state["pm25"] = min(
            state["pm25"] + random.uniform(0.02, 0.08), 45432.26
        )

    # Other rooms remain near normal for this simple model
    else:

        state["temperature"] = smooth_change(
            state["temperature"], 30.0, 0.08
        )
        state["smoke"] = smooth_change(
            state["smoke"], 0.10, 0.005
        )

    # Humidity
    state["humidity"] = smooth_change(
        state["humidity"], 45.0, 0.2
    )

    # Simulated occupancy should not exceed the configured base
        # Simulated evacuation
    if state["smoke"] >= 0.6 or state["temperature"] >= 38:
        state["occupancy"] -= random.randint(0, 1)

    state["occupancy"] = int(
        clamp(state["occupancy"], 0, zone["base_occupancy"])
    )

    # Sensor safety limits
    state["temperature"] = clamp(
        state["temperature"], 25, 55
    )
    state["smoke"] = clamp(
        state["smoke"], 0, 1
    )



    # --------------------------------------------------------
    # Humidity
    # --------------------------------------------------------

    state["humidity"] = smooth_change(
        state["humidity"],
        45,
        0.5
    )

    # --------------------------------------------------------
    # Occupancy
    # --------------------------------------------------------

    # People gradually leave affected areas.
    if (
        state["smoke"] >= 0.6
        or state["temperature"] >= 38
    ):

        state["occupancy"] -= random.randint(
            0,
            2
        )

    else:

        state["occupancy"] += random.choice(
            [-1, 0, 0, 1]
        )

    state["occupancy"] = int(
        clamp(
            state["occupancy"],
            0,
            50
        )
    )
 
    # Simulated air-quality sensor changes during fire
    
    if zone_id == EMERGENCY_ZONE:
        state["tvoc"] = 1103.0
        state["eco2"] = 404.0
        state["pm1"] = 1.87
        state["pm25"] = 1.94
    else:
        state["tvoc"] = clamp(state["tvoc"], 0, 60000)
        state["eco2"] = clamp(state["eco2"], 400, 60000)
        state["pm1"] = clamp(state["pm1"], 0, 14333.69)
        state["pm25"] = clamp(state["pm25"], 0, 45432.26)
    # --------------------------------------------------------
    # Safety limits
    # --------------------------------------------------------

    state["temperature"] = clamp(
        state["temperature"],
        25,
        55
    )

    state["smoke"] = clamp(
        state["smoke"],
        0,
        1
    )


# ============================================================
# SMOKE SIMULATION
# ============================================================

def update_smoke(zone_id, zone):

    state = sensor_state[zone_id]

    if zone_id == EMERGENCY_ZONE:

        state["smoke"] += random.uniform(
            0.03,
            0.07
        )

        state["temperature"] += random.uniform(
            0.05,
            0.20
        )

    else:

        state["smoke"] += random.uniform(
            0.005,
            0.025
        )

        state["temperature"] += random.uniform(
            0.01,
            0.05
        )

    state["smoke"] = clamp(
        state["smoke"],
        0,
        1
    )

    state["temperature"] = clamp(
        state["temperature"],
        25,
        50
    )

    state["humidity"] = smooth_change(
        state["humidity"],
        47,
        0.5
    )


# ============================================================
# CROWD SIMULATION
# ============================================================

def update_crowd(zone_id, zone):

    state = sensor_state[zone_id]

    state["occupancy"] += random.randint(
        0,
        2
    )

    state["occupancy"] = int(
        clamp(
            state["occupancy"],
            0,
            50
        )
    )

    state["temperature"] += random.uniform(
        0.01,
        0.08
    )

    state["humidity"] += random.uniform(
        0.02,
        0.10
    )

    state["temperature"] = clamp(
        state["temperature"],
        25,
        45
    )

    state["humidity"] = clamp(
        state["humidity"],
        30,
        80
    )


# ============================================================
# UPDATE SENSOR
# ============================================================

def update_sensor(zone):

    zone_id = zone["zone_id"]

    if SIMULATION_MODE == "FIRE":

        update_fire(
            zone_id,
            zone
        )

    elif SIMULATION_MODE == "SMOKE":

        update_smoke(
            zone_id,
            zone
        )

    elif SIMULATION_MODE == "CROWD":

        update_crowd(
            zone_id,
            zone
        )

    else:

        update_normal(
            zone_id,
            zone
        )


# ============================================================
# CREATE MQTT PAYLOAD
# ============================================================

def create_payload(zone):

    zone_id = zone["zone_id"]

    state = sensor_state[zone_id]

    return {

        "device_id": f"sensor-{zone_id}",

        "building_id": BUILDING_ID,

        "zone_id": zone_id,

        "floor": zone["floor"],

        "temperature": round(
            state["temperature"],
            2
        ),

        "humidity": round(
            state["humidity"],
            2
        ),

        "smoke": round(
            state["smoke"],
            3
        ),
        
        "TVOC[ppb]": round(state["tvoc"], 2),

        "eCO2[ppm]": round(state["eco2"], 2),

        "PM1.0": round(state["pm1"], 2),

        "PM2.5": round(state["pm25"], 2),

        "occupancy": int(
            state["occupancy"]
        ),

        "simulation_mode": SIMULATION_MODE,

        "timestamp": datetime.now(
            timezone.utc
        ).isoformat()
    }


# ============================================================
# PUBLISH SENSOR DATA
# ============================================================

def publish_sensor(zone):

    update_sensor(zone)

    payload = create_payload(zone)

    message = json.dumps(
        payload
    )

    client.publish(
        MQTT_TOPIC,
        message
    )

    print(
        f"[{payload['zone_id']}] "
        f"T={payload['temperature']}°C | "
        f"H={payload['humidity']}% | "
        f"Smoke={payload['smoke']} | "
        f"People={payload['occupancy']}"
    )


# ============================================================
# MAIN LOOP
# ============================================================

def main():

    global EMERGENCY_ZONE

    if SIMULATION_MODE == "FIRE" and not EMERGENCY_ZONE:
        EMERGENCY_ZONE = random.choice(ZONES)["zone_id"]
        print(f"Fire started in Zone {EMERGENCY_ZONE}")

    try:

        print("Connecting to MQTT...")

        client.connect(
            MQTT_BROKER,
            MQTT_PORT,
            60
        )

        client.loop_start()

        time.sleep(1)

        while True:

            for zone in ZONES:

                publish_sensor(zone)

            print(
                "-" * 60
            )

            time.sleep(
                PUBLISH_INTERVAL
            )

    except KeyboardInterrupt:

        print()
        print(
            "Stopping IoT simulator..."
        )

    except Exception as error:

        print(
            f"Simulator error: {error}"
        )

    finally:

        client.loop_stop()
        client.disconnect()


# ============================================================
# START
# ============================================================

if __name__ == "__main__":
    main()


Connecting to MQTT...

EVAC-TWIN IoT SIMULATOR
Connected to MQTT broker
Broker : localhost:1883
Topic  : evac-twin/sensors
Mode   : FIRE
Zone   : 104 (emergency origin)

[101] T=30.26°C | H=52.83% | Smoke=0.066 | People=6
[102] T=29.82°C | H=51.72% | Smoke=0.13 | People=7
[103] T=30.04°C | H=49.02% | Smoke=0.097 | People=5
[104] T=30.96°C | H=50.18% | Smoke=0.074 | People=4
[201] T=29.94°C | H=55.56% | Smoke=0.129 | People=7
[202] T=29.14°C | H=57.27% | Smoke=0.138 | People=6
[203] T=29.98°C | H=52.3% | Smoke=0.067 | People=5
[204] T=29.72°C | H=52.77% | Smoke=0.103 | People=5
[301] T=29.59°C | H=55.99% | Smoke=0.1 | People=7
[302] T=30.14°C | H=56.6% | Smoke=0.121 | People=6
[303] T=30.0°C | H=56.83% | Smoke=0.126 | People=5
[304] T=29.8°C | H=53.55% | Smoke=0.125 | People=3
------------------------------------------------------------
[101] T=30.18°C | H=52.13% | Smoke=0.071 | People=6
[102] T=29.9°C | H=51.02% | Smoke=0.125 | People=7
[103] T=30.07°C | H=48.32% | Smoke=0.1 | People=4

### backend/app/mqtt_subscriber.py

**File:** `backend/app/mqtt_subscriber.py`

This is the actual project source copied into the notebook for inspection.
It is a reference section, not an instruction to execute the entire file in a notebook cell.

> This file has 908 lines and is split into numbered reference sections of at most 220 lines. All lines are included.

#### backend/app/mqtt_subscriber.py — lines 1–220

In [11]:

import json
import os
from datetime import datetime, timezone


import paho.mqtt.client as mqtt

from app.database import (
    sensors_collection,
    events_collection,
)
from app.ml.predict import predict_risk
from app.ml.fire_predict import predict_fire
from app.services.sms_service import send_sms
from app.services.routing_service import calculate_all_routes
from app.services.sensor_service import latest_zone_intelligence
from app.services.alert_state import (
    claim_sms_alert,
    release_sms_alert,
    clear_sms_alert,
)


# ============================================================
# EVAC-TWIN — MQTT SENSOR INGESTION
# ============================================================

MQTT_BROKER = os.getenv(
    "MQTT_BROKER",
    "localhost",
)

MQTT_PORT = int(
    os.getenv(
        "MQTT_PORT",
        "1883",
    )
)

MQTT_TOPIC = os.getenv(
    "MQTT_TOPIC",
    "evac-twin/sensors",
)


# ============================================================
# RISK ORDER
# ============================================================

RISK_ORDER = {
    "LOW": 1,
    "MODERATE": 2,
    "HIGH": 3,
    "CRITICAL": 4,
}


# ============================================================
# PREVIOUS RISK STATE
# ============================================================
#
# Keeps the most recent risk level for every zone in memory.
#
# Example:
#
# {
#     "304": "CRITICAL",
#     "303": "HIGH"
# }
#
# This prevents a new event from being created every time
# the simulator publishes the same risk level.
# ============================================================

previous_risk_levels = {}
previous_fire_alarms = {}

# ============================================================
# SAFE NUMBER
# ============================================================

def safe_float(
    value,
    default=0.0,
):
    try:
        return float(value)

    except (
        TypeError,
        ValueError,
    ):
        return default


# ============================================================
# SAFE INTEGER
# ============================================================

def safe_int(
    value,
    default=0,
):
    try:
        return int(
            float(value)
        )

    except (
        TypeError,
        ValueError,
    ):
        return default


# ============================================================
# TIMESTAMP
# ============================================================

def utc_timestamp():
    return datetime.now(
        timezone.utc
    ).isoformat()


# ============================================================
# CREATE RISK EVENT
# ============================================================

def create_risk_event(
    sensor,
    previous_level,
    current_level,
):
    """
    Create an event when a zone's risk level changes.

    Events are generated for:

        LOW → MODERATE
        MODERATE → HIGH
        HIGH → CRITICAL

    and also for recovery:

        CRITICAL → HIGH
        HIGH → MODERATE
        MODERATE → LOW
    """

    zone_id = str(
        sensor.get(
            "zone_id",
            "",
        )
    )

    building_id = sensor.get(
        "building_id",
        "building-alpha",
    )

    risk_score = safe_float(
        sensor.get(
            "risk_score",
            0,
        )
    )

    # --------------------------------------------------------
    # Determine event type
    # --------------------------------------------------------

    previous_rank = RISK_ORDER.get(
        previous_level,
        0,
    )

    current_rank = RISK_ORDER.get(
        current_level,
        0,
    )

    if current_rank > previous_rank:

        event_type = "RISK_ESCALATION"

        severity = current_level

        message = (
            f"Risk escalated in Zone {zone_id}: "
            f"{previous_level} → {current_level}."
        )

    elif current_rank < previous_rank:

        event_type = "RISK_REDUCTION"

        severity = current_level

        message = (
            f"Risk reduced in Zone {zone_id}: "
            f"{previous_level} → {current_level}."
        )

    else:

        return



    
    # Resolve previous active risk events when risk decreases
    if current_rank < previous_rank:
        events_collection.update_many(
            {
                "building_id": building_id,
                "zone_id": zone_id,
                "status": "ACTIVE",


SyntaxError: incomplete input (160314739.py, line 219)

#### backend/app/mqtt_subscriber.py — lines 221–440

In [ ]:
                "event_type": "RISK_ESCALATION",
            },
            {
                "$set": {
                    "status": "RESOLVED",
                    "resolved_at": utc_timestamp(),
                }
            },
        )
    # Resolve older escalation events when risk decreases.
   
    # --------------------------------------------------------
    # Create event document
    # --------------------------------------------------------

    event = {
        "event_type": event_type,

        "building_id": building_id,

        "zone_id": zone_id,

        "device_id": sensor.get(
            "device_id",
            f"sensor-{zone_id}",
        ),

        "previous_risk_level": previous_level,

        "risk_level": current_level,

        "risk_score": risk_score,

        "severity": severity,

        "message": message,

        "recommendation": sensor.get(
            "recommendation",
            "Continue monitoring sensor conditions.",
        ),

        "simulation_mode": sensor.get(
            "simulation_mode",
            "LIVE",
        ),

        "timestamp": utc_timestamp(),

        "status": (
    "RESOLVED"
    if event_type == "RISK_REDUCTION"
    else "ACTIVE"
),
    }

    try:

        events_collection.insert_one(
            event
        )

        print(
            "[EVENT] "
            f"{event_type} | "
            f"Zone {zone_id} | "
            f"{previous_level} → {current_level}"
        )

    except Exception as error:

        print(
            "[EVENT ERROR] "
            f"Unable to save event: {error}"
        )




def create_fire_alarm_event(sensor):
    risk_level = str(
        sensor.get("risk_level", "LOW")
    ).upper()

    risk_score = safe_float(
        sensor.get("risk_score", 0)
    )

    fire_probability = safe_float(
        sensor.get("fire_alarm_probability", 0)
    )

    # Do not create a CRITICAL fire event from LOW-risk data
    # unless the fire model independently confirms a strong alarm.
    if (
        sensor.get("fire_alarm_prediction") != 1
        or fire_probability < 80
    ):
        return

    event = {
        "event_type": "FIRE_ALARM_PREDICTED",
        "building_id": sensor.get(
            "building_id", "building-alpha"
        ),
        "zone_id": sensor.get("zone_id", ""),
        "device_id": sensor.get("device_id", ""),
        "risk_level": risk_level,
        "risk_score": risk_score,
        "severity": "CRITICAL",
        "message": (
            f"FIRE ALERT: Fire alarm predicted in Zone "
            f"{sensor.get('zone_id', '')}, "
            f"Floor {sensor.get('floor', 'Unknown')}."
        ),
        "recommendation": (
            "Check the affected zone and follow the verified "
            "evacuation route."
        ),
        "simulation_mode": sensor.get(
            "simulation_mode", "LIVE"
        ),
        "timestamp": utc_timestamp(),
        "status": "ACTIVE",
    }

    try:
        events_collection.insert_one(event)
        print("\nFIRE ALERT")
        print(event["message"])
        print("Risk:", risk_level)
        print("Fire probability:", fire_probability)
        print(event["recommendation"])

    except Exception as error:
        print(f"[FIRE ALERT ERROR] {error}")

# ============================================================
# MQTT CONNECT
# ============================================================

def on_connect(
    client,
    userdata,
    flags,
    reason_code,
    properties,
):

    if reason_code == 0:

        print()
        print(
            "MQTT connected"
        )

        client.subscribe(
            MQTT_TOPIC
        )

        print(
            f"Subscribed to {MQTT_TOPIC}"
        )

    else:

        print(
            "MQTT connection failed: "
            f"{reason_code}"
        )


# ============================================================
# MQTT MESSAGE
# ============================================================

def on_message(
    client,
    userdata,
    msg,
):

    try:

        # ====================================================
        # READ PAYLOAD
        # ====================================================

        payload = msg.payload.decode(
            "utf-8"
        )

        sensor = json.loads(
            payload
        )

        if not isinstance(sensor, dict):

            print(
                "[MQTT] "
                "Ignoring invalid sensor payload."
            )

            return

        # ====================================================
        # NORMALIZE ZONE
        # ====================================================

        sensor["Temperature[C]"] = sensor.get("Temperature[C]", sensor.get("temperature"))
        sensor["Humidity[%]"] = sensor.get("Humidity[%]", sensor.get("humidity"))
        sensor["TVOC[ppb]"] = sensor.get("TVOC[ppb]")
        sensor["eCO2[ppm]"] = sensor.get("eCO2[ppm]")
        sensor["PM1.0"] = sensor.get("PM1.0")
        sensor["PM2.5"] = sensor.get("PM2.5")
        zone_id = sensor.get(
            "zone_id"
        )

        device_id = sensor.get(


#### backend/app/mqtt_subscriber.py — lines 441–660

In [ ]:
            "device_id"
        )

        if zone_id is None and device_id:

            zone_id = str(
                device_id
            ).replace(
                "sensor-",
                "",
                1,
            )

        if zone_id is not None:

            zone_id = str(
                zone_id
            )

        else:

            zone_id = ""

        if not device_id:

            device_id = (
                f"sensor-{zone_id}"
            )

        sensor["zone_id"] = zone_id

        sensor["device_id"] = device_id

        # ====================================================
        # NORMALIZE SENSOR VALUES
        # ====================================================

        sensor["temperature"] = safe_float(
            sensor.get(
                "temperature",
                0,
            )
        )

        sensor["humidity"] = safe_float(
            sensor.get(
                "humidity",
                0,
            )
        )

        sensor["smoke"] = safe_float(
            sensor.get(
                "smoke",
                0,
            )
        )

        sensor["occupancy"] = safe_int(
            sensor.get(
                "occupancy",
                0,
            )
        )

        # ====================================================
        # TIMESTAMP
        # ====================================================

        if not sensor.get(
            "timestamp"
        ):

            sensor["timestamp"] = (
                utc_timestamp()
            )

                # ====================================================
        # DEEP LEARNING RISK PREDICTION
        # ====================================================

        dl_result = predict_risk(
            {
                "temperature": sensor["temperature"],
                "humidity": sensor["humidity"],
                "smoke": sensor["smoke"],
                "occupancy": sensor["occupancy"],
            }
        )

        current_level = str(
            dl_result.get("level", "LOW")
        ).upper()

        sensor["risk_score"] = float(
            dl_result.get("score", 0)
        )

        sensor["risk_level"] = current_level
        sensor["dl_prediction"] = current_level

        sensor["confidence"] = float(
            dl_result.get("confidence", 0)
        )

        sensor["dl_confidence"] = sensor["confidence"]

        sensor["dl_probabilities"] = dl_result.get(
            "probabilities", {}
        )

        sensor["recommendation"] = dl_result.get(
            "recommendation",
            "Continue monitoring sensor conditions."
        )
        
        
        # DATASET-BASED FIRE PREDICTION

        fire_features = [
            "Temperature[C]",
            "Humidity[%]",
            "TVOC[ppb]",
            "eCO2[ppm]",
            "PM1.0",
            "PM2.5",
        ]

        has_fire_features = all(
            feature in sensor and sensor[feature] is not None
            for feature in fire_features
        )

        print(
            "FIRE FEATURES:",
            {
                feature: sensor.get(feature)
                for feature in fire_features
            }
        )


        if has_fire_features:
            fire_result = predict_fire(
                {
                    feature: safe_float(sensor[feature])
                    for feature in fire_features
                }
            )

            print("FIRE MODEL RESULT:", fire_result)

            current_fire_alarm = fire_result["fire_alarm"]
            fire_probability = fire_result.get("probability") or 0

            sensor["fire_alarm_dl_prediction"] = current_fire_alarm
            sensor["fire_alarm_probability"] = fire_probability
            sensor["fire_alarm_prediction"] = (
               1
               if current_fire_alarm == 1 and fire_probability >= 80
               else 0
)
            sensor["fire_alarm_message"] = fire_result["message"]
            sensor["fire_alarm_source"] = "DL_MODEL"
            print(
    "[FIRE CHECK]",
    sensor["device_id"],
    "Probability =", sensor["fire_alarm_probability"],
    "Final =", sensor["fire_alarm_prediction"]
)

           
            # Create an event only when the final prediction
            # crosses the configured threshold.
            print(
                "[FIRE CHECK]",
                sensor["device_id"],
                "Probability =", sensor["fire_alarm_probability"],
                "Final =", sensor["fire_alarm_prediction"]
            )

            print(
                "[FIRE DEBUG]",
                "Device =", sensor.get("device_id"),
                "Prediction =", sensor.get("fire_alarm_prediction"),
                "Probability =", fire_probability,
                "Previous =", previous_fire_alarms.get(zone_id, 0)
            )

            if (
                sensor["fire_alarm_prediction"] == 1
                and fire_probability >= 80
            ):
                previous_fire_alarm = previous_fire_alarms.get(
                    zone_id, 0
                )

                if previous_fire_alarm != 1:
                    create_fire_alarm_event(sensor)

                previous_fire_alarms[zone_id] = 1

            else:
                previous_fire_alarms[zone_id] = 0

                # Resolve previous fire alerts when prediction clears.
                events_collection.update_many(
                    {
                        "building_id": sensor.get(
                            "building_id",
                            "building-alpha",
                        ),
                        "zone_id": zone_id,
                        "event_type": "FIRE_ALARM_PREDICTED",
                        "status": "ACTIVE",
                    },
                    {
                        "$set": {
                            "status": "RESOLVED",
                            "resolved_at": utc_timestamp(),


#### backend/app/mqtt_subscriber.py — lines 661–880

In [ ]:
                        }
                    },
                )


        # ====================================================
        # SIMULATION MODE
        # ====================================================

        if "simulation_mode" not in sensor:

            sensor["simulation_mode"] = (
                "LIVE"
            )

        # ====================================================
        # SOURCE
        # ====================================================

        sensor["source"] = "MQTT"

        # ====================================================
        # RISK EVENT DETECTION
        # ====================================================

        previous_level = (
            previous_risk_levels.get(
                zone_id
            )
        )

        # Resolve stale active risk events
        if (
            current_level not in ("HIGH", "CRITICAL")
            and (
                previous_level is None
                or previous_level in ("HIGH", "CRITICAL")
            )
        ):
            events_collection.update_many(
                {
                    "building_id": sensor.get(
                        "building_id",
                        "building-alpha",
                    ),
                    "zone_id": zone_id,
                    "event_type": "RISK_ESCALATION",
                    "status": "ACTIVE",
                },
                {
                    "$set": {
                        "status": "RESOLVED",
                        "resolved_at": utc_timestamp(),
                    }
                },
            )

        # Existing escalation/recovery code continues here
        effective_previous = (
            previous_level
            if previous_level is not None
            else "LOW"
        )
        # --------------------------------------------------
        # ESCALATION: zone is now HIGH or CRITICAL.
        # Fire an event when entering this band for the
        # first time (previous_level is None → treat as LOW)
        # or when transitioning up from a lower band.
        # --------------------------------------------------
        effective_previous = previous_level if previous_level is not None else "LOW"

        if current_level in ("HIGH", "CRITICAL"):
            if effective_previous not in ("HIGH", "CRITICAL"):
                create_risk_event(
                    sensor,
                    effective_previous,
                    current_level,
                )

        # --------------------------------------------------
        # RECOVERY: zone was HIGH/CRITICAL and has now
        # dropped below that threshold.
        # --------------------------------------------------
        elif previous_level in ("HIGH", "CRITICAL") and current_level not in ("HIGH", "CRITICAL"):
            create_risk_event(
                sensor,
                previous_level,
                current_level,
            )

        # Save the latest state.
        previous_risk_levels[
            zone_id
        ] = current_level

    
           # ====================================================
        # ONE-TIME EMERGENCY SMS ALERT
        # ====================================================

        ml_fire_alarm = (
           sensor.get("fire_alarm_prediction", 0) == 1
)

        risk_emergency = current_level in (
    "HIGH",
    "CRITICAL"
)

        emergency_active = ml_fire_alarm or risk_emergency

        if emergency_active:

            if claim_sms_alert(zone_id):

                try:
                    sensors = latest_zone_intelligence() or []

                    sensors = [
                        s for s in sensors
                        if str(s.get("zone_id")) != zone_id
                    ]

                    sensors.append(sensor)

                    route_result = calculate_all_routes(
                        sensors,
                        selected_zone=zone_id
                    )

                    recommended = route_result.get(
                        "selected_zone_recommendation"
                    )

                    route_name = (
                        f"{recommended.get('exit')} - "
                        f"{recommended.get('name')}"
                        if recommended
                        else "No safe exit available"
                    )

                    message = (
                        f"EVAC-TWIN EMERGENCY ALERT! "
                        f"Zone: {zone_id}. "
                        f"Risk: {current_level}. "
                        f"Risk Score: {sensor['risk_score']}. "
                        f"Recommended Exit: {route_name}. "
                        f"Please evacuate immediately."
                    )

                    result = send_sms(message)

                    if result:
                        print(f"[SMS] Alert sent for Zone {zone_id}")
                    else:
                        release_sms_alert(zone_id)
                        print(f"[SMS] Failed for Zone {zone_id}")

                except Exception as e:
                    release_sms_alert(zone_id)
                    print("[SMS] Emergency alert error:", e)

        else:
            clear_sms_alert(zone_id)

        

        # ====================================================
        # SAVE SENSOR DATA
        # ====================================================

        sensors_collection.insert_one(
            sensor
        )

        # ====================================================
        # CONSOLE LOG
        # ====================================================

        print(
            "[SAVED] "
            f"Zone {zone_id} | "
            f"{sensor['temperature']:.2f}°C | "
            f"Humidity "
            f"{sensor['humidity']:.2f}% | "
            f"Smoke "
            f"{sensor['smoke']:.3f} | "
            f"People "
            f"{sensor['occupancy']} | "
            f"DL "
            f"{sensor['dl_prediction']} | "
            f"Score "
            f"{sensor['risk_score']:.2f} | "
            f"Confidence "
            f"{sensor['dl_confidence']:.2f}%"
        )

    except json.JSONDecodeError as error:

        print(
            f"[MQTT JSON ERROR] {error}"
        )

    except Exception as error:

        print(
            f"[MQTT ERROR] {error}"
        )


# ============================================================
# START MQTT SUBSCRIBER
# ============================================================

def start_mqtt_subscriber():

    client = mqtt.Client(
        mqtt.CallbackAPIVersion.VERSION2
    )



#### backend/app/mqtt_subscriber.py — lines 881–908

In [ ]:
    client.on_connect = on_connect

    client.on_message = on_message

    client.connect(
        MQTT_BROKER,
        MQTT_PORT,
        60,
    )

    client.loop_start()

    print()
    print(
        "MQTT subscriber started"
    )

    print(
        f"Broker : "
        f"{MQTT_BROKER}:{MQTT_PORT}"
    )

    print(
        f"Topic  : "
        f"{MQTT_TOPIC}"
    )

    return client


### backend/app/risk_engine.py

**File:** `backend/app/risk_engine.py`

This is the actual project source copied into the notebook for inspection.
It is a reference section, not an instruction to execute the entire file in a notebook cell.

In [ ]:

# ============================================================
# EVAC-TWIN - DETERMINISTIC RISK ENGINE
# ============================================================

def _safe_float(value, default=0.0):
    try:
        if value is None or value == "":
            return default
        return float(value)
    except (TypeError, ValueError):
        return default


def _safe_int(value, default=0):
    try:
        if value is None or value == "":
            return default
        return int(float(value))
    except (TypeError, ValueError):
        return default


def calculate_risk(sensor):
    if not sensor:
        return {
            "score": 0,
            "level": "NO DATA",
            "recommendation": "Waiting for sensor data.",
            "factors": {
                "temperature": 0,
                "smoke": 0,
                "occupancy": 0,
            },
        }

    temperature = max(
        _safe_float(sensor.get("temperature")),
        0.0
    )

    smoke = max(
        _safe_float(sensor.get("smoke")),
        0.0
    )

    occupancy = max(
        _safe_int(sensor.get("occupancy")),
        0
    )

    # Temperature risk - Maximum 35
    if temperature >= 50:
        temperature_risk = 35
    elif temperature >= 45:
        temperature_risk = 30
    elif temperature >= 40:
        temperature_risk = 25
    elif temperature >= 35:
        temperature_risk = 15
    elif temperature >= 32:
        temperature_risk = 5
    else:
        temperature_risk = 0

    # Smoke risk - Maximum 40
    if smoke >= 0.8:
        smoke_risk = 40
    elif smoke >= 0.6:
        smoke_risk = 35
    elif smoke >= 0.5:
        smoke_risk = 30
    elif smoke >= 0.3:
        smoke_risk = 15
    elif smoke >= 0.15:
        smoke_risk = 5
    else:
        smoke_risk = 0

    # Occupancy risk - Maximum 25
    if occupancy >= 50:
        occupancy_risk = 25
    elif occupancy >= 40:
        occupancy_risk = 20
    elif occupancy >= 30:
        occupancy_risk = 15
    elif occupancy >= 15:
        occupancy_risk = 5
    else:
        occupancy_risk = 0

    # Total risk score
    score = min(
        temperature_risk + smoke_risk + occupancy_risk,
        100
    )

    # Risk classification
    if score >= 70:
        level = "CRITICAL"
        recommendation = (
            "Immediate evacuation recommended. "
            "Activate emergency procedures."
        )
    elif score >= 40:
        level = "HIGH"
        recommendation = (
            "High-risk conditions detected. "
            "Prepare evacuation routes and alert occupants."
        )
    elif score >= 20:
        level = "MODERATE"
        recommendation = (
            "Elevated conditions detected. "
            "Continue close monitoring and prepare for escalation."
        )
    else:
        level = "LOW"
        recommendation = (
            "Environmental conditions are stable. "
            "Continue normal monitoring."
        )

    return {
        "score": score,
        "level": level,
        "recommendation": recommendation,
        "factors": {
            "temperature": temperature_risk,
            "smoke": smoke_risk,
            "occupancy": occupancy_risk,
        },
    }


### backend/app/ml/predict.py

**File:** `backend/app/ml/predict.py`

This is the actual project source copied into the notebook for inspection.
It is a reference section, not an instruction to execute the entire file in a notebook cell.

> This file has 360 lines and is split into numbered reference sections of at most 220 lines. All lines are included.

#### backend/app/ml/predict.py — lines 1–220

In [ ]:

import os
import json
import joblib
import numpy as np
import torch
import torch.nn as nn


# ============================================================
# EVAC-TWIN - DEEP LEARNING RISK PREDICTION
# ============================================================

BASE_DIR = os.path.dirname(os.path.abspath(__file__))

MODEL_DIR = os.path.join(BASE_DIR, "models")

MODEL_PATH = os.path.join(
    MODEL_DIR,
    "risk_model.pth"
)

SCALER_PATH = os.path.join(
    MODEL_DIR,
    "risk_scaler.pkl"
)

CONFIG_PATH = os.path.join(
    MODEL_DIR,
    "risk_config.json"
)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


# ============================================================
# LOAD MODEL CONFIGURATION
# ============================================================

try:

    with open(CONFIG_PATH, "r") as f:
        config = json.load(f)

    CLASSES = config["classes"]
    FEATURES = config["features"]

except Exception as error:

    print(f"[DL ERROR] Config loading failed: {error}")

    CLASSES = ["CRITICAL", "HIGH", "LOW", "MODERATE"]
    FEATURES = ["temperature", "humidity", "smoke", "occupancy"]


# ============================================================
# DEEP LEARNING MODEL
# ============================================================

class RiskNet(nn.Module):

    def __init__(self):

        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(4, 64),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.20),
            nn.Linear(32, len(CLASSES))
        )

    def forward(self, x):

        return self.net(x)


# ============================================================
# LOAD TRAINED MODEL AND SCALER
# ============================================================

model = None
scaler = None

try:

    scaler = joblib.load(SCALER_PATH)

    model = RiskNet().to(DEVICE)

    state_dict = torch.load(
        MODEL_PATH,
        map_location=DEVICE
    )

    model.load_state_dict(state_dict)
    model.eval()

    print("[DL] Risk model loaded successfully")
    print("[DL] Device:", DEVICE)
    print("[DL] Classes:", CLASSES)

except Exception as error:

    print("[DL ERROR] Model loading failed:", error)

    model = None
    scaler = None


# ============================================================
# RISK SCORES
# ============================================================

RISK_SCORES = {
    "LOW": 10,
    "MODERATE": 40,
    "HIGH": 70,
    "CRITICAL": 100,
}


# ============================================================
# SAFE NUMERIC CONVERSION
# ============================================================

def safe_float(value, default=0.0):

    try:

        return float(value)

    except (TypeError, ValueError):

        return default


# ============================================================
# RECOMMENDATION
# ============================================================

def get_recommendation(level):

    level = str(level or "LOW").upper()

    if level == "CRITICAL":

        return (
            "Immediate evacuation recommended. "
            "Activate emergency procedures."
        )

    if level == "HIGH":

        return (
            "High-risk conditions detected. "
            "Prepare evacuation routes and alert occupants."
        )

    if level == "MODERATE":

        return (
            "Some elevated conditions are present. "
            "Continue monitoring sensor values."
        )

    return (
        "Environmental conditions are stable. "
        "Continue normal monitoring."
    )


# ============================================================
# FALLBACK RISK CALCULATION
# ============================================================

def fallback_risk(
    temperature,
    humidity,
    smoke,
    occupancy
):

    score = 0

    if temperature >= 40:
        score += 35

    elif temperature >= 38:
        score += 25

    elif temperature >= 35:
        score += 15

    if smoke >= 0.8:
        score += 40

    elif smoke >= 0.5:
        score += 25

    elif smoke >= 0.3:
        score += 10

    if occupancy >= 45:
        score += 25

    elif occupancy >= 30:
        score += 15

    elif occupancy >= 15:
        score += 5

    score = min(score, 100)

    if score >= 70:


#### backend/app/ml/predict.py — lines 221–360

In [ ]:
        level = "CRITICAL"

    elif score >= 40:
        level = "HIGH"

    elif score >= 20:
        level = "MODERATE"

    else:
        level = "LOW"

    return {
        "level": level,
        "score": score,
        "confidence": 100.0,
        "probabilities": {
            name: 100.0 if name == level else 0.0
            for name in RISK_SCORES
        },
        "recommendation": get_recommendation(level),
    }


# ============================================================
# DEEP LEARNING RISK PREDICTION
# ============================================================

def predict_risk(sensor_data):

    temperature = safe_float(sensor_data.get("temperature"))
    humidity = safe_float(sensor_data.get("humidity"))
    smoke = safe_float(sensor_data.get("smoke"))
    occupancy = safe_float(sensor_data.get("occupancy"))

    if model is None or scaler is None:

        return fallback_risk(
            temperature,
            humidity,
            smoke,
            occupancy
        )

    try:

        features = np.array(
            [[temperature, humidity, smoke, occupancy]],
            dtype=np.float32
        )

        scaled_features = scaler.transform(features)

        input_tensor = torch.tensor(
            scaled_features,
            dtype=torch.float32
        ).to(DEVICE)

        model.eval()

        with torch.no_grad():

            logits = model(input_tensor)

            probabilities = torch.softmax(
                logits,
                dim=1
            )[0]

        probabilities = probabilities.cpu().numpy()

        prediction_index = int(np.argmax(probabilities))

        ml_level = CLASSES[prediction_index]

        probability_map = {
            name: 0.0 for name in RISK_SCORES
        }

        for label, probability in zip(CLASSES, probabilities):

            probability_map[label] = round(
                float(probability) * 100,
                2
            )

        confidence = round(
            float(np.max(probabilities)) * 100,
            2
        )

        risk_score = sum(
            probability_map[level] * weight / 100
            for level, weight in RISK_SCORES.items()
        )

        rule_result = fallback_risk(
            temperature,
            humidity,
            smoke,
            occupancy
        )

        risk_priority = {
            "LOW": 1,
            "MODERATE": 2,
            "HIGH": 3,
            "CRITICAL": 4,
        }

        rule_level = rule_result["level"]

        level = (
            rule_level
            if risk_priority[rule_level] > risk_priority[ml_level]
            else ml_level
        )

        score = round(
            min(max(max(risk_score, rule_result["score"]), 0), 100),
            2
        )

        return {
            "level": level,
            "score": score,
            "confidence": confidence,
            "probabilities": probability_map,
            "recommendation": get_recommendation(level),
        }

    except Exception as error:

        print("[DL ERROR] Prediction failed:", error)

        return fallback_risk(
            temperature,
            humidity,
            smoke,
            occupancy
        )


### backend/app/ml/fire_predict.py

**File:** `backend/app/ml/fire_predict.py`

This is the actual project source copied into the notebook for inspection.
It is a reference section, not an instruction to execute the entire file in a notebook cell.

In [ ]:

import os
import json
import joblib
import numpy as np
import torch
import torch.nn as nn


BASE_DIR = os.path.dirname(os.path.abspath(__file__))
MODEL_DIR = os.path.join(BASE_DIR, "models")

MODEL_PATH = os.path.join(MODEL_DIR, "fire_alarm_model.pth")
SCALER_PATH = os.path.join(MODEL_DIR, "fire_alarm_scaler.pkl")
CONFIG_PATH = os.path.join(MODEL_DIR, "fire_alarm_config.json")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class FireModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(6, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.network(x)


FEATURES = [
    "Temperature[C]",
    "Humidity[%]",
    "TVOC[ppb]",
    "eCO2[ppm]",
    "PM1.0",
    "PM2.5"
]

model = None
scaler = None
temperature = 1.0
threshold = 0.5


try:
    with open(CONFIG_PATH, "r") as f:
        config = json.load(f)

    FEATURES = config["features"]
    temperature = float(config["temperature"])
    threshold = float(config["threshold"])

    scaler = joblib.load(SCALER_PATH)

    model = FireModel().to(DEVICE)

    state_dict = torch.load(
        MODEL_PATH,
        map_location=DEVICE
    )

    model.load_state_dict(state_dict)
    model.eval()

    print("[DL] Fire Detection model loaded successfully")
    print("[DL] Device:", DEVICE)
    print("[DL] Threshold:", threshold)
    print("[DL] Calibration temperature:", temperature)

except Exception as error:
    print("[DL ERROR] Fire model loading failed:", error)
    model = None
    scaler = None


def predict_fire(sensor_data):

    if model is None or scaler is None:
        return {
            "fire_alarm": None,
            "probability": None,
            "message": "Fire DL model is not available."
        }

    try:

        values = []

        for feature in FEATURES:

            value = sensor_data.get(feature)

            if value is None:
                raise ValueError(
                    f"Missing feature: {feature}"
                )

            value = float(value)

            if not np.isfinite(value):
                raise ValueError(
                    f"Invalid feature: {feature}"
                )

            values.append(value)

        input_data = np.array(
            [values],
            dtype=np.float32
        )

        scaled_data = scaler.transform(input_data)

        input_tensor = torch.tensor(
            scaled_data,
            dtype=torch.float32
        ).to(DEVICE)

        model.eval()

        with torch.no_grad():

            logits = model(input_tensor)

            probability = torch.sigmoid(
                logits / temperature
            ).item()

        prediction = int(probability >= threshold)

        return {
            "fire_alarm": prediction,
            "probability": round(probability * 100, 2),
            "message": (
                "Fire detected by Deep Learning model."
                if prediction == 1
                else "No fire detected by Deep Learning model."
            )
        }

    except Exception as error:

        print("[DL ERROR] Fire prediction failed:", error)

        return {
            "fire_alarm": None,
            "probability": None,
            "message": str(error)
        }


### backend/app/ml/zone_risk.py

**File:** `backend/app/ml/zone_risk.py`

This is the actual project source copied into the notebook for inspection.
It is a reference section, not an instruction to execute the entire file in a notebook cell.

> This file has 346 lines and is split into numbered reference sections of at most 220 lines. All lines are included.

#### backend/app/ml/zone_risk.py — lines 1–220

In [ ]:

from app.ml.predict import predict_risk


# ============================================================
# EVAC-TWIN — ZONE RISK INTELLIGENCE
# ============================================================


# ============================================================
# FLOOR DETECTION
# ============================================================

def get_floor(device_id):

    try:

        device_number = int(
            str(device_id).replace(
                "sensor-",
                "",
                1
            )
        )

        return device_number // 100

    except (
        ValueError,
        TypeError
    ):

        return 0


# ============================================================
# ZONE ID
# ============================================================

def get_zone_id(sensor):

    if not sensor:
        return ""

    zone_id = sensor.get(
        "zone_id"
    )

    if zone_id is not None:

        return str(
            zone_id
        )

    device_id = str(
        sensor.get(
            "device_id",
            ""
        )
    )

    if device_id.startswith(
        "sensor-"
    ):

        return device_id.replace(
            "sensor-",
            "",
            1
        )

    return device_id


# ============================================================
# ZONE NAME
# ============================================================

def get_zone_name(device_id):

    zone_id = str(
        device_id
    ).replace(
        "sensor-",
        "",
        1
    )

    return f"Zone {zone_id}"


# ============================================================
# ZONE RISK
# ============================================================

def calculate_zone_risk(sensor):

    zone_id = get_zone_id(
        sensor
    )

    device_id = sensor.get(
        "device_id",
        f"sensor-{zone_id}"
    )

    # --------------------------------------------------------
    # RUN DEEP LEARNING MODEL
    # --------------------------------------------------------

    dl_result = predict_risk({

        "temperature":
            sensor.get(
                "temperature",
                0
            ),

        "humidity":
            sensor.get(
                "humidity",
                0
            ),

        "smoke":
            sensor.get(
                "smoke",
                0
            ),

        "occupancy":
            sensor.get(
                "occupancy",
                0
            ),

    })

    # --------------------------------------------------------
    # RETURN COMPLETE ZONE RESULT
    # --------------------------------------------------------

    return {

        "device_id":
            device_id,

        "zone_id":
            zone_id,

        "zone":
            get_zone_name(
                zone_id
            ),

        "floor":
            get_floor(
                zone_id
            ),

        "temperature":
            sensor.get(
                "temperature",
                0
            ),

        "humidity":
            sensor.get(
                "humidity",
                0
            ),

        "smoke":
            sensor.get(
                "smoke",
                0
            ),

        "occupancy":
            sensor.get(
                "occupancy",
                0
            ),

        # ----------------------------------------------------
        # DEEP LEARNING RESULTS
        # ----------------------------------------------------

        "risk_level":
            dl_result[
                "level"
            ],

        "risk_score":
            dl_result[
                "score"
            ],

        "confidence":
            dl_result[
                "confidence"
            ],

        "dl_confidence":
            dl_result[
                "confidence"
            ],

        "dl_prediction":
            dl_result[
                "level"
            ],

        "dl_probabilities":
            dl_result[
                "probabilities"
            ],

        "recommendation":
            dl_result[


#### backend/app/ml/zone_risk.py — lines 221–346

In [ ]:
                "recommendation"
            ],

        # ----------------------------------------------------
        # SENSOR METADATA
        # ----------------------------------------------------

        "simulation_mode":
            sensor.get(
                "simulation_mode"
            ),

        "timestamp":
            sensor.get(
                "timestamp"
            ),
    }


# ============================================================
# RANK ZONES
# ============================================================

def rank_zones(sensors):

    zone_results = []

    for sensor in sensors:

        try:

            result = calculate_zone_risk(
                sensor
            )

            zone_results.append(
                result
            )

        except Exception as error:

            print(
                "[ZONE DL ERROR] "
                f"{sensor.get('device_id')}: "
                f"{error}"
            )

    # --------------------------------------------------------
    # RISK ORDER
    # --------------------------------------------------------

    risk_order = {

        "CRITICAL": 4,

        "HIGH": 3,

        "MODERATE": 2,

        "LOW": 1,

        "SAFE": 0,

    }

    # --------------------------------------------------------
    # SORT
    # --------------------------------------------------------

    zone_results.sort(

        key=lambda zone: (

            risk_order.get(

                str(
                    zone.get(
                        "risk_level",
                        "LOW"
                    )
                ).upper(),

                0

            ),

            float(
                zone.get(
                    "confidence",
                    0
                )
            ),

            float(
                zone.get(
                    "risk_score",
                    0
                )
            ),

        ),

        reverse=True

    )

    return zone_results


# ============================================================
# MOST DANGEROUS ZONE
# ============================================================

def get_most_dangerous_zone(
    sensors
):

    ranked_zones = rank_zones(
        sensors
    )

    if not ranked_zones:

        return None

    return ranked_zones[0]


### backend/app/ml/train_risk_model.py

**File:** `backend/app/ml/train_risk_model.py`

This is the actual project source copied into the notebook for inspection.
It is a reference section, not an instruction to execute the entire file in a notebook cell.

> This file has 969 lines and is split into numbered reference sections of at most 220 lines. All lines are included.

#### backend/app/ml/train_risk_model.py — lines 1–220

In [ ]:

import os
import json
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, label_binarize
from sklearn.utils.class_weight import compute_class_weight

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_curve,
    auc,
    precision_recall_curve,
    average_precision_score,
    roc_auc_score
)

from torch.utils.data import TensorDataset, DataLoader

# =====================================================
# 1. PATH CONFIGURATION
# =====================================================

BASE = os.path.dirname(os.path.abspath(__file__))

DATA = os.path.join(BASE, "datasets", "risk_training_data.csv")
MODEL_DIR = os.path.join(BASE, "models")
RESULTS_DIR = os.path.join(BASE, "results")

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

torch.manual_seed(42)
np.random.seed(42)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("\nEVAC-TWIN DEEP LEARNING RISK MODEL")
print("Device:", device)

# =====================================================
# 2. LOAD DATASET
# =====================================================

if not os.path.exists(DATA):
    raise FileNotFoundError(f"Dataset not found: {DATA}")

df = pd.read_csv(DATA)

features = [
    "temperature",
    "humidity",
    "smoke",
    "occupancy"
]

target = "risk_level"

df = df[features + [target]].copy()

df = df.replace(
    [np.inf, -np.inf],
    np.nan
)

df = df.dropna()
df = df.drop_duplicates()

X = df[features].values.astype(np.float32)

y_text = df[target].astype(str).str.upper()

encoder = LabelEncoder()
y = encoder.fit_transform(y_text)

num_classes = len(encoder.classes_)

print("\nDATASET SUMMARY")
print("-----------------------------")
print("Total Records:", len(df))
print("Total Features:", len(features))
print("Target Column:", target)
print("Risk Classes:", list(encoder.classes_))

print("\nClass Distribution:")
print(df[target].value_counts())

# =====================================================
# 3. SAVE DATASET SUMMARY
# =====================================================

dataset_summary = pd.DataFrame({
    "Details": [
        "Total Records",
        "Total Features",
        "Target Column",
        "Number of Risk Classes"
    ],
    "Values": [
        len(df),
        len(features),
        target,
        num_classes
    ]
})

dataset_summary.to_csv(
    os.path.join(RESULTS_DIR, "risk_dataset_summary.csv"),
    index=False
)

class_distribution = (
    pd.Series(y_text)
    .value_counts()
    .reindex(encoder.classes_, fill_value=0)
    .reset_index()
)

class_distribution.columns = [
    "Risk Level",
    "Records"
]

class_distribution.to_csv(
    os.path.join(RESULTS_DIR, "risk_class_distribution.csv"),
    index=False
)

# =====================================================
# 4. DATASET DISTRIBUTION GRAPH
# =====================================================

plt.figure(figsize=(8, 5))

plt.bar(
    class_distribution["Risk Level"],
    class_distribution["Records"]
)

plt.xlabel("Risk Level")
plt.ylabel("Number of Records")

plt.title("Risk Dataset Class Distribution")

plt.grid(axis="y")

plt.tight_layout()

plt.savefig(
    os.path.join(RESULTS_DIR, "risk_dataset_distribution.png"),
    dpi=300
)

plt.close()

# =====================================================
# 5. DATASET SPLITTING
# =====================================================

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

split_summary = pd.DataFrame({
    "Dataset": [
        "Training",
        "Validation",
        "Testing"
    ],
    "Records": [
        len(X_train),
        len(X_val),
        len(X_test)
    ],
    "Percentage": [
        len(X_train) / len(X) * 100,
        len(X_val) / len(X) * 100,
        len(X_test) / len(X) * 100
    ]
})

split_summary.to_csv(
    os.path.join(RESULTS_DIR, "risk_dataset_split.csv"),
    index=False
)

print("\nDATASET SPLIT")
print(split_summary.to_string(index=False))

# =====================================================
# 6. FEATURE SCALING
# =====================================================

scaler = StandardScaler()


#### backend/app/ml/train_risk_model.py — lines 221–440

In [ ]:

X_train = scaler.fit_transform(X_train).astype(np.float32)
X_val = scaler.transform(X_val).astype(np.float32)
X_test = scaler.transform(X_test).astype(np.float32)

# =====================================================
# 7. DEEP LEARNING MODEL
# =====================================================

class RiskNet(nn.Module):

    def __init__(self):

        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(4, 64),
            nn.ReLU(),
            nn.Dropout(0.25),

            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.20),

            nn.Linear(32, num_classes)
        )

    def forward(self, x):
        return self.net(x)

# =====================================================
# 8. CLASS WEIGHTS
# =====================================================

weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)

weights = torch.tensor(
    weights,
    dtype=torch.float32
).to(device)

model = RiskNet().to(device)

criterion = nn.CrossEntropyLoss(
    weight=weights
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.001
)

# =====================================================
# 9. DATA LOADERS
# =====================================================

def make_loader(x, y, shuffle=False):

    dataset = TensorDataset(
        torch.tensor(x, dtype=torch.float32),
        torch.tensor(y, dtype=torch.long)
    )

    return DataLoader(
        dataset,
        batch_size=128,
        shuffle=shuffle
    )

train_loader = make_loader(
    X_train,
    y_train,
    True
)

val_loader = make_loader(
    X_val,
    y_val
)

test_loader = make_loader(
    X_test,
    y_test
)

# =====================================================
# 10. MODEL TRAINING
# =====================================================

best_loss = float("inf")

patience = 12
wait = 0

train_losses = []
val_losses = []

train_accuracies = []
val_accuracies = []

best_epoch = 0

epochs = 100

print("\nTRAINING STARTED")
print("-----------------------------")

for epoch in range(epochs):

    model.train()

    total_train_loss = 0
    train_correct = 0
    train_total = 0

    for xb, yb in train_loader:

        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()

        outputs = model(xb)

        loss = criterion(outputs, yb)

        loss.backward()

        optimizer.step()

        total_train_loss += loss.item() * len(xb)

        predicted = outputs.argmax(dim=1)

        train_correct += (
            predicted == yb
        ).sum().item()

        train_total += len(yb)

    avg_train_loss = (
        total_train_loss / train_total
    )

    train_accuracy = (
        train_correct / train_total
    )

    # VALIDATION

    model.eval()

    total_val_loss = 0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for xb, yb in val_loader:

            xb = xb.to(device)
            yb = yb.to(device)

            outputs = model(xb)

            loss = criterion(outputs, yb)

            total_val_loss += loss.item() * len(xb)

            predicted = outputs.argmax(dim=1)

            val_correct += (
                predicted == yb
            ).sum().item()

            val_total += len(yb)

    val_loss = (
        total_val_loss / val_total
    )

    val_accuracy = (
        val_correct / val_total
    )

    train_losses.append(avg_train_loss)
    val_losses.append(val_loss)

    train_accuracies.append(train_accuracy)
    val_accuracies.append(val_accuracy)

    print(
        f"Epoch {epoch + 1}: "
        f"Training Loss = {avg_train_loss:.4f}, "
        f"Validation Loss = {val_loss:.4f}, "
        f"Training Accuracy = {train_accuracy:.4f}, "
        f"Validation Accuracy = {val_accuracy:.4f}"
    )

    if val_loss < best_loss:

        best_loss = val_loss
        best_epoch = epoch + 1

        wait = 0

        torch.save(
            model.state_dict(),
            os.path.join(MODEL_DIR, "risk_model.pth")
        )

    else:

        wait += 1

    if wait >= patience:


#### backend/app/ml/train_risk_model.py — lines 441–660

In [ ]:

        print("\nEarly stopping.")
        break

# =====================================================
# 11. SAVE TRAINING HISTORY
# =====================================================

history = pd.DataFrame({
    "Epoch": range(1, len(train_losses) + 1),
    "Training Loss": train_losses,
    "Validation Loss": val_losses,
    "Training Accuracy": train_accuracies,
    "Validation Accuracy": val_accuracies
})

history.to_csv(
    os.path.join(RESULTS_DIR, "risk_training_history.csv"),
    index=False
)

print("\nBest Epoch:", best_epoch)
print("Best Validation Loss:", round(best_loss, 4))

# =====================================================
# 12. TRAINING AND VALIDATION LOSS GRAPH
# =====================================================

plt.figure(figsize=(9, 5))

plt.plot(
    history["Epoch"],
    history["Training Loss"],
    label="Training Loss"
)

plt.plot(
    history["Epoch"],
    history["Validation Loss"],
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.title("Risk Model Training and Validation Loss")

plt.legend()
plt.grid(True)

plt.tight_layout()

plt.savefig(
    os.path.join(RESULTS_DIR, "risk_training_loss.png"),
    dpi=300
)

plt.close()

# =====================================================
# 13. TRAINING AND VALIDATION ACCURACY GRAPH
# =====================================================

plt.figure(figsize=(9, 5))

plt.plot(
    history["Epoch"],
    history["Training Accuracy"],
    label="Training Accuracy"
)

plt.plot(
    history["Epoch"],
    history["Validation Accuracy"],
    label="Validation Accuracy"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")

plt.title("Risk Model Training and Validation Accuracy")

plt.legend()
plt.grid(True)

plt.tight_layout()

plt.savefig(
    os.path.join(RESULTS_DIR, "risk_training_validation_accuracy.png"),
    dpi=300
)

plt.close()

# =====================================================
# 14. LOAD BEST MODEL
# =====================================================

model.load_state_dict(
    torch.load(
        os.path.join(MODEL_DIR, "risk_model.pth"),
        map_location=device
    )
)

model.eval()

# =====================================================
# 15. MODEL TESTING
# =====================================================

predictions = []
actual_values = []
probability_values = []

with torch.no_grad():

    for xb, yb in test_loader:

        xb = xb.to(device)

        outputs = model(xb)

        probabilities = torch.softmax(
            outputs,
            dim=1
        )

        predicted = outputs.argmax(dim=1)

        predictions.extend(
            predicted.cpu().numpy()
        )

        actual_values.extend(
            yb.numpy()
        )

        probability_values.extend(
            probabilities.cpu().numpy()
        )

predictions = np.array(predictions)

actual_values = np.array(actual_values)

probability_values = np.array(
    probability_values
)

# =====================================================
# 16. PERFORMANCE METRICS
# =====================================================

accuracy = accuracy_score(
    actual_values,
    predictions
)

precision = precision_score(
    actual_values,
    predictions,
    average="weighted",
    zero_division=0
)

recall = recall_score(
    actual_values,
    predictions,
    average="weighted",
    zero_division=0
)

f1 = f1_score(
    actual_values,
    predictions,
    average="weighted",
    zero_division=0
)

macro_f1 = f1_score(
    actual_values,
    predictions,
    average="macro",
    zero_division=0
)

# =====================================================
# 17. MULTICLASS ROC CURVE
# =====================================================

y_test_binary = label_binarize(
    actual_values,
    classes=np.arange(num_classes)
)

roc_values = []

plt.figure(figsize=(9, 7))

for i, class_name in enumerate(encoder.classes_):

    if len(np.unique(y_test_binary[:, i])) < 2:

        print(
            f"ROC skipped for {class_name}: "
            "test data does not contain both classes."
        )

        continue

    fpr, tpr, thresholds = roc_curve(
        y_test_binary[:, i],
        probability_values[:, i]
    )

    class_auc = auc(fpr, tpr)

    roc_values.append(class_auc)



#### backend/app/ml/train_risk_model.py — lines 661–880

In [ ]:
    plt.plot(
        fpr,
        tpr,
        label=f"{class_name} (AUC = {class_auc:.4f})"
    )

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random Classifier"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")

plt.title("Risk Prediction Multiclass ROC Curve")

plt.legend()
plt.grid(True)

plt.tight_layout()

plt.savefig(
    os.path.join(RESULTS_DIR, "risk_roc_curve.png"),
    dpi=300
)

plt.close()

roc_auc = (
    float(np.mean(roc_values))
    if roc_values else float("nan")
)

print("\nMean One-vs-Rest ROC AUC:", roc_auc)

# =====================================================
# 18. MULTICLASS PRECISION-RECALL CURVE
# =====================================================

plt.figure(figsize=(9, 7))

pr_values = []

for i, class_name in enumerate(encoder.classes_):

    if len(np.unique(y_test_binary[:, i])) < 2:
        continue

    pr_precision, pr_recall, thresholds = precision_recall_curve(
        y_test_binary[:, i],
        probability_values[:, i]
    )

    class_ap = average_precision_score(
        y_test_binary[:, i],
        probability_values[:, i]
    )

    pr_values.append(class_ap)

    plt.plot(
        pr_recall,
        pr_precision,
        label=f"{class_name} (AP = {class_ap:.4f})"
    )

plt.xlabel("Recall")
plt.ylabel("Precision")

plt.title("Risk Prediction Precision-Recall Curve")

plt.legend()
plt.grid(True)

plt.tight_layout()

plt.savefig(
    os.path.join(RESULTS_DIR, "risk_precision_recall_curve.png"),
    dpi=300
)

plt.close()

average_precision = (
    float(np.mean(pr_values))
    if pr_values else float("nan")
)

# =====================================================
# 19. SAVE PERFORMANCE METRICS
# =====================================================

metrics = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Weighted Precision",
        "Weighted Recall",
        "Weighted F1 Score",
        "Macro F1 Score",
        "Mean One-vs-Rest ROC AUC",
        "Mean Average Precision",
        "Best Epoch",
        "Best Validation Loss"
    ],
    "Value": [
        accuracy,
        precision,
        recall,
        f1,
        macro_f1,
        roc_auc,
        average_precision,
        best_epoch,
        best_loss
    ]
})

metrics.to_csv(
    os.path.join(RESULTS_DIR, "risk_performance_metrics.csv"),
    index=False
)

print("\nPERFORMANCE METRICS")
print("-----------------------------")
print(metrics.to_string(index=False))

# =====================================================
# 20. CLASSIFICATION REPORT
# =====================================================

report = classification_report(
    actual_values,
    predictions,
    labels=np.arange(num_classes),
    target_names=encoder.classes_,
    zero_division=0
)

print("\nCLASSIFICATION REPORT")
print("-----------------------------")
print(report)

with open(
    os.path.join(RESULTS_DIR, "risk_classification_report.txt"),
    "w",
    encoding="utf-8"
) as f:

    f.write("RISK MODEL CLASSIFICATION REPORT\n\n")
    f.write(report)

# =====================================================
# 21. CONFUSION MATRIX
# =====================================================

cm = confusion_matrix(
    actual_values,
    predictions,
    labels=np.arange(num_classes)
)

print("\nCONFUSION MATRIX")
print("-----------------------------")
print(cm)

cm_df = pd.DataFrame(
    cm,
    index=encoder.classes_,
    columns=encoder.classes_
)

cm_df.to_csv(
    os.path.join(RESULTS_DIR, "risk_confusion_matrix.csv")
)

plt.figure(figsize=(8, 6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=encoder.classes_,
    yticklabels=encoder.classes_
)

plt.xlabel("Predicted Risk Level")
plt.ylabel("Actual Risk Level")

plt.title("Risk Model Confusion Matrix")

plt.tight_layout()

plt.savefig(
    os.path.join(RESULTS_DIR, "risk_confusion_matrix.png"),
    dpi=300
)

plt.close()

# =====================================================
# 22. SAVE MODEL FILES
# =====================================================

joblib.dump(
    scaler,
    os.path.join(MODEL_DIR, "risk_scaler.pkl")
)

joblib.dump(
    encoder,
    os.path.join(MODEL_DIR, "risk_encoder.pkl")
)

config = {
    "features": features,
    "classes": encoder.classes_.tolist(),
    "input_size": 4,


#### backend/app/ml/train_risk_model.py — lines 881–969

In [ ]:
    "hidden_sizes": [64, 32],
    "num_classes": num_classes
}

with open(
    os.path.join(MODEL_DIR, "risk_config.json"),
    "w"
) as f:

    json.dump(
        config,
        f,
        indent=4
    )

# =====================================================
# 23. SAVE COMPLETE TRAINING SUMMARY
# =====================================================

with open(
    os.path.join(RESULTS_DIR, "risk_training_results.txt"),
    "w",
    encoding="utf-8"
) as f:

    f.write("RISK DEEP LEARNING MODEL TRAINING RESULTS\n")
    f.write("=" * 50 + "\n\n")

    f.write(f"Total Records: {len(df)}\n")
    f.write(f"Training Records: {len(X_train)}\n")
    f.write(f"Validation Records: {len(X_val)}\n")
    f.write(f"Testing Records: {len(X_test)}\n\n")

    f.write(f"Features: {features}\n")
    f.write(f"Classes: {encoder.classes_.tolist()}\n\n")

    f.write("Architecture: 4 -> 64 -> 32 -> Classes\n")
    f.write("Activation: ReLU\n")
    f.write("Optimizer: AdamW\n")
    f.write("Loss Function: Weighted Cross Entropy\n\n")

    f.write(f"Best Epoch: {best_epoch}\n")
    f.write(f"Best Validation Loss: {best_loss:.4f}\n\n")

    f.write(f"Accuracy: {accuracy:.4f}\n")
    f.write(f"Weighted Precision: {precision:.4f}\n")
    f.write(f"Weighted Recall: {recall:.4f}\n")
    f.write(f"Weighted F1 Score: {f1:.4f}\n")
    f.write(f"Macro F1 Score: {macro_f1:.4f}\n")
    f.write(f"Mean One-vs-Rest ROC AUC: {roc_auc:.4f}\n")
    f.write(f"Mean Average Precision: {average_precision:.4f}\n\n")

    f.write("DATASET SUMMARY\n")
    f.write(dataset_summary.to_string(index=False))

    f.write("\n\nDATASET SPLIT\n")
    f.write(split_summary.to_string(index=False))

    f.write("\n\nPERFORMANCE METRICS\n")
    f.write(metrics.to_string(index=False))

    f.write("\n\nCLASSIFICATION REPORT\n")
    f.write(report)

    f.write("\nCONFUSION MATRIX\n")
    f.write(str(cm))

# =====================================================
# 24. FINAL OUTPUT
# =====================================================

print("\nMODEL FILES SAVED:")
print("-----------------------------")

print("risk_model.pth")
print("risk_scaler.pkl")
print("risk_encoder.pkl")
print("risk_config.json")

print("\nRESULT FILES SAVED:")
print("-----------------------------")

for file in sorted(os.listdir(RESULTS_DIR)):

    if file.startswith("risk_"):

        print(file)

print("\nRisk DL Model Training Completed Successfully!")

### backend/app/ml/train_fire_model.py

**File:** `backend/app/ml/train_fire_model.py`

This is the actual project source copied into the notebook for inspection.
It is a reference section, not an instruction to execute the entire file in a notebook cell.

> This file has 1004 lines and is split into numbered reference sections of at most 220 lines. All lines are included.

#### backend/app/ml/train_fire_model.py — lines 1–220

In [ ]:

import copy
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_curve,
    auc,
    precision_recall_curve,
    average_precision_score
)
from sklearn.calibration import calibration_curve

from torch.utils.data import DataLoader, TensorDataset


BASE_DIR = Path(__file__).resolve().parent
DATA_PATH = BASE_DIR / "datasets" / "smoke_detection_iot.csv"
MODEL_DIR = BASE_DIR / "models"
RESULTS_DIR = BASE_DIR / "results"

MODEL_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

FEATURES = [
    "Temperature[C]",
    "Humidity[%]",
    "TVOC[ppb]",
    "eCO2[ppm]",
    "PM1.0",
    "PM2.5"
]

TARGET = "Fire Alarm"

torch.manual_seed(42)
np.random.seed(42)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


class FireModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(6, 64),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.network(x)


def temperature_calibration(logits, labels):

    temperature = nn.Parameter(
        torch.ones(1, device=device)
    )

    optimizer = torch.optim.LBFGS(
        [temperature],
        lr=0.01,
        max_iter=100
    )

    loss_function = nn.BCEWithLogitsLoss()

    def closure():

        optimizer.zero_grad()

        temp = torch.clamp(
            temperature,
            min=0.05,
            max=10
        )

        loss = loss_function(
            logits / temp,
            labels
        )

        loss.backward()

        return loss

    optimizer.step(closure)

    return float(
        torch.clamp(
            temperature,
            min=0.05,
            max=10
        ).item()
    )


print("\nEVAC-TWIN DEEP LEARNING FIRE MODEL")
print("Device:", device)


# DATASET LOADING

if not DATA_PATH.exists():

    raise FileNotFoundError(
        f"Dataset not found: {DATA_PATH}"
    )

df = pd.read_csv(DATA_PATH)

df = df[FEATURES + [TARGET]].copy()

df = df.replace(
    [np.inf, -np.inf],
    np.nan
)

df = df.dropna()

X = df[FEATURES].astype("float32")
y = df[TARGET].astype("int64")

print("\nTotal records:", len(df))

print("\nClass distribution:")
print(y.value_counts().sort_index())


# DATASET SUMMARY

dataset_summary = pd.DataFrame({
    "Details": [
        "Total Records",
        "Total Features",
        "Target Column",
        "No Fire Records",
        "Fire Records"
    ],
    "Values": [
        len(df),
        len(FEATURES),
        TARGET,
        int((df[TARGET] == 0).sum()),
        int((df[TARGET] == 1).sum())
    ]
})

print("\n===== DATASET SUMMARY =====")
print(dataset_summary.to_string(index=False))

dataset_summary.to_csv(
    RESULTS_DIR / "dataset_summary.csv",
    index=False
)


# DATASET DISTRIBUTION GRAPH

class_counts = y.value_counts().sort_index()

plt.figure(figsize=(7, 5))

plt.bar(
    ["No Fire", "Fire"],
    [
        class_counts.get(0, 0),
        class_counts.get(1, 0)
    ]
)

plt.title("Fire Detection Dataset Distribution")
plt.xlabel("Class")
plt.ylabel("Number of Records")

plt.tight_layout()

plt.savefig(
    RESULTS_DIR / "dataset_distribution.png",
    dpi=300
)

plt.close()


# DATASET SPLITTING

X_train, X_remaining, y_train, y_remaining = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)


#### backend/app/ml/train_fire_model.py — lines 221–440

In [ ]:

X_temp, X_test, y_temp, y_test = train_test_split(
    X_remaining,
    y_remaining,
    test_size=1/3,
    random_state=42,
    stratify=y_remaining
)

X_val, X_cal, y_val, y_cal = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)


split_table = pd.DataFrame({
    "Dataset": [
        "Training",
        "Validation",
        "Calibration",
        "Testing"
    ],
    "Records": [
        len(X_train),
        len(X_val),
        len(X_cal),
        len(X_test)
    ]
})

print("\n===== DATASET SPLIT =====")
print(split_table.to_string(index=False))

split_table.to_csv(
    RESULTS_DIR / "dataset_split.csv",
    index=False
)


# FEATURE SCALING

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_cal = scaler.transform(X_cal)
X_test = scaler.transform(X_test)


# DATA LOADER

def make_loader(X_data, y_data, shuffle=False):

    x_tensor = torch.tensor(
        X_data,
        dtype=torch.float32
    )

    y_tensor = torch.tensor(
        np.asarray(y_data).reshape(-1, 1),
        dtype=torch.float32
    )

    dataset = TensorDataset(
        x_tensor,
        y_tensor
    )

    return DataLoader(
        dataset,
        batch_size=256,
        shuffle=shuffle
    )


train_loader = make_loader(
    X_train, y_train, True
)

val_loader = make_loader(
    X_val, y_val
)

cal_loader = make_loader(
    X_cal, y_cal
)

test_loader = make_loader(
    X_test, y_test
)


# MODEL INITIALIZATION

model = FireModel().to(device)

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.001,
    weight_decay=0.0001
)

best_loss = float("inf")
best_weights = None

patience = 12
wait = 0
epochs = 100

train_losses = []
val_losses = []

train_accuracies = []
val_accuracies = []


# MODEL TRAINING

print("\nTraining started...\n")

for epoch in range(epochs):

    model.train()

    train_loss = 0
    train_correct = 0

    for xb, yb in train_loader:

        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()

        output = model(xb)

        loss = criterion(output, yb)

        loss.backward()

        optimizer.step()

        train_loss += loss.item() * len(xb)

        train_correct += (
            (torch.sigmoid(output) >= 0.5) == (yb >= 0.5)
        ).sum().item()

    train_loss /= len(train_loader.dataset)

    train_accuracy = (
        train_correct / len(train_loader.dataset)
    )

    # VALIDATION

    model.eval()

    val_loss = 0
    val_correct = 0

    with torch.no_grad():

        for xb, yb in val_loader:

            xb = xb.to(device)
            yb = yb.to(device)

            output = model(xb)

            loss = criterion(output, yb)

            val_loss += loss.item() * len(xb)

            val_correct += (
                (torch.sigmoid(output) >= 0.5) == (yb >= 0.5)
            ).sum().item()

    val_loss /= len(val_loader.dataset)

    val_accuracy = (
        val_correct / len(val_loader.dataset)
    )

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    train_accuracies.append(train_accuracy)
    val_accuracies.append(val_accuracy)

    if (epoch + 1) % 10 == 0:

        print(
            f"Epoch [{epoch + 1}/{epochs}] "
            f"Train Loss: {train_loss:.4f} "
            f"Validation Loss: {val_loss:.4f} "
            f"Train Accuracy: {train_accuracy:.4f} "
            f"Validation Accuracy: {val_accuracy:.4f}"
        )

    if val_loss < best_loss:

        best_loss = val_loss

        best_weights = copy.deepcopy(
            model.state_dict()
        )

        wait = 0

    else:

        wait += 1

    if wait >= patience:


#### backend/app/ml/train_fire_model.py — lines 441–660

In [ ]:

        print("\nEarly stopping applied.")

        break


model.load_state_dict(best_weights)


# TRAINING CONFIGURATION

print("\n===== MODEL TRAINING CONFIGURATION =====")

print("Architecture: 6 -> 64 -> 32 -> 1")
print("Activation Function: ReLU")
print("Dropout:", model.network[2].p)
print("Batch Size:", train_loader.batch_size)
print("Maximum Epochs:", epochs)
print("Early Stopping Patience:", patience)

print("\nOptimizer: AdamW")
print("Learning Rate:", optimizer.param_groups[0]["lr"])
print("Weight Decay:", optimizer.param_groups[0]["weight_decay"])

print("\nBest Validation Loss:", round(best_loss, 4))
print("Epochs Completed:", epoch + 1)


# TRAINING HISTORY CSV

history = pd.DataFrame({
    "Epoch": range(1, len(train_losses) + 1),
    "Training Loss": train_losses,
    "Validation Loss": val_losses,
    "Training Accuracy": train_accuracies,
    "Validation Accuracy": val_accuracies
})

history.to_csv(
    RESULTS_DIR / "training_history.csv",
    index=False
)


# TEMPERATURE CALIBRATION

model.eval()

cal_logits = []
cal_labels = []

with torch.no_grad():

    for xb, yb in cal_loader:

        xb = xb.to(device)

        logits = model(xb)

        cal_logits.append(logits)
        cal_labels.append(yb.to(device))


cal_logits = torch.cat(cal_logits)
cal_labels = torch.cat(cal_labels)

temperature = temperature_calibration(
    cal_logits,
    cal_labels
)

print(
    "\nCalibration temperature:",
    round(temperature, 4)
)


# FINAL TESTING

test_logits = []
test_labels = []

with torch.no_grad():

    for xb, yb in test_loader:

        xb = xb.to(device)

        logits = model(xb)

        test_logits.append(logits)
        test_labels.append(yb)


test_logits = torch.cat(test_logits)

test_labels = torch.cat(
    test_labels
).numpy().ravel()


probabilities = torch.sigmoid(
    test_logits / temperature
).cpu().numpy().ravel()

predictions = (
    probabilities >= 0.5
).astype(int)


# PERFORMANCE METRICS

accuracy = accuracy_score(
    test_labels,
    predictions
)

precision = precision_score(
    test_labels,
    predictions,
    zero_division=0
)

recall = recall_score(
    test_labels,
    predictions,
    zero_division=0
)

macro_f1 = f1_score(
    test_labels,
    predictions,
    average="macro"
)

fpr, tpr, roc_thresholds = roc_curve(
    test_labels,
    probabilities
)

roc_auc = auc(fpr, tpr)

pr_precision, pr_recall, pr_thresholds = precision_recall_curve(
    test_labels,
    probabilities
)

average_precision = average_precision_score(
    test_labels,
    probabilities
)

metrics_table = pd.DataFrame({
    "Metric": [
        "Accuracy (%)",
        "Precision",
        "Recall",
        "Macro F1 Score",
        "ROC AUC Score",
        "Average Precision",
        "Best Validation Loss",
        "Epochs Completed",
        "Calibration Temperature"
    ],
    "Value": [
        round(accuracy * 100, 2),
        round(precision, 4),
        round(recall, 4),
        round(macro_f1, 4),
        round(roc_auc, 4),
        round(average_precision, 4),
        round(best_loss, 4),
        epoch + 1,
        round(temperature, 4)
    ]
})

print("\n===== FINAL TEST RESULTS =====")
print(metrics_table.to_string(index=False))

metrics_table.to_csv(
    RESULTS_DIR / "performance_metrics.csv",
    index=False
)


# CLASSIFICATION REPORT

report = classification_report(
    test_labels,
    predictions,
    labels=[0, 1],
    target_names=["No Fire", "Fire"],
    zero_division=0
)

print("\n===== CLASSIFICATION REPORT =====")
print(report)

with open(
    RESULTS_DIR / "classification_report.txt",
    "w",
    encoding="utf-8"
) as f:

    f.write(report)


# CONFUSION MATRIX

cm = confusion_matrix(
    test_labels,
    predictions,
    labels=[0, 1]
)

print("\n===== CONFUSION MATRIX =====")
print(cm)

plt.figure(figsize=(6, 5))


#### backend/app/ml/train_fire_model.py — lines 661–880

In [ ]:

plt.imshow(
    cm,
    interpolation="nearest",
    cmap="Blues"
)

plt.title("Fire Detection Confusion Matrix")
plt.colorbar()

plt.xticks(
    [0, 1],
    ["No Fire", "Fire"]
)

plt.yticks(
    [0, 1],
    ["No Fire", "Fire"]
)

plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")

for i in range(2):

    for j in range(2):

        plt.text(
            j,
            i,
            str(cm[i, j]),
            ha="center",
            va="center",
            color="black",
            fontsize=13
        )

plt.tight_layout()

plt.savefig(
    RESULTS_DIR / "confusion_matrix.png",
    dpi=300
)

plt.close()


# ROC CURVE

plt.figure(figsize=(7, 5))

plt.plot(
    fpr,
    tpr,
    label=f"Fire Detection (AUC = {roc_auc:.4f})"
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random Classifier"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")

plt.title("Fire Detection ROC Curve")

plt.legend()
plt.grid(True)

plt.tight_layout()

plt.savefig(
    RESULTS_DIR / "roc_curve.png",
    dpi=300
)

plt.close()


# PRECISION RECALL CURVE

plt.figure(figsize=(7, 5))

plt.plot(
    pr_recall,
    pr_precision,
    label=f"Average Precision = {average_precision:.4f}"
)

plt.xlabel("Recall")
plt.ylabel("Precision")

plt.title("Fire Detection Precision-Recall Curve")

plt.legend()
plt.grid(True)

plt.tight_layout()

plt.savefig(
    RESULTS_DIR / "precision_recall_curve.png",
    dpi=300
)

plt.close()


# TRAINING AND VALIDATION LOSS

plt.figure(figsize=(8, 5))

plt.plot(
    range(1, len(train_losses) + 1),
    train_losses,
    label="Training Loss"
)

plt.plot(
    range(1, len(val_losses) + 1),
    val_losses,
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.title("Training and Validation Loss")

plt.legend()
plt.grid(True)

plt.tight_layout()

plt.savefig(
    RESULTS_DIR / "training_loss.png",
    dpi=300
)

plt.close()


# TRAINING AND VALIDATION ACCURACY

plt.figure(figsize=(8, 5))

plt.plot(
    range(1, len(train_accuracies) + 1),
    train_accuracies,
    label="Training Accuracy"
)

plt.plot(
    range(1, len(val_accuracies) + 1),
    val_accuracies,
    label="Validation Accuracy"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")

plt.title("Training and Validation Accuracy")

plt.legend()
plt.grid(True)

plt.tight_layout()

plt.savefig(
    RESULTS_DIR / "training_validation_accuracy.png",
    dpi=300
)

plt.close()


# CALIBRATION CURVE

fraction_positive, mean_predicted = calibration_curve(
    test_labels,
    probabilities,
    n_bins=10,
    strategy="uniform"
)

plt.figure(figsize=(7, 5))

plt.plot(
    mean_predicted,
    fraction_positive,
    marker="o",
    label="Fire Detection"
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Perfect Calibration"
)

plt.xlabel("Mean Predicted Probability")
plt.ylabel("Fraction of Positives")

plt.title("Fire Detection Calibration Curve")

plt.legend()
plt.grid(True)

plt.tight_layout()

plt.savefig(
    RESULTS_DIR / "calibration_curve.png",
    dpi=300
)

plt.close()



#### backend/app/ml/train_fire_model.py — lines 881–1004

In [ ]:

# SAVE MODEL

torch.save(
    model.state_dict(),
    MODEL_DIR / "fire_alarm_model.pth"
)

joblib.dump(
    scaler,
    MODEL_DIR / "fire_alarm_scaler.pkl"
)

with open(
    MODEL_DIR / "fire_alarm_config.json",
    "w"
) as f:

    json.dump(
        {
            "features": FEATURES,
            "temperature": temperature,
            "threshold": 0.5
        },
        f,
        indent=4
    )


# SAVE COMPLETE RESULTS

with open(
    RESULTS_DIR / "training_results.txt",
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "EVAC-TWIN DEEP LEARNING FIRE DETECTION RESULTS\n"
    )

    f.write("=" * 55 + "\n\n")

    f.write(f"Total Records: {len(df)}\n")
    f.write(f"Device: {device}\n")
    f.write("Architecture: 6 -> 64 -> 32 -> 1\n")
    f.write("Activation: ReLU\n")
    f.write("Optimizer: AdamW\n")

    f.write(
        f"Learning Rate: {optimizer.param_groups[0]['lr']}\n"
    )

    f.write(
        f"Best Validation Loss: {best_loss:.4f}\n"
    )

    f.write(
        f"Epochs Completed: {epoch + 1}\n"
    )

    f.write(
        f"Calibration Temperature: {temperature:.4f}\n"
    )

    f.write(
        f"Accuracy: {accuracy * 100:.2f}%\n"
    )

    f.write(
        f"Precision: {precision:.4f}\n"
    )

    f.write(
        f"Recall: {recall:.4f}\n"
    )

    f.write(
        f"Macro F1 Score: {macro_f1:.4f}\n"
    )

    f.write(
        f"ROC AUC Score: {roc_auc:.4f}\n"
    )

    f.write(
        f"Average Precision: {average_precision:.4f}\n\n"
    )

    f.write("DATASET SUMMARY\n")
    f.write(dataset_summary.to_string(index=False))

    f.write("\n\nDATASET SPLIT\n")
    f.write(split_table.to_string(index=False))

    f.write("\n\nPERFORMANCE METRICS\n")
    f.write(metrics_table.to_string(index=False))

    f.write("\n\nCLASSIFICATION REPORT\n")
    f.write(report)

    f.write("\n\nCONFUSION MATRIX\n")
    f.write(str(cm))


# FINAL OUTPUT

print("\n===== SAVED MODEL FILES =====")

print("fire_alarm_model.pth")
print("fire_alarm_scaler.pkl")
print("fire_alarm_config.json")

print("\n===== SAVED RESULT FILES =====")

for file in sorted(RESULTS_DIR.iterdir()):

    if file.is_file():

        print(file.name)

print("\nResults folder:", RESULTS_DIR)

print("\nTraining completed successfully.")


### backend/app/ml/train_model.py

**File:** `backend/app/ml/train_model.py`

This is the actual project source copied into the notebook for inspection.
It is a reference section, not an instruction to execute the entire file in a notebook cell.

> This file has 493 lines and is split into numbered reference sections of at most 220 lines. All lines are included.

#### backend/app/ml/train_model.py — lines 1–220

In [ ]:

import os
import numpy as np
import torch
import torch.nn as nn
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

BASE_DIR = os.path.dirname(os.path.abspath(__file__))

MODEL_DIR = os.path.join(BASE_DIR, "models")

MODEL_PATH = os.path.join(MODEL_DIR, "risk_model.pth")
SCALER_PATH = os.path.join(MODEL_DIR, "risk_scaler.pkl")

FEATURE_NAMES = [
    "temperature",
    "humidity",
    "smoke",
    "occupancy"
]

CLASSES = ["LOW", "MODERATE", "HIGH", "CRITICAL"]


def calculate_training_risk(temperature, smoke, occupancy):

    score = 0

    if temperature >= 50:
        score += 35
    elif temperature >= 45:
        score += 30
    elif temperature >= 40:
        score += 25
    elif temperature >= 35:
        score += 15
    elif temperature >= 32:
        score += 5

    if smoke >= 0.80:
        score += 40
    elif smoke >= 0.60:
        score += 35
    elif smoke >= 0.50:
        score += 30
    elif smoke >= 0.30:
        score += 15
    elif smoke >= 0.15:
        score += 5

    if occupancy >= 50:
        score += 25
    elif occupancy >= 40:
        score += 20
    elif occupancy >= 30:
        score += 15
    elif occupancy >= 15:
        score += 5

    score = min(score, 100)

    if score >= 70:
        return 3
    elif score >= 40:
        return 2
    elif score >= 20:
        return 1

    return 0


def create_training_data(samples=8000):

    rng = np.random.default_rng(42)

    temperature = rng.uniform(25, 55, samples)
    humidity = rng.uniform(25, 80, samples)
    smoke = rng.uniform(0, 1, samples)
    occupancy = rng.integers(0, 51, samples)

    X = np.column_stack((
        temperature,
        humidity,
        smoke,
        occupancy
    ))

    y = np.array([
        calculate_training_risk(t, s, o)
        for t, s, o in zip(temperature, smoke, occupancy)
    ])

    return X, y


class RiskMLP(nn.Module):

    def __init__(self):

        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(4, 64),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 4)
        )

    def forward(self, x):

        return self.network(x)


def train_model():

    print("\nEVAC-TWIN DEEP LEARNING RISK MODEL")

    X, y = create_training_data()

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42,
        stratify=y
    )

    scaler = StandardScaler()

    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    X_train = torch.tensor(X_train, dtype=torch.float32)
    X_test = torch.tensor(X_test, dtype=torch.float32)

    y_train = torch.tensor(y_train, dtype=torch.long)
    y_test = torch.tensor(y_test, dtype=torch.long)

    model = RiskMLP()

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=0.001
    )

    epochs = 100

    for epoch in range(epochs):

        model.train()

        optimizer.zero_grad()

        outputs = model(X_train)

        loss = criterion(outputs, y_train)

        loss.backward()

        optimizer.step()

        if (epoch + 1) % 10 == 0:
            print(
                f"Epoch [{epoch + 1}/{epochs}] "
                f"Loss: {loss.item():.4f}"
            )

    model.eval()

    with torch.no_grad():

        outputs = model(X_test)

        predictions = torch.argmax(
            outputs,
            dim=1
        )

    accuracy = accuracy_score(
        y_test.numpy(),
        predictions.numpy()
    )

    print("\nTest Accuracy:", round(accuracy * 100, 2), "%")

    print("\nClassification Report:")

    print(classification_report(
        y_test.numpy(),
        predictions.numpy(),
        labels=[0, 1, 2, 3],
        target_names=CLASSES,
        zero_division=0
    ))

    os.makedirs(MODEL_DIR, exist_ok=True)

    torch.save(
        model.state_dict(),
        MODEL_PATH
    )

    joblib.dump(
        scaler,
        SCALER_PATH
    )
import json
import copy
from pathlib import Path



#### backend/app/ml/train_model.py — lines 221–440

In [ ]:
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from torch.utils.data import DataLoader, TensorDataset


BASE_DIR = Path(__file__).resolve().parent
DATA_PATH = BASE_DIR / "datasets" / "risk_training_data.csv"
MODEL_DIR = BASE_DIR / "models"
MODEL_DIR.mkdir(exist_ok=True)

FEATURES = ["temperature", "humidity", "smoke", "occupancy"]
CLASSES = ["LOW", "MODERATE", "HIGH", "CRITICAL"]

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def calculate_risk(row):
    score = 0

    if row["temperature"] >= 50:
        score += 35
    elif row["temperature"] >= 45:
        score += 30
    elif row["temperature"] >= 40:
        score += 25
    elif row["temperature"] >= 35:
        score += 15
    elif row["temperature"] >= 32:
        score += 5

    if row["smoke"] >= 0.80:
        score += 40
    elif row["smoke"] >= 0.60:
        score += 35
    elif row["smoke"] >= 0.50:
        score += 30
    elif row["smoke"] >= 0.30:
        score += 15
    elif row["smoke"] >= 0.15:
        score += 5

    if row["occupancy"] >= 50:
        score += 25
    elif row["occupancy"] >= 40:
        score += 20
    elif row["occupancy"] >= 30:
        score += 15
    elif row["occupancy"] >= 15:
        score += 5

    if score >= 70:
        return 3
    elif score >= 40:
        return 2
    elif score >= 20:
        return 1
    return 0


class RiskMLP(nn.Module):
    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(4, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 4)
        )

    def forward(self, x):
        return self.network(x)


def make_loader(X, y, shuffle=False):
    dataset = TensorDataset(
        torch.tensor(X, dtype=torch.float32),
        torch.tensor(y, dtype=torch.long)
    )
    return DataLoader(dataset, batch_size=256, shuffle=shuffle)


def main():
    print("\nEVAC-TWIN DEEP LEARNING RISK MODEL")
    print("Device:", device)

    if not DATA_PATH.exists():
        raise FileNotFoundError(
            f"Create the prepared dataset first: {DATA_PATH}"
        )

    df = pd.read_csv(DATA_PATH)

    missing = [c for c in FEATURES if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}")

    df = df[FEATURES].replace([np.inf, -np.inf], np.nan)
    df = df.dropna()

    if len(df) < 100:
        raise ValueError("Collect more real sensor records before training.")

    if (df["occupancy"] < 0).any():
        raise ValueError("Occupancy cannot be negative.")

    if (df["smoke"] < 0).any():
        raise ValueError("Smoke values cannot be negative.")

    # Derived risk labels: these are rule-based labels, not original labels.
    y = df.apply(calculate_risk, axis=1).to_numpy(dtype=np.int64)
    X = df[FEATURES].to_numpy(dtype=np.float32)

    print("Total records:", len(df))
    print("\nRisk label distribution:")
    print(pd.Series(y).value_counts().sort_index())

    # 70% training, 10% validation, 10% calibration, 10% testing
    X_train, X_rem, y_train, y_rem = train_test_split(
        X, y, test_size=0.30, random_state=42, stratify=y
    )

    X_temp, X_test, y_temp, y_test = train_test_split(
        X_rem, y_rem, test_size=1/3, random_state=42, stratify=y_rem
    )

    X_val, X_cal, y_val, y_cal = train_test_split(
        X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
    )

    print("\nDataset split:")
    print("Training:", len(X_train))
    print("Validation:", len(X_val))
    print("Calibration:", len(X_cal))
    print("Testing:", len(X_test))

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train).astype(np.float32)
    X_val = scaler.transform(X_val).astype(np.float32)
    X_cal = scaler.transform(X_cal).astype(np.float32)
    X_test = scaler.transform(X_test).astype(np.float32)

    train_loader = make_loader(X_train, y_train, True)
    val_loader = make_loader(X_val, y_val)
    cal_loader = make_loader(X_cal, y_cal)
    test_loader = make_loader(X_test, y_test)

    model = RiskMLP().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=0.001, weight_decay=0.0001
    )

    best_loss = float("inf")
    best_weights = None
    patience = 12
    wait = 0

    for epoch in range(100):
        model.train()
        total_loss = 0

        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(xb)

        train_loss = total_loss / len(train_loader.dataset)

        model.eval()
        val_loss = 0

        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                loss = criterion(model(xb), yb)
                val_loss += loss.item() * len(xb)

        val_loss /= len(val_loader.dataset)

        if (epoch + 1) % 10 == 0:
            print(
                f"Epoch [{epoch + 1}/100] "
                f"Train Loss: {train_loss:.4f} "
                f"Validation Loss: {val_loss:.4f}"
            )

        if val_loss < best_loss:
            best_loss = val_loss
            best_weights = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1

        if wait >= patience:
            print("Early stopping.")
            break

    model.load_state_dict(best_weights)

    # Calibration records are kept separate from training.
    model.eval()
    calibration_count = 0
    with torch.no_grad():


#### backend/app/ml/train_model.py — lines 441–493

In [ ]:
        for xb, yb in cal_loader:
            xb = xb.to(device)
            calibration_count += len(xb)

    # Test evaluation
    predictions = []
    actual = []

    with torch.no_grad():
        for xb, yb in test_loader:
            logits = model(xb.to(device))
            predictions.extend(torch.argmax(logits, dim=1).cpu().numpy())
            actual.extend(yb.numpy())

    print("\nTest Accuracy:",
          round(accuracy_score(actual, predictions) * 100, 2), "%")

    print("\nClassification Report:")
    print(classification_report(
        actual,
        predictions,
        labels=[0, 1, 2, 3],
        target_names=CLASSES,
        zero_division=0
    ))

    print("Confusion Matrix:")
    print(confusion_matrix(actual, predictions, labels=[0, 1, 2, 3]))

    torch.save(model.state_dict(), MODEL_DIR / "risk_model.pth")
    joblib.dump(scaler, MODEL_DIR / "risk_scaler.pkl")

    with open(MODEL_DIR / "risk_config.json", "w") as f:
        json.dump({
            "features": FEATURES,
            "classes": CLASSES,
            "label_method": "domain_based_derived_labels",
            "calibration_records": calibration_count
        }, f, indent=4)

    print("\nRisk DL model saved successfully.")


if __name__ == "__main__":
    main()


    print("\nDeep Learning model saved:", MODEL_PATH)
    print("Scaler saved:", SCALER_PATH)


if __name__ == "__main__":
    train_model()


### backend/app/ml/compare_models.py

**File:** `backend/app/ml/compare_models.py`

This is the actual project source copied into the notebook for inspection.
It is a reference section, not an instruction to execute the entire file in a notebook cell.

> This file has 669 lines and is split into numbered reference sections of at most 220 lines. All lines are included.

#### backend/app/ml/compare_models.py — lines 1–220

In [ ]:
import os
import copy
import warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from pytorch_tabnet.tab_model import TabNetClassifier

warnings.filterwarnings("ignore")

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
DATA_PATH = os.path.join(
    BASE_DIR, "datasets", "smoke_detection_iot.csv"
)

FEATURES = [
    "Temperature[C]",
    "Humidity[%]",
    "TVOC[ppb]",
    "eCO2[ppm]",
    "PM1.0",
    "PM2.5"
]

TARGET = "Fire Alarm"

RANDOM_STATE = 42
EPOCHS = 60
PATIENCE = 10
BATCH_SIZE = 512

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


# --------------------------------------------------
# FT-TRANSFORMER MODEL
# --------------------------------------------------

class FTTransformer(nn.Module):

    def __init__(self, n_features, d_model=32, heads=4):

        super().__init__()

        self.weight = nn.Parameter(
            torch.randn(n_features, d_model) * 0.02
        )

        self.bias = nn.Parameter(
            torch.zeros(n_features, d_model)
        )

        self.cls_token = nn.Parameter(
            torch.zeros(1, 1, d_model)
        )

        self.encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=heads,
            dim_feedforward=64,
            dropout=0.1,
            batch_first=True,
            activation="gelu"
        )

        self.encoder = nn.TransformerEncoder(
            self.encoder_layer,
            num_layers=2
        )

        self.output = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 1)
        )

    def forward(self, x):

        tokens = (
            x.unsqueeze(-1) * self.weight.unsqueeze(0)
            + self.bias.unsqueeze(0)
        )

        cls = self.cls_token.expand(
            x.size(0), -1, -1
        )

        tokens = torch.cat([cls, tokens], dim=1)
        encoded = self.encoder(tokens)

        return self.output(encoded[:, 0]).squeeze(1)


# --------------------------------------------------
# SAINT-STYLE TABULAR TRANSFORMER
# --------------------------------------------------

class SAINTStyle(nn.Module):

    def __init__(self, n_features, d_model=32, heads=4):

        super().__init__()

        self.weight = nn.Parameter(
            torch.randn(n_features, d_model) * 0.02
        )

        self.bias = nn.Parameter(
            torch.zeros(n_features, d_model)
        )

        self.attention = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=heads,
            dropout=0.1,
            batch_first=True
        )

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, 64),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(64, d_model)
        )

        self.output = nn.Sequential(
            nn.Linear(d_model, 32),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(32, 1)
        )

    def forward(self, x):

        tokens = (
            x.unsqueeze(-1) * self.weight.unsqueeze(0)
            + self.bias.unsqueeze(0)
        )

        attended, _ = self.attention(
            tokens, tokens, tokens
        )

        tokens = self.norm1(tokens + attended)
        tokens = self.norm2(
            tokens + self.feed_forward(tokens)
        )

        pooled = tokens.mean(dim=1)

        return self.output(pooled).squeeze(1)


# --------------------------------------------------
# PYTORCH TRAINING FUNCTION
# --------------------------------------------------

def train_torch_model(model, X_train, y_train,
                      X_val, y_val):

    model = model.to(DEVICE)

    train_x = torch.tensor(
        X_train, dtype=torch.float32
    )

    train_y = torch.tensor(
        y_train, dtype=torch.float32
    )

    val_x = torch.tensor(
        X_val, dtype=torch.float32
    ).to(DEVICE)

    val_y = torch.tensor(
        y_val, dtype=torch.float32
    ).to(DEVICE)

    loader = DataLoader(
        TensorDataset(train_x, train_y),
        batch_size=BATCH_SIZE,
        shuffle=True
    )

    optimizer = torch.optim.Adam(
        model.parameters(), lr=0.001
    )

    loss_function = nn.BCEWithLogitsLoss()

    best_loss = float("inf")
    best_weights = None
    patience_count = 0
    best_epoch = 0



#### backend/app/ml/compare_models.py — lines 221–440

In [ ]:
    for epoch in range(EPOCHS):

        model.train()
        total_loss = 0

        for batch_x, batch_y in loader:

            batch_x = batch_x.to(DEVICE)
            batch_y = batch_y.to(DEVICE)

            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = loss_function(outputs, batch_y)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        model.eval()

        with torch.no_grad():
            val_outputs = model(val_x)
            val_loss = loss_function(
                val_outputs, val_y
            ).item()

        if val_loss < best_loss:

            best_loss = val_loss
            best_weights = copy.deepcopy(
                model.state_dict()
            )
            best_epoch = epoch + 1
            patience_count = 0

        else:
            patience_count += 1

        if (epoch + 1) % 10 == 0:
            print(
                "Epoch:", epoch + 1,
                "Train Loss:", round(
                    total_loss / len(loader), 4
                ),
                "Validation Loss:", round(val_loss, 4)
            )

        if patience_count >= PATIENCE:
            print("Early stopping at epoch:", epoch + 1)
            break

    if best_weights is not None:
        model.load_state_dict(best_weights)

    return model, best_epoch


def torch_predict(model, X):

    model.eval()

    values = torch.tensor(
        X, dtype=torch.float32
    ).to(DEVICE)

    with torch.no_grad():
        logits = model(values)
        probabilities = torch.sigmoid(logits)

    return probabilities.cpu().numpy()


# --------------------------------------------------
# PROBABILITY CALIBRATION
# --------------------------------------------------

def calibrate_predictions(cal_prob, y_cal, test_prob):

    cal_prob = np.clip(cal_prob, 1e-6, 1 - 1e-6)
    test_prob = np.clip(test_prob, 1e-6, 1 - 1e-6)

    cal_logits = np.log(
        cal_prob / (1 - cal_prob)
    ).reshape(-1, 1)

    test_logits = np.log(
        test_prob / (1 - test_prob)
    ).reshape(-1, 1)

    calibrator = LogisticRegression(max_iter=1000)
    calibrator.fit(cal_logits, y_cal)

    return calibrator.predict(test_logits)


# --------------------------------------------------
# EVALUATION FUNCTION
# --------------------------------------------------

def evaluate_model(name, y_test, predictions):

    accuracy = accuracy_score(y_test, predictions)
    precision = precision_score(
        y_test, predictions, zero_division=0
    )
    recall = recall_score(
        y_test, predictions, zero_division=0
    )
    f1 = f1_score(
        y_test, predictions, zero_division=0
    )

    print("\n", name)
    print(
        classification_report(
            y_test, predictions, zero_division=0
        )
    )

    print("Confusion Matrix:")
    print(confusion_matrix(y_test, predictions))

    return {
        "Model": name,
        "Accuracy": round(accuracy * 100, 2),
        "Precision": round(precision * 100, 2),
        "Recall": round(recall * 100, 2),
        "F1 Score": round(f1 * 100, 2)
    }


# --------------------------------------------------
# MAIN COMPARISON
# --------------------------------------------------

def compare_models():

    print("\nEVAC-TWIN - ML AND DL MODEL COMPARISON")
    print("Device:", DEVICE)

    # Load and clean dataset

    data = pd.read_csv(DATA_PATH)
    data = data[FEATURES + [TARGET]].copy()
    data = data.replace([np.inf, -np.inf], np.nan)
    data = data.dropna()

    X = data[FEATURES]
    y = data[TARGET].astype(int)

    print("\nTotal samples:", len(X))

    # Same split is used for all eight models:
    # 70% training, 10% validation,
    # 10% calibration, 10% testing.

    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y,
        test_size=0.30,
        random_state=RANDOM_STATE,
        stratify=y
    )

    X_val, X_temp, y_val, y_temp = train_test_split(
        X_temp, y_temp,
        test_size=2/3,
        random_state=RANDOM_STATE,
        stratify=y_temp
    )

    X_cal, X_test, y_cal, y_test = train_test_split(
        X_temp, y_temp,
        test_size=0.50,
        random_state=RANDOM_STATE,
        stratify=y_temp
    )

    print("\nDATA SPLIT")
    print("Training:", len(X_train))
    print("Validation:", len(X_val))
    print("Calibration:", len(X_cal))
    print("Testing:", len(X_test))

    # Scaling is fitted only on training data.
    # The same scaled splits are used by DL models.

    scaler = StandardScaler()

    X_train_s = scaler.fit_transform(X_train)
    X_val_s = scaler.transform(X_val)
    X_cal_s = scaler.transform(X_cal)
    X_test_s = scaler.transform(X_test)

    results = []

    # --------------------------------------------------
    # 1-4. TRADITIONAL ML MODELS
    # --------------------------------------------------

    models = {

        "Random Forest": RandomForestClassifier(
            n_estimators=150,
            max_depth=16,
            min_samples_leaf=2,
            random_state=RANDOM_STATE,
            class_weight="balanced",
            n_jobs=-1
        ),

        "Logistic Regression": make_pipeline(
            StandardScaler(),
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced"
            )
        ),

        "KNN": make_pipeline(
            StandardScaler(),


#### backend/app/ml/compare_models.py — lines 441–660

In [ ]:
            KNeighborsClassifier(n_neighbors=5)
        ),

        "SVM": make_pipeline(
            StandardScaler(),
            SVC(
                class_weight="balanced",
                probability=True,
                random_state=RANDOM_STATE
            )
        )
    }

    for name, model in models.items():

        print("\nTraining:", name)

        # All models learn from exactly X_train, y_train.
        model.fit(X_train, y_train)

        # All models use the same calibration and test sets.
        cal_prob = model.predict_proba(X_cal)[:, 1]
        test_prob = model.predict_proba(X_test)[:, 1]

        predictions = calibrate_predictions(
            cal_prob, y_cal, test_prob
        )

        results.append(
            evaluate_model(name, y_test, predictions)
        )

    # --------------------------------------------------
    # 5. MLP
    # --------------------------------------------------

    print("\nTraining: Deep Learning MLP")

    mlp = make_pipeline(
        StandardScaler(),
        MLPClassifier(
            hidden_layer_sizes=(128, 64),
            activation="relu",
            solver="adam",
            alpha=0.0001,
            learning_rate_init=0.001,
            batch_size=256,
            max_iter=100,
            early_stopping=True,
            random_state=RANDOM_STATE
        )
    )

    # MLP also learns from the same X_train, y_train.
    mlp.fit(X_train, y_train)

    cal_prob = mlp.predict_proba(X_cal)[:, 1]
    test_prob = mlp.predict_proba(X_test)[:, 1]

    predictions = calibrate_predictions(
        cal_prob, y_cal, test_prob
    )

    results.append(
        evaluate_model(
            "Deep Learning MLP", y_test, predictions
        )
    )

    # --------------------------------------------------
    # 6. TABNET
    # --------------------------------------------------

    print("\nTraining: TabNet")

    tabnet = TabNetClassifier(
        n_d=16,
        n_a=16,
        n_steps=5,
        gamma=1.5,
        lambda_sparse=1e-4,
        optimizer_params={"lr": 0.02},
        seed=RANDOM_STATE,
        verbose=10
    )

    tabnet.fit(
        X_train=X_train_s.astype(np.float32),
        y_train=y_train.to_numpy(),
        eval_set=[
            (
                X_train_s.astype(np.float32),
                y_train.to_numpy()
            ),
            (
                X_val_s.astype(np.float32),
                y_val.to_numpy()
            )
        ],
        eval_name=["train", "validation"],
        eval_metric=["auc"],
        max_epochs=100,
        patience=15,
        batch_size=1024,
        virtual_batch_size=128,
        num_workers=0,
        drop_last=False
    )

    cal_prob = tabnet.predict_proba(
        X_cal_s.astype(np.float32)
    )[:, 1]

    test_prob = tabnet.predict_proba(
        X_test_s.astype(np.float32)
    )[:, 1]

    predictions = calibrate_predictions(
        cal_prob, y_cal, test_prob
    )

    results.append(
        evaluate_model("TabNet", y_test, predictions)
    )

    # --------------------------------------------------
    # 7. FT-TRANSFORMER
    # --------------------------------------------------

    print("\nTraining: FT-Transformer")

    ft_model = FTTransformer(
        n_features=len(FEATURES)
    )

    ft_model, ft_epochs = train_torch_model(
        ft_model,
        X_train_s.astype(np.float32),
        y_train.to_numpy().astype(np.float32),
        X_val_s.astype(np.float32),
        y_val.to_numpy().astype(np.float32)
    )

    cal_prob = torch_predict(
        ft_model, X_cal_s.astype(np.float32)
    )

    test_prob = torch_predict(
        ft_model, X_test_s.astype(np.float32)
    )

    predictions = calibrate_predictions(
        cal_prob, y_cal, test_prob
    )

    results.append(
        evaluate_model(
            "FT-Transformer", y_test, predictions
        )
    )

    print("FT-Transformer best epoch:", ft_epochs)

    # --------------------------------------------------
    # 8. SAINT-STYLE TRANSFORMER
    # --------------------------------------------------

    print("\nTraining: SAINT-style Transformer")

    saint_model = SAINTStyle(
        n_features=len(FEATURES)
    )

    saint_model, saint_epochs = train_torch_model(
        saint_model,
        X_train_s.astype(np.float32),
        y_train.to_numpy().astype(np.float32),
        X_val_s.astype(np.float32),
        y_val.to_numpy().astype(np.float32)
    )

    cal_prob = torch_predict(
        saint_model, X_cal_s.astype(np.float32)
    )

    test_prob = torch_predict(
        saint_model, X_test_s.astype(np.float32)
    )

    predictions = calibrate_predictions(
        cal_prob, y_cal, test_prob
    )

    results.append(
        evaluate_model(
            "SAINT-style Transformer", y_test, predictions
        )
    )

    print("SAINT-style best epoch:", saint_epochs)

    # --------------------------------------------------
    # FINAL COMPARISON
    # --------------------------------------------------

    results_df = pd.DataFrame(results)

    results_df = results_df.sort_values(
        by="F1 Score",
        ascending=False
    )

    print("\nFINAL MODEL COMPARISON")
    print(results_df.to_string(index=False))

    print("\nBEST MODEL:")
    print(results_df.iloc[0]["Model"])

    output_path = os.path.join(
        BASE_DIR, "model_comparison_results.csv"


#### backend/app/ml/compare_models.py — lines 661–669

In [ ]:
    )

    results_df.to_csv(output_path, index=False)

    print("\nResults saved to:", output_path)


if __name__ == "__main__":
    compare_models()


### backend/app/ml/compare_dl_models.py

**File:** `backend/app/ml/compare_dl_models.py`

This is the actual project source copied into the notebook for inspection.
It is a reference section, not an instruction to execute the entire file in a notebook cell.

In [ ]:

import os
import warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix
)

from pytorch_tabnet.tab_model import TabNetClassifier

warnings.filterwarnings("ignore")

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
DATA_PATH = os.path.join(
    BASE_DIR, "datasets", "smoke_detection_iot.csv"
)

FEATURES = [
    "Temperature[C]", "Humidity[%]", "TVOC[ppb]",
    "eCO2[ppm]", "PM1.0", "PM2.5"
]

TARGET = "Fire Alarm"


def compare_dl_models():

    print("\nEVAC-TWIN - DEEP LEARNING MODEL COMPARISON")

    # Load and clean dataset
    data = pd.read_csv(DATA_PATH)
    data = data[FEATURES + [TARGET]].copy()
    data = data.replace([np.inf, -np.inf], np.nan)
    data = data.dropna()

    X = data[FEATURES].astype(np.float32)
    y = data[TARGET].astype(int)

    print("Total samples:", len(X))

    # 70% training, 10% validation,
    # 10% calibration, 10% testing
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=0.30,
        random_state=42, stratify=y
    )

    X_val, X_temp, y_val, y_temp = train_test_split(
        X_temp, y_temp, test_size=2/3,
        random_state=42, stratify=y_temp
    )

    X_cal, X_test, y_cal, y_test = train_test_split(
        X_temp, y_temp, test_size=0.50,
        random_state=42, stratify=y_temp
    )

    print("\nDATA SPLIT")
    print("Training:", len(X_train))
    print("Validation:", len(X_val))
    print("Calibration:", len(X_cal))
    print("Testing:", len(X_test))

    # Convert data into NumPy arrays
    X_train_np = X_train.to_numpy()
    X_val_np = X_val.to_numpy()
    X_cal_np = X_cal.to_numpy()
    X_test_np = X_test.to_numpy()

    y_train_np = y_train.to_numpy()
    y_val_np = y_val.to_numpy()
    y_cal_np = y_cal.to_numpy()

    # TabNet model
    model = TabNetClassifier(
        n_d=16,
        n_a=16,
        n_steps=5,
        gamma=1.5,
        lambda_sparse=1e-4,
        optimizer_params=dict(lr=0.02),
        seed=42,
        verbose=10
    )

    print("\nTRAINING TABNET")

    model.fit(
        X_train=X_train_np,
        y_train=y_train_np,
        eval_set=[
            (X_train_np, y_train_np),
            (X_val_np, y_val_np)
        ],
        eval_name=["train", "validation"],
        eval_metric=["auc"],
        max_epochs=100,
        patience=15,
        batch_size=1024,
        virtual_batch_size=128,
        num_workers=0,
        drop_last=False,
        weights=1
    )

    # Calibration using separate calibration data
    raw_prob = model.predict_proba(X_cal_np)[:, 1]
    raw_prob = np.clip(raw_prob, 1e-6, 1 - 1e-6)

    logits = np.log(raw_prob / (1 - raw_prob)).reshape(-1, 1)

    calibrator = LogisticRegression()
    calibrator.fit(logits, y_cal_np)

    # Final testing
    test_prob = model.predict_proba(X_test_np)[:, 1]
    test_prob = np.clip(test_prob, 1e-6, 1 - 1e-6)

    test_logits = np.log(
        test_prob / (1 - test_prob)
    ).reshape(-1, 1)

    predictions = calibrator.predict(test_logits)

    # Evaluation
    accuracy = accuracy_score(y_test, predictions)
    precision = precision_score(
        y_test, predictions, zero_division=0
    )
    recall = recall_score(
        y_test, predictions, zero_division=0
    )
    f1 = f1_score(
        y_test, predictions, zero_division=0
    )

    print("\nTABNET CLASSIFICATION REPORT")
    print(classification_report(
        y_test, predictions, zero_division=0
    ))

    print("Confusion Matrix:")
    print(confusion_matrix(y_test, predictions))

    results = pd.DataFrame([{
        "Model": "TabNet",
        "Accuracy": round(accuracy * 100, 2),
        "Precision": round(precision * 100, 2),
        "Recall": round(recall * 100, 2),
        "F1 Score": round(f1 * 100, 2)
    }])

    print("\nTABNET RESULTS")
    print(results.to_string(index=False))


if __name__ == "__main__":
    compare_dl_models()


### backend/app/routing/evacuation_router.py

**File:** `backend/app/routing/evacuation_router.py`

This is the actual project source copied into the notebook for inspection.
It is a reference section, not an instruction to execute the entire file in a notebook cell.

> This file has 408 lines and is split into numbered reference sections of at most 220 lines. All lines are included.

#### backend/app/routing/evacuation_router.py — lines 1–220

In [ ]:
from fastapi import APIRouter, HTTPException, Body
from app.database import events_collection
from app.building_layout import BUILDING_ID, ROOMS, CORRIDORS, STAIRS, EXITS, EDGES, EXIT_NODES
from app.services.sensor_service import latest_zone_intelligence
from app.services.routing_service import calculate_all_routes, calculate_zone_routes, build_risk_map, BUILDING_GRAPH,compare_algorithms
from app.ml.predict import predict_risk
from app.services.sms_service import send_sms
from app.services.alert_state import (
    claim_sms_alert,
    release_sms_alert,
    clear_sms_alert,
)
import os

FRONTEND_URL = os.getenv("FRONTEND_URL", "http://localhost:5173")
router = APIRouter(prefix="/api/evacuation", tags=["Evacuation"])


def _zone_payload(sensor):
    zone_id = str(sensor["zone_id"])
    return {
        **sensor,
        "connections": BUILDING_GRAPH.get(zone_id, []),
    }


@router.get("/health")
def evacuation_health():
    return {
        "status": "healthy",
        "service": "Dynamic Topology-Based Evacuation Router",
        "routing": "active",
        "zone_intelligence": "enabled",
        "building_topology": "enabled",
        "pathfinding": "Dijkstra",
        "dynamic_risk_routing": "enabled",
        "critical_start_escape": "enabled",
        "zones": len(ROOMS),
        "exits": len(EXITS),
        "staircases": len(STAIRS),
        "deep_learning": "PyTorch MLP",
    }


@router.get("/topology")
def get_building_topology():
    return {
        "building_id": BUILDING_ID,
        "zones": ROOMS,
        "corridors": CORRIDORS,
        "stairs": STAIRS,
        "exits": EXITS,
        "exit_nodes": EXIT_NODES,
        "edges": EDGES,
        "graph": BUILDING_GRAPH,
    }

@router.get("/routes")
def get_evacuation_routes(zone_id: str = None):

    sensors = latest_zone_intelligence()

    if not sensors:
        raise HTTPException(status_code=404, detail="No zone sensor data available")


    return calculate_all_routes(sensors, selected_zone=zone_id)



def _build_zone_route(device_id: str, simulation_data=None):
    zone_id = str(device_id).removeprefix("sensor-")

    if zone_id not in ROOMS:
        raise HTTPException(status_code=404, detail="Zone not found")

    sensors = latest_zone_intelligence()

    if not sensors:
        raise HTTPException(status_code=404, detail="No zone sensor data available")

    if simulation_data:
        simulation_sensor = {
            "temperature": simulation_data.get("temperature"),
            "smoke": simulation_data.get("smoke"),
            "occupancy": simulation_data.get("occupancy"),
        }

        risk = predict_risk({
    "temperature": simulation_sensor["temperature"],
    "humidity": float(
        next(
            (s.get("humidity", 50) for s in sensors
             if str(s["zone_id"]) == zone_id),
            50
        )
    ),
    "smoke": simulation_sensor["smoke"],
    "occupancy": simulation_sensor["occupancy"],
})

        for sensor in sensors:
            if str(sensor["zone_id"]) == zone_id:
                sensor["temperature"] = simulation_sensor["temperature"]
                sensor["smoke"] = simulation_sensor["smoke"]
                sensor["occupancy"] = simulation_sensor["occupancy"]
                sensor["risk_score"] = risk["score"]
                sensor["risk_level"] = risk["level"]
                sensor["dl_prediction"] = risk["level"]
                sensor["dl_confidence"] = risk["confidence"]
                sensor["dl_probabilities"] = risk["probabilities"]
                sensor["recommendation"] = risk["recommendation"]
                break

    risk_map = build_risk_map(sensors)

    if zone_id not in risk_map:
        raise HTTPException(status_code=404, detail="No sensor data found for this zone")

    routes = calculate_zone_routes(zone_id, risk_map)
    available = [r for r in routes if r["status"] == "AVAILABLE"]
    caution = [r for r in routes if r["status"] == "CAUTION"]
    candidates = available or caution
    recommended = min(candidates, key=lambda r: r["risk_score"]) if candidates else None

    sensor = next(s for s in sensors if str(s["zone_id"]) == zone_id)

    return {
        "device_id": sensor.get("device_id", f"sensor-{zone_id}"),
        "zone_id": zone_id,
        "route_status": (
    "SAFE ROUTE"
    if recommended and recommended["status"] == "AVAILABLE"
    else "CAUTION ROUTE"
    if recommended and recommended["status"] == "CAUTION"
    else "NO SAFE EXIT"
),
        "recommended_exit": recommended,
        "alternative_exits": sorted(
            [r for r in routes if r["status"] != "BLOCKED" and (not recommended or r["exit"] != recommended["exit"])],
            key=lambda r: r["risk_score"],
        ),
        "all_routes": routes,
        "zone": _zone_payload(sensor),
    }


@router.post("/route/{device_id}")
def get_zone_evacuation_route(
    device_id: str,
    simulation_data: dict = Body(default=None),
):
    return _build_zone_route(device_id, simulation_data)

@router.post("/compare/{device_id}")
def compare_zone_algorithms(
    device_id: str,
    simulation_data: dict = Body(None),
):
    zone_id = str(device_id).removeprefix("sensor-")

    sensors = latest_zone_intelligence()

    if simulation_data:
        risk = predict_risk({
    "temperature": float(simulation_data.get("temperature", 0)),
    "humidity": float(
        next(
            (s.get("humidity", 50) for s in sensors
             if str(s["zone_id"]) == zone_id),
            50
        )
    ),
    "smoke": float(simulation_data.get("smoke", 0)),
    "occupancy": int(simulation_data.get("occupancy", 0)),
})

        for sensor in sensors:
            if str(sensor["zone_id"]) == zone_id:
                sensor["risk_score"] = risk["score"]
                sensor["risk_level"] = risk["level"]
                sensor["dl_prediction"] = risk["level"]
                sensor["dl_confidence"] = risk["confidence"]
                sensor["dl_probabilities"] = risk["probabilities"]
                sensor["recommendation"] = risk["recommendation"]
                break

    risk_map = build_risk_map(sensors)
    comparison = compare_algorithms(zone_id, risk_map)

    return {
        "device_id": device_id,
        "zone_id": zone_id,
        "comparison": comparison,
    }


@router.get("/zone/{device_id}")
def get_zone_route(device_id: str):
    return _build_zone_route(device_id)


@router.get("/summary")
def evacuation_summary():
    sensors = latest_zone_intelligence()
    if not sensors:
        return {
            "route_status": "NO DATA",
            "recommended_exit": None,
            "recommended_exit_name": None,
            "risk_score": 100,
            "critical_zones": 0,
            "high_risk_zones": 0,
            "total_zones": 0,
        }

    result = calculate_all_routes(sensors)
    recommended = result.get("recommended_exit")
    return {
        "route_status": result["route_status"],


#### backend/app/routing/evacuation_router.py — lines 221–408

In [ ]:
        "recommended_exit": recommended["exit"] if recommended else None,
        "recommended_exit_name": recommended["name"] if recommended else None,
        "risk_score": recommended["risk_score"] if recommended else 100,
        "critical_zones": result["critical_zones"],
        "high_risk_zones": result["high_risk_zones"],
        "total_zones": result["total_zones"],
    }


@router.get("/exits")
def evacuation_exits():
    sensors = latest_zone_intelligence()
    if not sensors:
        raise HTTPException(status_code=404, detail="No sensor data available")
    result = calculate_all_routes(sensors)
    return {
        "exits": result["exits"],
        "recommended_exit": result["recommended_exit"],
        "alternative_exits": result["alternative_exits"],
        "route_status": result["route_status"],
    }
@router.post("/simulation/{device_id}")
def run_simulation_sms(
    device_id: str,
    simulation_data: dict = Body(...)
):
    zone_id = str(device_id).removeprefix("sensor-")

    if zone_id not in ROOMS:
        raise HTTPException(status_code=404, detail="Zone not found")

    sensors = latest_zone_intelligence() or []

    live_sensor = next(
        (
            s for s in sensors
            if str(s.get("zone_id")) == zone_id
        ),
        None
    )

    if not live_sensor:
        raise HTTPException(
            status_code=404,
            detail="No sensor data available for this zone"
        )

    temperature = float(simulation_data.get("temperature", 0))
    smoke = float(simulation_data.get("smoke", 0))
    occupancy = int(simulation_data.get("occupancy", 0))
    humidity = float(live_sensor.get("humidity", 50))

    scenario = simulation_data.get("scenario", "Fire Emergency")

    scenario_multipliers = {
        "Fire Emergency": {
            "temperature": 1.2,
            "smoke": 1.4,
            "occupancy": 0.8,
        },
        "High Temperature": {
            "temperature": 1.6,
            "smoke": 0.5,
            "occupancy": 0.6,
        },
        "Smoke Emergency": {
            "temperature": 0.7,
            "smoke": 1.7,
            "occupancy": 0.7,
        },
        "Overcrowding": {
            "temperature": 0.5,
            "smoke": 0.6,
            "occupancy": 1.8,
        },
        "Combined Emergency": {
            "temperature": 1.3,
            "smoke": 1.3,
            "occupancy": 1.3,
        },
    }

    multipliers = scenario_multipliers.get(
        scenario,
        scenario_multipliers["Fire Emergency"]
    )

    adjusted_temperature = temperature * multipliers["temperature"]
    adjusted_smoke = smoke * multipliers["smoke"]
    adjusted_occupancy = round(
        occupancy * multipliers["occupancy"]
    )

    dl_result = predict_risk({
        "temperature": adjusted_temperature,
        "humidity": humidity,
        "smoke": adjusted_smoke,
        "occupancy": adjusted_occupancy,
    })

    risk_score = dl_result["score"]
    risk_level = dl_result["level"]

    if risk_level not in ("HIGH", "CRITICAL"):
        return {
            "sms_sent": False,
            "risk_level": risk_level,
            "risk_score": risk_score,
            "message": "No emergency SMS required."
        }

    if not claim_sms_alert(zone_id):
        return {
            "sms_sent": False,
            "risk_level": risk_level,
            "risk_score": risk_score,
            "message": "SMS already sent for this active zone emergency."
        }

    try:
        simulation_sensor = {
            **live_sensor,
            "temperature": adjusted_temperature,
            "humidity": humidity,
            "smoke": adjusted_smoke,
            "occupancy": adjusted_occupancy,
            "risk_level": risk_level,
            "risk_score": risk_score,
            "dl_prediction": risk_level,
            "dl_confidence": dl_result["confidence"],
            "dl_probabilities": dl_result["probabilities"],
            "recommendation": dl_result["recommendation"],
        }

        updated_sensors = [
            s for s in sensors
            if str(s.get("zone_id")) != zone_id
        ]

        updated_sensors.append(simulation_sensor)

        route_result = calculate_all_routes(
            updated_sensors,
            selected_zone=zone_id
        )

        recommended = route_result.get(
            "selected_zone_recommendation"
        )

        route_name = (
            f"{recommended.get('exit')} - {recommended.get('name')}"
            if recommended
            else "No safe exit available"
        )


        message = (
            f"EVAC-TWIN SIMULATION ALERT! "
            f"Zone: {zone_id}. "
            f"Risk: {risk_level}. "
            f"Risk Score: {risk_score}. "
            f"Recommended Exit: {route_name}. "
            f"Please evacuate immediately."
        )

        success = send_sms(message)
        if not success:
            release_sms_alert(zone_id)

        return {
            "sms_sent": success,
            "risk_level": risk_level,
            "risk_score": risk_score,
            "recommended_exit": route_name,
            "message": (
                "Simulation emergency SMS sent."
                if success
                else "SMS sending failed."
            )
        }

    except Exception as error:
        release_sms_alert(zone_id)
        raise HTTPException(
            status_code=500,
            detail=f"Simulation alert failed: {error}"
        )

### backend/app/services/routing_service.py

**File:** `backend/app/services/routing_service.py`

This is the actual project source copied into the notebook for inspection.
It is a reference section, not an instruction to execute the entire file in a notebook cell.

> This file has 1666 lines and is split into numbered reference sections of at most 220 lines. All lines are included.

#### backend/app/services/routing_service.py — lines 1–220

In [ ]:
from heapq import heappop, heappush
from math import inf
from time import perf_counter
from collections import deque

from app.building_layout import (
    BUILDING_ID,
    ROOMS,
    CORRIDORS,
    STAIRS,
    EXITS,
    EXIT_NODES,
    EDGES,
)


# =========================================================
# EVAC-TWIN — ROUTING CONFIGURATION
# =========================================================

NORMAL_EDGE_COST = 1.0
CORRIDOR_EDGE_COST = 1.0
STAIR_EDGE_COST = 2.0
EXIT_EDGE_COST = 1.0


# Risk penalties applied when ENTERING a room.
#
# CRITICAL is handled separately because another critical
# room must never be entered during evacuation.
RISK_PENALTY = {
    "LOW": 0.0,
    "MODERATE": 3.0,
    "HIGH": 12.0,
    "CRITICAL": inf,
    "UNKNOWN": 20.0,
}


# Additional continuous risk contribution.
# This allows two HIGH zones with different risk scores
# to have different routing costs.
CONTINUOUS_RISK_WEIGHT = 5.0


# Missing sensor data is not considered safe.
# It remains traversable so one missing sensor does not
# completely destroy the evacuation network, but the route
# receives a caution penalty.
UNKNOWN_RISK_SCORE = 50.0


# =========================================================
# BUILDING GRAPH
# =========================================================

def build_graph():
    """
    Build an undirected graph from the physical building edges.

    The graph contains:
        - rooms
        - corridor sections
        - stair nodes
        - exit nodes
    """

    graph = {}

    for a, b in EDGES:
        graph.setdefault(a, set()).add(b)
        graph.setdefault(b, set()).add(a)

    return {
        node: sorted(neighbors)
        for node, neighbors in graph.items()
    }


BUILDING_GRAPH = build_graph()


# =========================================================
# NODE HELPERS
# =========================================================

def is_exit(node):
    return node in EXIT_NODES.values()


def is_room(node):
    return node in ROOMS


def is_corridor(node):
    return node in CORRIDORS


def is_stair(node):
    return str(node).startswith("STAIR_")


def node_risk(node, risk_map):
    """
    Return normalized risk information for a node.

    Room nodes use live sensor intelligence.

    Non-room topology nodes such as corridors and stairs do
    not currently have individual sensors, so they are treated
    as LOW physical-topology nodes.

    Missing room sensor data is explicitly represented as
    UNKNOWN rather than silently becoming LOW.
    """

    if node in risk_map:
        risk = risk_map[node]

        return {
            "risk_score": float(
                risk.get("risk_score", 0.0)
            ),
            "risk_level": str(
                risk.get("risk_level", "UNKNOWN")
            ).upper(),
            "occupancy": int(
                risk.get("occupancy", 0) or 0
            ),
            "sensor_available": bool(
                risk.get("sensor_available", True)
            ),
        }

    # Topology nodes without individual sensors.
    if node in CORRIDORS or is_stair(node) or is_exit(node):
        return {
            "risk_score": 0.0,
            "risk_level": "LOW",
            "occupancy": 0,
            "sensor_available": False,
        }

    # A room without current sensor information.
    return {
        "risk_score": UNKNOWN_RISK_SCORE,
        "risk_level": "UNKNOWN",
        "occupancy": 0,
        "sensor_available": False,
    }


# =========================================================
# EDGE COST
# =========================================================

def edge_cost(current, neighbor, risk_map, start):
    """
    Calculate movement cost from current -> neighbor.

    Priority:

        EXIT
        ↓
        STAIR
        ↓
        CORRIDOR
        ↓
        ROOM
        ↓
        NORMAL

    Risk is applied to room entry.

    Rules:
        - The starting critical room is allowed.
        - Another critical room cannot be entered.
        - HIGH/MODERATE rooms remain traversable.
        - UNKNOWN rooms remain traversable with a caution penalty.
        - Corridors have normal movement cost.
        - Stairs have higher physical movement cost.
        - Exit nodes are terminal.
    """

    # =====================================================
    # EXIT
    # =====================================================

    if is_exit(neighbor):
        return EXIT_EDGE_COST

    # Once an exit has been reached, it is terminal.
    if is_exit(current):
        return inf

    # =====================================================
    # STAIR
    # =====================================================

    if is_stair(neighbor) or is_stair(current):
        return STAIR_EDGE_COST

    # =====================================================
    # CORRIDOR
    # =====================================================

    if is_corridor(neighbor) or is_corridor(current):
        return CORRIDOR_EDGE_COST

    # =====================================================
    # ROOM RISK
    # =====================================================

    if is_room(neighbor):

        risk = node_risk(
            neighbor,
            risk_map,
        )



#### backend/app/services/routing_service.py — lines 221–440

In [ ]:
        level = str(
            risk.get("risk_level", "UNKNOWN")
        ).upper()

        # Critical zones cannot be entered.
        # The starting critical zone is allowed because
        # occupants are already inside it.
        if level == "CRITICAL" and neighbor != start:
            return inf

        score = float(
            risk.get(
                "risk_score",
                UNKNOWN_RISK_SCORE,
            )
        )

        level_penalty = RISK_PENALTY.get(
            level,
            RISK_PENALTY["UNKNOWN"],
        )

        continuous_penalty = (
            score / 100.0
        ) * CONTINUOUS_RISK_WEIGHT

        return (
            NORMAL_EDGE_COST
            + level_penalty
            + continuous_penalty
        )

    return NORMAL_EDGE_COST



# =========================================================
# SHORTEST SAFE PHYSICAL PATH
# =========================================================


def bfs_path(start, target):
    queue = deque([start])
    visited = {start}
    nodes_explored = 0
    previous = {}

    while queue:
        current = queue.popleft()
        nodes_explored += 1

        if current == target:
            break

        for neighbor in BUILDING_GRAPH[current]:
            if neighbor not in visited:
                visited.add(neighbor)
                previous[neighbor] = current
                queue.append(neighbor)

    if target not in visited:
        return None

    path = [target]
    current = target

    while current != start:
        current = previous[current]
        path.append(current)

    path.reverse()

    return {
    "path": path,
    "edges": len(path) - 1,
    "nodes_explored": nodes_explored
}
def test_bfs():
    result = bfs_path("101", "EXIT_A")
    print("BFS TEST PATH:", result)

def dfs_path(start, target):
    stack = [start]
    visited = {start}
    nodes_explored = 0
    previous = {}

    while stack:
        current = stack.pop()
        nodes_explored += 1

        if current == target:
            break

        for neighbor in BUILDING_GRAPH[current]:
            if neighbor not in visited:
                visited.add(neighbor)
                previous[neighbor] = current
                stack.append(neighbor)

    if target not in visited:
        return None

    path = [target]
    current = target

    while current != start:
        current = previous[current]
        path.append(current)

    path.reverse()

    return {
    "path": path,
    "edges": len(path) - 1,
    "nodes_explored": nodes_explored
}

def a_star_path(start, target, risk_map):
    distances = {target: 0}
    queue = [target]

    while queue:
        current = queue.pop(0)

        for neighbor in BUILDING_GRAPH.get(current, []):
            if neighbor not in distances:
                distances[neighbor] = distances[current] + 1
                queue.append(neighbor)

    def heuristic(node):
        return distances.get(node, 0)

    open_set = [(heuristic(start), 0, start)]
    previous = {}
    g_score = {start: 0}
    visited = set()
    nodes_explored = 0

    while open_set:
        _, current_cost, current = heappop(open_set)

        if current in visited:
            continue

        visited.add(current)
        nodes_explored += 1

        if current == target:
            break

        if is_exit(current):
            continue

        for neighbor in BUILDING_GRAPH[current]:
            if neighbor in visited:
                continue

            cost = edge_cost(current, neighbor, risk_map, start)

            if cost == inf:
                continue

            new_score = current_cost + cost

            if new_score < g_score.get(neighbor, inf):
                g_score[neighbor] = new_score
                previous[neighbor] = current

                f_score = new_score + heuristic(neighbor)

                heappush(
                    open_set,
                    (f_score, new_score, neighbor)
                )

    if target not in previous and start != target:
        return None

    path = [target]
    current = target

    while current != start:
        current = previous[current]
        path.append(current)

    path.reverse()

    return {
        "path": path,
        "cost": round(g_score[target], 2),
        "edges": len(path) - 1,
        "nodes_explored": nodes_explored
    }
def shortest_path(start, target, risk_map):
    """
    Dijkstra shortest-path search using the physical
    Building Alpha topology and dynamic room risk.

    A critical starting room is allowed because people
    are already inside that room and must escape.

    Other critical rooms are blocked from entry.
    """

    if start not in BUILDING_GRAPH:
        return None

    if target not in BUILDING_GRAPH:
        return None

    distances = {
        node: inf
        for node in BUILDING_GRAPH
    }

    previous = {}

    distances[start] = 0.0



#### backend/app/services/routing_service.py — lines 441–660

In [ ]:
    queue = [
        (0.0, start)
    ]

    visited = set()
    nodes_explored = 0

    while queue:

        distance, current = heappop(queue)

        if current in visited or distance > distances[current]:
           continue


        visited.add(current)
        nodes_explored += 1

        if current == target:
            break

        if is_exit(current):
            continue

        for neighbor in BUILDING_GRAPH[current]:

            if neighbor in visited:
                continue

            cost = edge_cost(
                current,
                neighbor,
                risk_map,
                start,
            )

            if cost == inf:
                continue

            candidate = distance + cost

            if candidate < distances[neighbor]:

                distances[neighbor] = candidate

                previous[neighbor] = current

                heappush(
                    queue,
                    (
                        candidate,
                        neighbor,
                    ),
                )

    if distances[target] == inf:
        return None

    # =====================================================
    # RECONSTRUCT PATH
    # =====================================================

    path = [target]
    current = target

    while current != start:

        if current not in previous:
            return None

        current = previous[current]
        path.append(current)

    path.reverse()

    return {
    "path": path,
    "cost": round(distances[target], 2),
    "edges": len(path) - 1,
    "nodes_explored": nodes_explored,
}

# =========================================================
# PATH RISK ANALYSIS
# =========================================================

def analyze_path(path, risk_map):
    """
    Analyze every room appearing in an evacuation path.

    The starting room is included in the risk statistics.

    A critical starting room does NOT block the route.

    A critical room encountered after the starting room
    makes the route BLOCKED.

    UNKNOWN room sensor data produces CAUTION rather than
    being treated as safe.
    """

    critical = []
    high = []
    moderate = []
    unknown = []
    scores = []

    # =====================================================
    # ANALYZE ROOMS
    # =====================================================

    for node in path:

        if node not in ROOMS:
            continue

        risk = node_risk(
            node,
            risk_map,
        )

        level = str(
            risk.get(
                "risk_level",
                "UNKNOWN",
            )
        ).upper()

        score = float(
            risk.get(
                "risk_score",
                UNKNOWN_RISK_SCORE,
            )
        )

        scores.append(score)

        if level == "CRITICAL":
            critical.append(node)

        elif level == "HIGH":
            high.append(node)

        elif level == "MODERATE":
            moderate.append(node)

        elif level == "UNKNOWN":
            unknown.append(node)

    # =====================================================
    # RISK STATISTICS
    # =====================================================

    average = (
        sum(scores) / len(scores)
        if scores
        else 0.0
    )

    maximum = (
        max(scores)
        if scores
        else 0.0
    )

    # =====================================================
    # CRITICAL ZONES AFTER START
    # =====================================================

    blocking_critical = []

    for index, node in enumerate(path):

        # The first node is always the evacuation origin.
        if index == 0:
            continue

        if node not in ROOMS:
            continue

        risk = node_risk(
            node,
            risk_map,
        )

        level = str(
            risk.get(
                "risk_level",
                "UNKNOWN",
            )
        ).upper()

        if level == "CRITICAL":
            blocking_critical.append(node)

    # =====================================================
    # ROUTE STATUS
    # =====================================================

    if blocking_critical:

        status = "BLOCKED"

        reason = (
            "A critical-risk zone blocks this path."
        )

    elif unknown:

        status = "CAUTION"

        reason = (
            "The path contains a zone with unavailable "
            "sensor intelligence."
        )

    elif maximum >= 60:

        status = "CAUTION"



#### backend/app/services/routing_service.py — lines 661–880

In [ ]:
        reason = (
            "The evacuation starts in or passes through "
            "a critical-risk condition."
        )

    elif maximum >= 35:

        status = "CAUTION"

        reason = (
            "The path contains elevated-risk conditions."
        )

    else:

        status = "AVAILABLE"

        reason = (
            "Path is clear of significant hazards."
        )

    return {
        "status": status,
        "reason": reason,
        "average_risk": round(
            average,
            2,
        ),
        "maximum_risk": round(
            maximum,
            2,
        ),
        "critical_zones": critical,
        "high_zones": high,
        "moderate_zones": moderate,
        "unknown_zones": unknown,
    }


# =========================================================
# ROUTE SCORE
# =========================================================

def calculate_route_score(
    analysis,
    physical_cost,
):
    """
    Convert route characteristics into one comparable
    0-100 evacuation risk score.

    Components:

        Average room risk  -> 60%
        Maximum room risk  -> 25%
        Physical cost      -> 15%

    The score is used for comparing available exits.
    """

    average_risk = float(
        analysis.get(
            "average_risk",
            0.0,
        )
    )

    maximum_risk = float(
        analysis.get(
            "maximum_risk",
            0.0,
        )
    )

    # Keep the physical component bounded so a long but
    # otherwise safe route does not dominate the risk.
    physical_component = min(
        100.0,
        float(physical_cost) * 5.0,
    )

    score = (
        average_risk * 0.60
        + maximum_risk * 0.25
        + physical_component * 0.15
    )

    return round(
        min(
            100.0,
            score,
        ),
        2,
    )


# =========================================================
# CALCULATE ONE ZONE -> EXIT ROUTE
# =========================================================

def calculate_route(
    zone_id,
    exit_name,
    risk_map,
):
    """
    Calculate one complete evacuation option:

        zone → physical topology → exit
    """

    zone_id = str(zone_id)

    exit_config = EXITS.get(
        exit_name
    )

    if zone_id not in ROOMS:
        return None

    if not exit_config:
        return None

    target = EXIT_NODES.get(
        exit_name
    )

    if not target:
        return None

    # =====================================================
    # DEFAULT BLOCKED RESPONSE
    # =====================================================

    base = {
        "exit": exit_name,
        "name": exit_config["name"],
        "status": "BLOCKED",
        "risk_score": 100.0,
        "reason": (
            "No safe physical path exists."
        ),
        "path": [],
        "distance": None,
        "critical_zones": [],
        "high_zones": [],
        "moderate_zones": [],
        "unknown_zones": [],
        "average_risk": 100.0,
        "maximum_risk": 100.0,
    }

    # =====================================================
    # FIND PHYSICAL PATH
    # =====================================================

    result = shortest_path(
        zone_id,
        target,
        risk_map,
    )

    if result is None:
        return base

    # =====================================================
    # ANALYZE PATH
    # =====================================================

    analysis = analyze_path(
        result["path"],
        risk_map,
    )

    # =====================================================
    # FINAL ROUTE SCORE
    # =====================================================

    final_score = calculate_route_score(
        analysis,
        result["cost"],
    )

    # Use the status determined by analyze_path(), which correctly
    # uses `blocking_critical` (excludes the starting room) for the
    # BLOCKED decision. Do NOT re-derive from critical_zones here
    # because that list includes the starting room and would
    # incorrectly mark every route as BLOCKED when the origin zone
    # is CRITICAL.
    status = analysis["status"]

    return {
    **base,
    **analysis,
    "status": status,
    "risk_score": final_score,
    "path": result["path"],
    "distance": result["edges"],
    "cost": result["cost"],
    "edges": result["edges"],
}


# =========================================================
# BUILD RISK MAP FROM SENSOR DATA
# =========================================================

def build_risk_map(sensors):
    """
    Convert sensor intelligence into a zone-indexed map.

    Only actual Building Alpha rooms are included.

    Missing zones are NOT fabricated here. They remain
    absent and are interpreted as UNKNOWN by node_risk().
    """

    risk_map = {}

    for sensor in sensors:


#### backend/app/services/routing_service.py — lines 881–1100

In [ ]:

        zone_id = str(
            sensor.get(
                "zone_id",
                "",
            )
        )

        if zone_id not in ROOMS:
            continue

        raw_level = sensor.get(
            "risk_level"
        )

        if raw_level is None:
            risk_level = "UNKNOWN"
        else:
            risk_level = str(
                raw_level
            ).upper()

        raw_score = sensor.get(
            "risk_score"
        )

        if raw_score is None:
            risk_score = (
                UNKNOWN_RISK_SCORE
                if risk_level == "UNKNOWN"
                else 0.0
            )
        else:
            try:
                risk_score = float(
                    raw_score
                )
            except (
                TypeError,
                ValueError,
            ):
                risk_score = (
                    UNKNOWN_RISK_SCORE
                    if risk_level == "UNKNOWN"
                    else 0.0
                )

        try:
            occupancy = int(
                sensor.get(
                    "occupancy",
                    0,
                )
                or 0
            )
        except (
            TypeError,
            ValueError,
        ):
            occupancy = 0

        risk_map[zone_id] = {
            "risk_score": max(
                0.0,
                min(
                    100.0,
                    risk_score,
                ),
            ),
            "risk_level": risk_level,
            "occupancy": max(
                0,
                occupancy,
            ),
            "sensor_available": True,
        }

    return risk_map


# =========================================================
# CALCULATE ALL EXIT OPTIONS FOR ONE ZONE
# =========================================================

def calculate_zone_routes(
    zone_id,
    risk_map,
):
    """
    Return all three exit options for one zone.
    """

    return [
        calculate_route(
            zone_id,
            exit_name,
            risk_map,
        )
        for exit_name in EXITS
    ]
def calculate_path_risk(path, risk_map):
    if not path:
        return None

    total_risk = 0
    high_risk_rooms = 0
    critical_rooms = 0
    unknown_rooms = 0

    for node in path:
        if is_room(node):
            risk = node_risk(node, risk_map)

            total_risk += risk["risk_score"]

            level = risk["risk_level"]

            if level == "HIGH":
                high_risk_rooms += 1
            elif level == "CRITICAL":
                critical_rooms += 1
            elif level == "UNKNOWN":
                unknown_rooms += 1

    return {
        "total_risk": round(total_risk, 2),
        "high_risk_rooms": high_risk_rooms,
        "critical_rooms": critical_rooms,
        "unknown_rooms": unknown_rooms
    }

def compare_algorithms(zone_id, risk_map):
    exits = EXIT_NODES.values()
    results = []

    for target in exits:

        start_time = perf_counter()
        bfs = bfs_path(zone_id, target)
        bfs_time = (perf_counter() - start_time) * 1000

        start_time = perf_counter()
        dfs = dfs_path(zone_id, target)
        dfs_time = (perf_counter() - start_time) * 1000

        start_time = perf_counter()
        dijkstra = shortest_path(zone_id, target, risk_map)
        dijkstra_time = (perf_counter() - start_time) * 1000

        start_time = perf_counter()
        astar = a_star_path(zone_id, target, risk_map)
        astar_time = (perf_counter() - start_time) * 1000

        results.append({
            "exit": target,
           "bfs": {
    "path": bfs["path"] if bfs else None,
    "edges": bfs["edges"] if bfs else None,
    "nodes_explored": bfs["nodes_explored"] if bfs else None,
    "execution_time_ms": round(bfs_time, 4),
    "risk_exposure": calculate_path_risk(bfs["path"] if bfs else None, risk_map),
},
          "dfs": {
    "path": dfs["path"] if dfs else None,
    "edges": dfs["edges"] if dfs else None,
    "nodes_explored": dfs["nodes_explored"] if dfs else None,
    "execution_time_ms": round(dfs_time, 4),
    "risk_exposure": calculate_path_risk(dfs["path"] if dfs else None, risk_map),
},
          "dijkstra": {
    "path": dijkstra["path"] if dijkstra else None,
    "cost": dijkstra["cost"] if dijkstra else None,
    "edges": dijkstra["edges"] if dijkstra else None,
    "nodes_explored": dijkstra["nodes_explored"] if dijkstra else None,
    "execution_time_ms": round(dijkstra_time, 4),
    "risk_exposure": calculate_path_risk(dijkstra["path"] if dijkstra else None, risk_map),
},
          "a_star": {
    "path": astar["path"] if astar else None,
    "cost": astar["cost"] if astar else None,
    "edges": astar["edges"] if astar else None,
    "nodes_explored": astar["nodes_explored"] if astar else None,
    "execution_time_ms": round(astar_time, 4),
    "risk_exposure": calculate_path_risk(astar["path"] if astar else None, risk_map),
},
        })

    return results
# =========================================================
# ROUTE WEIGHT
# =========================================================

def _route_weight(
    route,
    risk_map,
    zone_id,
):
    """
    Weight helper retained for compatibility.

    Routes are primarily ordered by safety score.
    Occupancy is used for building-level aggregation,
    not to make one person's route inherently less safe.
    """

    occupancy = max(
        0,
        int(
            risk_map
            .get(
                zone_id,
                {},
            )
            .get(
                "occupancy",
                0,
            )
        ),
    )



#### backend/app/services/routing_service.py — lines 1101–1320

In [ ]:
    return (
        route["risk_score"],
        -occupancy,
    )


# =========================================================
# CALCULATE COMPLETE BUILDING EVACUATION
# =========================================================

def calculate_all_routes(sensors, selected_zone=None):
    """
    Calculate evacuation intelligence for the entire building.

    Returns:

        - zone-by-zone routes
        - exit-level analysis
        - building-level recommendation
        - alternative exits
        - risk statistics
        - building graph
    """

    risk_map = build_risk_map(
        sensors
    )

    zones = []

    # =====================================================
    # ZONE-BY-ZONE ROUTES
    # =====================================================

    for sensor in sensors:

        zone_id = str(
            sensor.get(
                "zone_id",
                "",
            )
        )

        if zone_id not in ROOMS:
            continue

        routes = calculate_zone_routes(
            zone_id,
            risk_map,
        )

        # -------------------------------------------------
        # AVAILABLE
        # -------------------------------------------------

        available = [
            route
            for route in routes
            if route
            and route["status"]
            == "AVAILABLE"
        ]

        # -------------------------------------------------
        # CAUTION
        # -------------------------------------------------

        caution = [
            route
            for route in routes
            if route
            and route["status"]
            == "CAUTION"
        ]

        # Safe routes are always preferred over caution
        # routes.
        candidates = (
            available
            or caution
        )

        recommended = (
            min(
                candidates,
                key=lambda route:
                route["risk_score"],
            )
            if candidates
            else None
        )

        alternatives = sorted(
            [
                route
                for route in routes
                if (
                    route
                    and route["status"]
                    != "BLOCKED"
                    and (
                        not recommended
                        or route["exit"]
                        != recommended["exit"]
                    )
                )
            ],
            key=lambda route:
            route["risk_score"],
        )

        zones.append({
            "zone_id": zone_id,

            "floor": ROOMS[
                zone_id
            ]["floor"],

            "recommended_exit": recommended,

            "alternative_exits": alternatives,

            "all_routes": routes,
        })
    selected_zone_data = next(
        (
            zone
            for zone in zones
            if zone["zone_id"] == str(selected_zone)
        ),
        None,
    )
    # =====================================================
    # BUILDING-LEVEL EXIT ANALYSIS
    # =====================================================

    exits = []

    for exit_name, exit_config in EXITS.items():

        route_records = []

        for zone_id in risk_map:

            route = calculate_route(
                zone_id,
                exit_name,
                risk_map,
            )

            if route is not None:
                route_records.append(
                    (
                        zone_id,
                        route,
                    )
                )

        # -------------------------------------------------
        # USABLE ROUTES
        # -------------------------------------------------

        usable = [
            (
                zone_id,
                route,
            )
            for zone_id, route
            in route_records
            if route["status"]
            != "BLOCKED"
        ]

        connected = sorted(
            [
                zone_id
                for zone_id, _
                in usable
            ],
            key=int,
        )

        # -------------------------------------------------
        # EXIT HAS USABLE ROUTES
        # -------------------------------------------------

        if usable:

            # Actual occupancy only.
            total_people = sum(
                max(
                    0,
                    int(
                        risk_map[
                            zone_id
                        ].get(
                            "occupancy",
                            0,
                        )
                    ),
                )
                for zone_id, _
                in usable
            )

            # -------------------------------------------------
            # WEIGHTED EXIT RISK
            # -------------------------------------------------
            #
            # If occupancy exists, people are used as the
            # weighting factor.
            #
            # If every zone currently reports zero occupancy,
            # use an ordinary mean so the exit still receives
            # a meaningful risk score.

            if total_people > 0:

                weighted_score = (
                    sum(


#### backend/app/services/routing_service.py — lines 1321–1540

In [ ]:
                        route["risk_score"]
                        * max(
                            0,
                            int(
                                risk_map[
                                    zone_id
                                ].get(
                                    "occupancy",
                                    0,
                                )
                            ),
                        )
                        for zone_id, route
                        in usable
                    )
                    / total_people
                )

            else:

                weighted_score = (
                    sum(
                        route["risk_score"]
                        for _, route
                        in usable
                    )
                    / len(usable)
                )

            # -------------------------------------------------
            # BEST ROUTE TO THIS EXIT
            # -------------------------------------------------

            best_zone, best_route = min(
                usable,
                key=lambda item:
                item[1]["risk_score"],
            )

            # -------------------------------------------------
            # EXIT STATUS
            # -------------------------------------------------

            status = (
                "AVAILABLE"
                if any(
                    route["status"]
                    == "AVAILABLE"
                    for _, route
                    in usable
                )
                else "CAUTION"
            )

            if status == "AVAILABLE":

                reason = (
                    "Exit has at least one currently "
                    "available evacuation route."
                )

            else:

                reason = (
                    "Exit is reachable, but one or more "
                    "routes contain elevated or unknown risk."
                )

            exits.append({

                "exit": exit_name,

                "name": exit_config["name"],

                "risk_score": round(
                    min(
                        100.0,
                        weighted_score,
                    ),
                    2,
                ),

                "status": status,

                "reason": reason,

                "connected_zones": connected,

                "total_people": total_people,

                "best_path": best_route["path"],

                "best_zone": best_zone,

                "distance": best_route["distance"],
            })

        # -------------------------------------------------
        # EXIT COMPLETELY BLOCKED
        # -------------------------------------------------

        else:

            exits.append({

                "exit": exit_name,

                "name": exit_config["name"],

                "risk_score": 100.0,

                "status": "BLOCKED",

                "reason": (
                    "No safe physical route is currently "
                    "available from the monitored zones."
                ),

                "connected_zones": [],

                "total_people": 0,

                "best_path": [],

                "best_zone": None,

                "distance": None,
            })

    # =====================================================
    # BUILDING-LEVEL RECOMMENDED EXIT
    # =====================================================

    available_exits = [
        exit_data
        for exit_data in exits
        if exit_data["status"]
        == "AVAILABLE"
    ]

    caution_exits = [
        exit_data
        for exit_data in exits
        if exit_data["status"]
        == "CAUTION"
    ]

    # Always prefer a genuinely available exit.
    candidates = (
        available_exits
        or caution_exits
    )

    recommended = (
        min(
            candidates,
            key=lambda exit_data:
            (
                exit_data["risk_score"],
                -len(
                    exit_data[
                        "connected_zones"
                    ]
                ),
            ),
        )
        if candidates
        else None
    )

    # =====================================================
    # ALTERNATIVE EXITS
    # =====================================================

    alternatives = sorted(
        [
            exit_data
            for exit_data in exits
            if (
                exit_data["status"]
                != "BLOCKED"
                and (
                    not recommended
                    or exit_data["exit"]
                    != recommended["exit"]
                )
            )
        ],
        key=lambda exit_data:
        exit_data["risk_score"],
    )


    # =====================================================
    # FINAL BUILDING STATUS
    # =====================================================

    if available_exits:
        route_status = "SAFE ROUTE"
    elif caution_exits:
        route_status = "CAUTION ROUTE"
    else:
        route_status = "NO SAFE EXIT"

    building_condition = (
        "CRITICAL"
        if any(
            risk.get("risk_level") == "CRITICAL"
            for risk in risk_map.values()
        )
        else "HIGH RISK"
        if any(
            risk.get("risk_level") == "HIGH"
            for risk in risk_map.values()
        )
        else "NORMAL"
    )


    # =====================================================


#### backend/app/services/routing_service.py — lines 1541–1666

In [ ]:
    # ZONE STATISTICS
    # =====================================================

    monitored_zones = [
        sensor
        for sensor in sensors
        if str(
            sensor.get(
                "zone_id",
                "",
            )
        ) in ROOMS
    ]

    critical_zones = sum(
        1
        for sensor in monitored_zones
        if str(
            sensor.get(
                "risk_level",
                "",
            )
        ).upper()
        == "CRITICAL"
    )

    high_risk_zones = sum(
        1
        for sensor in monitored_zones
        if str(
            sensor.get(
                "risk_level",
                "",
            )
        ).upper()
        == "HIGH"
    )

    moderate_risk_zones = sum(
        1
        for sensor in monitored_zones
        if str(
            sensor.get(
                "risk_level",
                "",
            )
        ).upper()
        == "MODERATE"
    )

    unknown_zones = (
        len(ROOMS)
        - len(risk_map)
    )

    # =====================================================
    # TOTAL OCCUPANCY
    # =====================================================

    total_occupancy = sum(
        max(
            0,
            int(
                risk.get(
                    "occupancy",
                    0,
                )
            ),
        )
        for risk in risk_map.values()
    )

    # =====================================================
    # FINAL RESPONSE
    # =====================================================

    return {

        "building_id": BUILDING_ID,

        "zones": zones,

        "exits": exits,

        "recommended_exit": recommended,
        "selected_zone": selected_zone,
"selected_zone_recommendation": (
    selected_zone_data["recommended_exit"]
    if selected_zone_data
    else None
),
"selected_zone_routes": (
    selected_zone_data["all_routes"]
    if selected_zone_data
    else []
),

        "alternative_exits": alternatives,

        "route_status": route_status,
        "building_condition": building_condition,

        "total_zones": len(
            monitored_zones
        ),

        "building_zones": len(
            ROOMS
        ),

        "monitored_zones": len(
            risk_map
        ),

        "unknown_zones": unknown_zones,

        "critical_zones": critical_zones,

        "high_risk_zones": high_risk_zones,

        "moderate_risk_zones": moderate_risk_zones,

        "total_occupancy": total_occupancy,

        "building_graph": BUILDING_GRAPH,
    }


### backend/app/services/sensor_service.py

**File:** `backend/app/services/sensor_service.py`

This is the actual project source copied into the notebook for inspection.
It is a reference section, not an instruction to execute the entire file in a notebook cell.

In [ ]:

from app.database import sensors_collection
from app.ml.zone_risk import calculate_zone_risk
from app.building_layout import BUILDING_ID, ROOMS


# ============================================================
# EVAC-TWIN — SENSOR INTELLIGENCE SERVICE
# ============================================================


def get_zone_id(sensor):
    if not sensor:
        return ""

    zone_id = sensor.get("zone_id")

    if zone_id is not None:
        return str(zone_id)

    device_id = str(sensor.get("device_id", ""))

    return device_id.removeprefix("sensor-")


# ============================================================
# LATEST SENSOR PER ZONE
# ============================================================

def latest_sensors():
    sensors = []

    for zone_id in ROOMS:
        sensor = sensors_collection.find_one(
            {"zone_id": str(zone_id)},
            {"_id": 0},
            sort=[("timestamp", -1)],
        )

        if sensor:
            sensors.append(sensor)

    sensors.sort(key=lambda sensor: int(get_zone_id(sensor)))

    return sensors


# ============================================================
# SINGLE LATEST SENSOR
# ============================================================

def latest_sensor(zone_id):
    zone_id = str(zone_id)

    return sensors_collection.find_one(
        {"zone_id": zone_id},
        {"_id": 0},
        sort=[("timestamp", -1)],
    )


# ============================================================
# SENSOR INTELLIGENCE
# ============================================================

def enrich_sensor(sensor):
    if not sensor:
        return None

    result = dict(sensor)

    zone_id = get_zone_id(result)

    result["zone_id"] = zone_id
    result.setdefault("device_id", f"sensor-{zone_id}")
    result.setdefault("building_id", BUILDING_ID)

    # --------------------------------------------------------
    # Existing DL result
    # --------------------------------------------------------

    has_dl_result = (
        result.get("risk_level") is not None
        and result.get("risk_score") is not None
    )

    if not has_dl_result:
        result.update(calculate_zone_risk(result))

    else:
        result["risk_level"] = str(
            result.get("risk_level", "LOW")
        ).upper()

        try:
            result["risk_score"] = float(
                result.get("risk_score", 0)
            )
        except (TypeError, ValueError):
            result["risk_score"] = 0.0

        try:
            result["confidence"] = float(
                result.get("confidence", 0)
            )
        except (TypeError, ValueError):
            result["confidence"] = 0.0

        result.setdefault("dl_prediction", result["risk_level"])
        result.setdefault("dl_confidence", result["confidence"])
        result.setdefault("dl_probabilities", {})

        result.setdefault(
            "recommendation",
            "Continue monitoring sensor conditions.",
        )

    # --------------------------------------------------------
    # Zone metadata
    # --------------------------------------------------------

    result["zone"] = f"Zone {zone_id}"

    try:
        result["floor"] = int(
            result.get(
                "floor",
                ROOMS.get(zone_id, {}).get("floor", 0),
            )
        )
    except (TypeError, ValueError):
        result["floor"] = ROOMS.get(
            zone_id, {}
        ).get("floor", 0)

    return result


# ============================================================
# LATEST ZONE INTELLIGENCE
# ============================================================

def latest_zone_intelligence():
    return [
        enrich_sensor(sensor)
        for sensor in latest_sensors()
    ]


# ============================================================
# SENSOR HISTORY
# ============================================================

def sensor_history(zone_id=None, limit=100):
    query = {}

    if zone_id:
        query["zone_id"] = str(zone_id)

    try:
        limit = int(limit)
    except (TypeError, ValueError):
        limit = 100

    limit = max(1, min(limit, 1000))

    return list(
        sensors_collection.find(
            query,
            {"_id": 0},
        )
        .sort("timestamp", -1)
        .limit(limit)
    )


### backend/app/services/alert_state.py

**File:** `backend/app/services/alert_state.py`

This is the actual project source copied into the notebook for inspection.
It is a reference section, not an instruction to execute the entire file in a notebook cell.

In [ ]:
from threading import Lock

sms_alerted_zones = set()
sms_lock = Lock()


def claim_sms_alert(zone_id):
    zone_id = str(zone_id)

    with sms_lock:
        if zone_id in sms_alerted_zones:
            return False

        sms_alerted_zones.add(zone_id)
        return True


def release_sms_alert(zone_id):
    with sms_lock:
        sms_alerted_zones.discard(str(zone_id))


def clear_sms_alert(zone_id):
    with sms_lock:
        sms_alerted_zones.discard(str(zone_id))

### backend/app/services/sms_service.py

**File:** `backend/app/services/sms_service.py`

This is the actual project source copied into the notebook for inspection.
It is a reference section, not an instruction to execute the entire file in a notebook cell.

In [ ]:
import os
import requests
from dotenv import load_dotenv

load_dotenv()

API_URL = "https://api.textbee.dev/api/v1/gateway/send-sms"

def send_sms(message):
    api_key = os.getenv("TEXTBEE_API_KEY")
    phones = os.getenv("ALERT_PHONES", "")

    recipients = [
        phone.strip()
        for phone in phones.split(",")
        if phone.strip()
    ]

    if not api_key or not recipients:
        print("Textbee configuration missing")
        return False

    headers = {
        "x-api-key": api_key,
        "Content-Type": "application/json"
    }

    data = {
        "recipients": recipients,
        "message": message
    }

    try:
        response = requests.post(
            API_URL,
            headers=headers,
            json=data,
            timeout=20
        )

        print("Textbee response:", response.status_code, response.text)
        response.raise_for_status()
        return True

    except requests.RequestException as e:
        print("SMS Error:", e)
        return False

### backend/test_routing.py

**File:** `backend/test_routing.py`

This is the actual project source copied into the notebook for inspection.
It is a reference section, not an instruction to execute the entire file in a notebook cell.

In [ ]:
from app.services.routing_service import build_risk_map, calculate_zone_routes

# Simulate zone 304 in CRITICAL state (as in FIRE scenario)
mock_sensors = [
    {'zone_id': '304', 'risk_level': 'CRITICAL', 'risk_score': 95.0, 'occupancy': 4},
    {'zone_id': '303', 'risk_level': 'HIGH',     'risk_score': 65.0, 'occupancy': 5},
    {'zone_id': '302', 'risk_level': 'MODERATE', 'risk_score': 35.0, 'occupancy': 6},
    {'zone_id': '301', 'risk_level': 'LOW',      'risk_score': 10.0, 'occupancy': 7},
    {'zone_id': '201', 'risk_level': 'LOW',      'risk_score': 8.0,  'occupancy': 8},
    {'zone_id': '202', 'risk_level': 'LOW',      'risk_score': 8.0,  'occupancy': 7},
    {'zone_id': '203', 'risk_level': 'LOW',      'risk_score': 8.0,  'occupancy': 6},
    {'zone_id': '204', 'risk_level': 'LOW',      'risk_score': 8.0,  'occupancy': 5},
    {'zone_id': '101', 'risk_level': 'LOW',      'risk_score': 8.0,  'occupancy': 6},
    {'zone_id': '102', 'risk_level': 'LOW',      'risk_score': 8.0,  'occupancy': 7},
    {'zone_id': '103', 'risk_level': 'LOW',      'risk_score': 8.0,  'occupancy': 5},
    {'zone_id': '104', 'risk_level': 'LOW',      'risk_score': 8.0,  'occupancy': 4},
]

risk_map = build_risk_map(mock_sensors)
routes = calculate_zone_routes('304', risk_map)

print("=== EVACUATION ROUTES FROM ZONE 304 (CRITICAL) ===")
for r in routes:
    print(f"  Exit: {r['exit']:<10}  Status: {r['status']:<12}  Score: {r['risk_score']}")

available  = [r for r in routes if r['status'] == 'AVAILABLE']
caution    = [r for r in routes if r['status'] == 'CAUTION']
candidates = available or caution
recommended = min(candidates, key=lambda r: r['risk_score']) if candidates else None

print()
if recommended:
    print(f"RECOMMENDED EXIT : {recommended['exit']} (score={recommended['risk_score']})")
else:
    print("RECOMMENDED EXIT : NONE — all routes blocked")

print()
print("=== TEST PASSED ===")


### backend/test_sms.py

**File:** `backend/test_sms.py`

This is the actual project source copied into the notebook for inspection.
It is a reference section, not an instruction to execute the entire file in a notebook cell.

In [ ]:
from app.services.sms_service import send_sms

send_sms("EVAC-TWIN TEST ALERT: SMS integration successful!")

### backend/requirements.txt

Configuration or command reference.

```text
fastapi
uvicorn
pymongo
python-dotenv
paho-mqtt
scikit-learn
numpy
pandas
joblib
torch
```

### backend/app/ml/models/risk_config.json

Configuration or command reference.

```json
{
    "features": [
        "temperature",
        "humidity",
        "smoke",
        "occupancy"
    ],
    "classes": [
        "CRITICAL",
        "HIGH",
        "LOW",
        "MODERATE"
    ],
    "input_size": 4,
    "hidden_sizes": [
        64,
        32
    ],
    "num_classes": 4
}
```

### backend/app/ml/models/fire_alarm_config.json

Configuration or command reference.

```json
{
    "features": [
        "Temperature[C]",
        "Humidity[%]",
        "TVOC[ppb]",
        "eCO2[ppm]",
        "PM1.0",
        "PM2.5"
    ],
    "temperature": 0.8120764493942261,
    "threshold": 0.5
}
```

### frontend/package.json

Configuration or command reference.

```json
{
  "name": "frontend",
  "private": true,
  "version": "0.0.0",
  "type": "module",
  "scripts": {
    "dev": "vite",
    "build": "vite build",
    "lint": "oxlint",
    "preview": "vite preview"
  },
  "dependencies": {
    "@react-three/drei": "^10.7.8",
    "@react-three/fiber": "^9.7.0",
    "axios": "^1.19.0",
    "lucide-react": "^1.31.0",
    "react": "^19.2.8",
    "react-dom": "^19.2.8",
    "react-router-dom": "^7.18.2",
    "recharts": "^3.10.1",
    "three": "^0.185.1"
  },
  "devDependencies": {
    "@types/react": "^19.2.17",
    "@types/react-dom": "^19.2.3",
    "@vitejs/plugin-react": "^6.0.4",
    "oxlint": "^1.75.0",
    "vite": "^8.2.0"
  }
}
```

### frontend/vite.config.js

Configuration or command reference.

```text
import { defineConfig } from 'vite'
import react from '@vitejs/plugin-react'

// https://vite.dev/config/
export default defineConfig({
  plugins: [react()],
})
```

### frontend/index.html

Configuration or command reference.

```text
<!doctype html>
<html lang="en">
  <head>
    <meta charset="UTF-8" />
    <link rel="icon" type="image/svg+xml" href="/favicon.svg" />
    <meta name="viewport" content="width=device-width, initial-scale=1.0" />
    <title>frontend</title>
  </head>
  <body>
    <div id="root"></div>
    <script type="module" src="/src/main.jsx"></script>
  </body>
</html>
```

### run_evactwin.bat

Configuration or command reference.

```powershell
@echo off
title EVAC-TWIN Launcher

echo ==========================================
echo          EVAC-TWIN COMMAND CENTER
echo ==========================================
echo.

echo Starting Mosquitto MQTT Broker...
start "EVAC-TWIN - MQTT" cmd /k "cd /d C:\Program Files\mosquitto && mosquitto.exe -v"

timeout /t 2 /nobreak >nul

echo Starting EVAC-TWIN Backend on port 8001...
start "EVAC-TWIN - Backend" cmd /k "cd /d C:\EVAC-TWIN\backend && .\.venv\Scripts\activate && python -m uvicorn app.main:app --reload --port 8001"

timeout /t 4 /nobreak >nul

echo Starting EVAC-TWIN Frontend...
start "EVAC-TWIN - Frontend" cmd /k "cd /d C:\EVAC-TWIN\frontend && npm run dev"

timeout /t 3 /nobreak >nul

echo Starting EVAC-TWIN IoT Simulator...
start "EVAC-TWIN - IoT Simulator" cmd /k "cd /d C:\EVAC-TWIN\backend && .\.venv\Scripts\activate && python -m app.iot_simulator"

echo.
echo ==========================================
echo       EVAC-TWIN SERVICES STARTED
echo ==========================================
echo.
echo MQTT     : localhost:1883
echo Backend  : http://localhost:8001
echo Frontend : http://localhost:5173
echo.
echo Opening dashboard...
timeout /t 8 /nobreak >nul

start http://localhost:5173

exit
```

### CommandsList.txt

Configuration or command reference.

```text
Required software on the new laptop

Python 3.10
Node.js + npm
MongoDB Community Server
MongoDB Shell (mongosh)
Mosquitto MQTT Broker

EVAC-TWIN — COMMANDS

1. PROJECT
cd C:\EVAC-TWIN
dir

2. PYTHON
py --list

3. BACKEND SETUP
cd C:\EVAC-TWIN\backend
py -3.10 -m venv .venv
.\.venv\Scripts\Activate.ps1
python --version
python -m pip install --upgrade pip
pip install -r requirements.txt

4. MONGODB CHECK
Test-NetConnection localhost -Port 27017

5. MOSQUITTO CHECK
Test-NetConnection localhost -Port 1883

6. START BACKEND
cd C:\EVAC-TWIN\backend
.\.venv\Scripts\Activate.ps1
python -m uvicorn app.main:app --reload --port 8001

7. BACKEND TEST
Invoke-WebRequest http://127.0.0.1:8001/api/health -UseBasicParsing

8. FRONTEND SETUP
cd C:\EVAC-TWIN\frontend
npm install

9. START FRONTEND
npm run dev

10. START IoT SIMULATOR
cd C:\EVAC-TWIN\backend
.\.venv\Scripts\Activate.ps1
python -m app.iot_simulator

11. OPEN
http://localhost:5173/monitoring

12. MONGODB DATA
mongosh
show dbs
use evac_twin
show collections
db.sensors.find().pretty()
db.events.find().pretty()
db.buildings.find().pretty()
exit

After first-time setup, every time only these 3 run:
# TERMINAL 1 — BACKEND
cd C:\EVAC-TWIN\backend
.\.venv\Scripts\Activate.ps1
python -m uvicorn app.main:app --reload --port 8001

# TERMINAL 2 — FRONTEND
cd C:\EVAC-TWIN\frontend
npm run dev

# TERMINAL 3 — IoT
cd C:\EVAC-TWIN\backend
.\.venv\Scripts\Activate.ps1
python -m app.iot_simulator
```

### frontend/src/main.jsx

Frontend source reference — not executable in this Python notebook.

```jsx
import { StrictMode } from 'react'
import { createRoot } from 'react-dom/client'
import './index.css'
import App from './App.jsx'

createRoot(document.getElementById('root')).render(
  <StrictMode>
    <App />
  </StrictMode>,
)
```

### frontend/src/App.jsx

Frontend source reference — not executable in this Python notebook.

```jsx
import { BrowserRouter, NavLink, Route, Routes } from "react-router-dom";

import AIInsights from "./pages/AIInsights";
import Analytics from "./pages/Analytics";
import Dashboard from "./pages/Dashboard";
import Evacuation from "./pages/Evacuation";
import Monitoring from "./pages/Monitoring";
import Simulation from "./pages/Simulation";

import "./App.css";

function Sidebar() {
  return (
    <aside className="sidebar">
      <div className="brand">
        <div className="brand-icon">E</div>

        <div>
          <h1>EVAC-TWIN</h1>
          <span>Emergency Intelligence</span>
        </div>
      </div>

      <nav className="navigation">

        <div className="nav-section">
          COMMAND CENTER
        </div>

        <NavLink
          to="/"
          end
          className={({ isActive }) =>
            `nav-item ${isActive ? "active" : ""}`
          }
        >
          <span>◈</span>
          Overview
        </NavLink>

        <NavLink
          to="/monitoring"
          className={({ isActive }) =>
            `nav-item ${isActive ? "active" : ""}`
          }
        >
          <span>⌁</span>
          Live Monitoring
        </NavLink>

        <NavLink
          to="/evacuation"
          className={({ isActive }) =>
            `nav-item ${isActive ? "active" : ""}`
          }
        >
          <span>◎</span>
          Evacuation
        </NavLink>

        <NavLink
          to="/simulation"
          className={({ isActive }) =>
            `nav-item ${isActive ? "active" : ""}`
          }
        >
          <span>◇</span>
          Simulation
        </NavLink>

        <div className="nav-section intelligence">
          INTELLIGENCE
        </div>

        <NavLink
          to="/ai-insights"
          className={({ isActive }) =>
            `nav-item ${isActive ? "active" : ""}`
          }
        >
          <span>⌁</span>
          AI Insights
        </NavLink>

        <NavLink
          to="/analytics"
          className={({ isActive }) =>
            `nav-item ${isActive ? "active" : ""}`
          }
        >
          <span>▦</span>
          Analytics
        </NavLink>

      </nav>

      <div className="system-card">
        <div className="system-status-dot"></div>

        <div>
          <strong>System Operational</strong>
          <span>All services online</span>
        </div>
      </div>
    </aside>
  );
}

function App() {
  return (
    <BrowserRouter>
      <div className="app-layout">

        <Sidebar />

        <main className="main-content">
          <Routes>
            <Route path="/" element={<Dashboard />} />
            <Route path="/monitoring" element={<Monitoring />} />
            <Route path="/analytics" element={<Analytics />} />
            <Route path="/evacuation" element={<Evacuation />} />
            <Route path="/simulation" element={<Simulation />} />
            <Route path="/ai-insights" element={<AIInsights />} />
          </Routes>
        </main>

      </div>
    </BrowserRouter>
  );
}

export default App;
```

### frontend/src/services/api.js

Frontend source reference — not executable in this Python notebook.

#### Lines 1–220

```javascript
const API_BASE_URL = "http://localhost:8001";


// ============================================================
// GENERIC API HELPER
// ============================================================

async function apiRequest(url) {
    const controller = new AbortController();

    const timeout = setTimeout(() => {
        controller.abort();
    }, 10000);

    try {
        const response = await fetch(url, {
            signal: controller.signal
        });

        if (!response.ok) {
            let message = `Request failed: ${response.status}`;

            try {
                const data = await response.json();

                if (data?.detail) {
                    message = data.detail;
                }
            } catch {
                // Ignore JSON parsing errors
            }

            throw new Error(message);
        }

        return await response.json();
    } catch (error) {
        if (error.name === "AbortError") {
            throw new Error("API request timed out after 10 seconds");
        }

        throw error;
    } finally {
        clearTimeout(timeout);
    }
}



// ============================================================
// CURRENT SENSOR DATA
// ============================================================

export async function getSensors() {

    return apiRequest(
        `${API_BASE_URL}/api/sensors`
    );
}


// ============================================================
// SENSOR HISTORY
// ============================================================

export async function getSensorHistory(
    zoneId,
    limit = 100
) {

    const url =
        `${API_BASE_URL}/api/sensors/history` +
        `?zone_id=${encodeURIComponent(zoneId)}` +
        `&limit=${limit}`;

    return apiRequest(url);
}


// ============================================================
// BUILDING SUMMARY
// ============================================================

export async function getBuildingSummary() {

    return apiRequest(
        `${API_BASE_URL}/api/buildings/summary`
    );
}


// ============================================================
// EVENTS
// ============================================================

export async function getEvents() {

    return apiRequest(
        `${API_BASE_URL}/api/events`
    );
}


// ============================================================
// BUILDINGS
// ============================================================

export async function getBuildings() {

    return apiRequest(
        `${API_BASE_URL}/api/buildings`
    );
}


// ============================================================
// EVACUATION — ALL ROUTES
// ============================================================

export async function getEvacuationRoutes(zoneId) {

    return apiRequest(
        `${API_BASE_URL}/api/evacuation/routes${zoneId ? `?zone_id=${zoneId}` : ""}`
    );
}

// ============================================================
// EVACUATION — SUMMARY
// ============================================================

export async function getEvacuationSummary() {

    return apiRequest(
        `${API_BASE_URL}/api/evacuation/summary`
    );
}


// ============================================================
// EVACUATION — EXITS
// ============================================================

export async function getEvacuationExits() {

    return apiRequest(
        `${API_BASE_URL}/api/evacuation/exits`
    );
}


// ============================================================
// EVACUATION — SINGLE ZONE
// ============================================================

export async function getZoneEvacuationRoute(
    deviceId,
    temperature,
    smoke,
    occupancy
) {
    const response = await fetch(
        `${API_BASE_URL}/api/evacuation/route/${encodeURIComponent(deviceId)}`,
        {
            method: "POST",
            headers: {
                "Content-Type": "application/json"
            },
            body: JSON.stringify({
                temperature,
                smoke,
                occupancy
            })
        }
    );

    if (!response.ok) {
        throw new Error("Route request failed");
    }

    return response.json();
}
export async function getAlgorithmComparison(
    deviceId,
    temperature,
    smoke,
    occupancy
) {
    const response = await fetch(
        `${API_BASE_URL}/api/evacuation/compare/${encodeURIComponent(deviceId)}`,
        {
            method: "POST",
            headers: {
                "Content-Type": "application/json"
            },
            body: JSON.stringify({
                temperature,
                smoke,
                occupancy
            })
        }
    );

    if (!response.ok) {
        throw new Error("Algorithm comparison failed");
    }

    return response.json();
}

// ============================================================
// EVACUATION — ZONE ALIAS
// ============================================================

export async function getZoneRoute(
    deviceId
) {

    return apiRequest(
        `${API_BASE_URL}/api/evacuation/zone/${encodeURIComponent(deviceId)}`
    );
```

#### Lines 221–319

```javascript
}


// ============================================================
// BACKEND HEALTH
// ============================================================

export async function getHealth() {

    return apiRequest(
        `${API_BASE_URL}/api/health`
    );
}


// ============================================================
// EVACUATION HEALTH
// ============================================================

export async function getEvacuationHealth() {

    return apiRequest(
        `${API_BASE_URL}/api/evacuation/health`
    );
}

// ============================================================
// SIMULATION — EMERGENCY SMS ALERT
// ============================================================

export async function sendSimulationAlert(
    deviceId,
    temperature,
    smoke,
    occupancy,
    scenario
) {
    const response = await fetch(
        `${API_BASE_URL}/api/evacuation/simulation/${encodeURIComponent(deviceId)}`,
        {
            method: "POST",
            headers: {
                "Content-Type": "application/json"
            },
            body: JSON.stringify({
                temperature,
                smoke,
                occupancy,
                scenario
            })
        }
    );

    if (!response.ok) {
        const data = await response.json().catch(() => ({}));

        throw new Error(
            data.detail || "Simulation SMS request failed"
        );
    }

    return response.json();
}
// ============================================================
// DEEP LEARNING — RISK PREDICTION
// ============================================================

export async function predictRiskDL(
    temperature,
    humidity,
    smoke,
    occupancy
) {
    const response = await fetch(
        `${API_BASE_URL}/api/ai/predict-risk`,
        {
            method: "POST",
            headers: {
                "Content-Type": "application/json"
            },
            body: JSON.stringify({
                temperature,
                humidity,
                smoke,
                occupancy
            })
        }
    );

    if (!response.ok) {
        const data = await response.json().catch(() => ({}));

        throw new Error(
            data.detail || "Deep Learning risk prediction failed"
        );
    }

    return response.json();
}
```

### frontend/src/utils/riskEngine.js

Frontend source reference — not executable in this Python notebook.

```javascript

// ============================================================
// EVAC-TWIN - DETERMINISTIC RISK ENGINE
// ============================================================

export function calculateRisk(sensor) {
    if (!sensor) {
        return {
            score: 0,
            level: "NO DATA",
            recommendation: "Waiting for sensor data.",
            factors: {
                temperature: 0,
                smoke: 0,
                occupancy: 0,
            },
        };
    }

    const temperature = Math.max(
        Number(sensor.temperature ?? 0),
        0
    );

    const smoke = Math.max(
        Number(sensor.smoke ?? 0),
        0
    );

    const occupancy = Math.max(
        Number(sensor.occupancy ?? 0),
        0
    );

    // Temperature risk - Maximum 35
    let temperatureRisk = 0;

    if (temperature >= 50) {
        temperatureRisk = 35;
    } else if (temperature >= 45) {
        temperatureRisk = 30;
    } else if (temperature >= 40) {
        temperatureRisk = 25;
    } else if (temperature >= 35) {
        temperatureRisk = 15;
    } else if (temperature >= 32) {
        temperatureRisk = 5;
    }

    // Smoke risk - Maximum 40
    let smokeRisk = 0;

    if (smoke >= 0.8) {
        smokeRisk = 40;
    } else if (smoke >= 0.6) {
        smokeRisk = 35;
    } else if (smoke >= 0.5) {
        smokeRisk = 30;
    } else if (smoke >= 0.3) {
        smokeRisk = 15;
    } else if (smoke >= 0.15) {
        smokeRisk = 5;
    }

    // Occupancy risk - Maximum 25
    let occupancyRisk = 0;

    if (occupancy >= 50) {
        occupancyRisk = 25;
    } else if (occupancy >= 40) {
        occupancyRisk = 20;
    } else if (occupancy >= 30) {
        occupancyRisk = 15;
    } else if (occupancy >= 15) {
        occupancyRisk = 5;
    }

    // Total risk score
    const score = Math.min(
        temperatureRisk + smokeRisk + occupancyRisk,
        100
    );

    // Risk classification
    let level = "LOW";

    let recommendation =
        "Environmental conditions are stable. Continue normal monitoring.";

    if (score >= 70) {
        level = "CRITICAL";
        recommendation =
            "Immediate evacuation recommended. Activate emergency procedures.";
    } else if (score >= 40) {
        level = "HIGH";
        recommendation =
            "High-risk conditions detected. Prepare evacuation routes and alert occupants.";
    } else if (score >= 20) {
        level = "MODERATE";
        recommendation =
            "Elevated conditions detected. Continue close monitoring and prepare for escalation.";
    }

    return {
        score,
        level,
        recommendation,
        factors: {
            temperature: temperatureRisk,
            smoke: smokeRisk,
            occupancy: occupancyRisk,
        },
    };
}


// ============================================================
// RISK TEXT COLOR
// ============================================================

export function getRiskColor(level) {
    const riskLevel = String(level || "").toUpperCase();

    switch (riskLevel) {
        case "CRITICAL":
            return "#dc2626";

        case "HIGH":
            return "#ea580c";

        case "MODERATE":
            return "#d97706";

        case "LOW":
            return "#16a34a";

        default:
            return "#64748b";
    }
}


// ============================================================
// RISK BACKGROUND COLOR
// ============================================================

export function getRiskBackground(level) {
    const riskLevel = String(level || "").toUpperCase();

    switch (riskLevel) {
        case "CRITICAL":
            return "#fee2e2";

        case "HIGH":
            return "#ffedd5";

        case "MODERATE":
            return "#fef3c7";

        case "LOW":
            return "#dcfce7";

        default:
            return "#f1f5f9";
    }
}
```

### frontend/src/pages/Dashboard.jsx

Frontend source reference — not executable in this Python notebook.

#### Lines 1–220

```jsx
import { useNavigate } from "react-router-dom";
import { useEffect, useMemo, useState } from "react";

import {
    getBuildingSummary,
    getEvacuationRoutes,
    getEvents,
    getSensors,
} from "../services/api";

import "./Dashboard.css";


function Dashboard() {
    const [simulationInput, setSimulationInput] = useState(() => {
  try {
    return JSON.parse(localStorage.getItem("simulationInput") || "null");
  } catch {
    return null;
  }
});

    const [sensors, setSensors] = useState([]);
    const [evacuationData, setEvacuationData] = useState(null);
    const [events, setEvents] = useState([]);
    const [buildingSummary, setBuildingSummary] = useState(null);

    const [error, setError] = useState("");
    const [evacuationError, setEvacuationError] = useState("");

    const [lastSync, setLastSync] = useState(null);

    const navigate = useNavigate();

const totalOccupancy = sensors.reduce(
    (total, sensor) =>
        total + (Number(sensor.occupancy) || 0),
    0
);



    // =========================================================
    // LIVE DATA
    // =========================================================

// =========================================================
// LIVE DATA
// =========================================================

useEffect(() => {
    let mounted = true;
    let loading = false;

    const loadDashboardData = async () => {
        if (loading) return;

        loading = true;

        try {
            const results = await Promise.allSettled([
                getSensors(),
                getEvents(),
                getEvacuationRoutes(),
                getBuildingSummary(),
            ]);

            if (!mounted) return;

            const [sensorResult, eventResult, evacuationResult, summaryResult] = results;

            if (sensorResult.status === "fulfilled") {
                setSensors(
                    Array.isArray(sensorResult.value)
                        ? sensorResult.value
                        : []
                );
                setError("");
            } else {
                console.error("Sensor API Error:", sensorResult.reason);
                setError("Sensor data temporarily unavailable. Retrying...");
            }

            if (eventResult.status === "fulfilled") {
                setEvents(
                    Array.isArray(eventResult.value)
                        ? eventResult.value
                        : []
                );
            } else {
                console.error("Events API Error:", eventResult.reason);
            }

            if (evacuationResult.status === "fulfilled") {
                setEvacuationData(evacuationResult.value || null);
                setEvacuationError("");
            } else {
                console.error("Evacuation API Error:", evacuationResult.reason);
                setEvacuationError("Evacuation routing temporarily unavailable.");
            }

            if (summaryResult.status === "fulfilled") {
                setBuildingSummary(summaryResult.value || null);
            } else {
                console.error("Building Summary API Error:", summaryResult.reason);
            }

            setLastSync(new Date());
        } finally {
            loading = false;
        }
    };

    loadDashboardData();

    const interval = setInterval(loadDashboardData, 3000);

    return () => {
        mounted = false;
        clearInterval(interval);
    };
}, []);




    // =========================================================
    // HELPERS
    // =========================================================

    const getZoneId = (sensor) => {

        if (!sensor) return "";

        if (
            sensor.zone_id !== undefined &&
            sensor.zone_id !== null
        ) {
            return String(sensor.zone_id);
        }

        if (sensor.device_id) {

            return String(
                sensor.device_id
            ).replace(/^sensor-/, "");

        }

        return "";
    };




const getRiskLevel = (sensor) => {
    if (!sensor) return "LOW";

    const score = getRiskScore(sensor);

    if (score >= 70) return "CRITICAL";
    if (score >= 40) return "HIGH";
    if (score >= 20) return "MODERATE";

    return "LOW";
};





    const getRiskScore = (sensor) => {

        if (!sensor) return 0;

        const value =
            sensor.risk_score ??
            sensor.score ??
            0;

        const numeric = Number(value);

        if (Number.isNaN(numeric)) return 0;

        return Math.max(
            0,
            Math.min(100, numeric)
        );
    };


    const getSensorForZone = (zoneId) => {

        return sensors.find(
            sensor =>
                getZoneId(sensor) ===
                String(zoneId)
        );

    };


    const getRiskClassForZone = (sensor) => {

        if (!sensor) {
            return "room-normal";
        }

        const level = getRiskLevel(sensor);

        if (level === "CRITICAL") {
            return "room-critical";
        }

        if (
            level === "HIGH" ||
            level === "MODERATE"
        ) {
            return "room-warning";
        }
```

#### Lines 221–440

```jsx

        return "room-normal";
    };



// =========================================================
// LATEST SENSOR
// =========================================================

const latest = useMemo(() => {
    if (!sensors.length) {
        return null;
    }

    return [...sensors].sort((a, b) => {
        const scoreDifference =
            getRiskScore(b) - getRiskScore(a);

        if (scoreDifference !== 0) {
            return scoreDifference;
        }

        return Number(getZoneId(b)) - Number(getZoneId(a));
    })[0];
}, [sensors]);





    // =========================================================
    // ACTIVE EVENT
    // =========================================================




const activeEvent = useMemo(() => {
    const priority = {
        CRITICAL: 4,
        HIGH: 3,
        MODERATE: 2,
        LOW: 1,
    };

    return events
        .filter((event) => {
            const status = String(event.status || "").toUpperCase();
            const score = Number(event.risk_score ?? event.score ?? 0);
            const type = String(event.event_type || "").toUpperCase();

            if (status !== "ACTIVE") return false;

            // Old fire predictions with low risk scores
            // should not automatically become critical alerts.
            if (
                type === "FIRE_ALARM_PREDICTED" &&
                score < 70
            ) {
                return false;
            }

            return true;
        })
        .sort((a, b) => {
            const levelA = String(
                a.level || a.risk_level || a.severity || "LOW"
            ).toUpperCase();

            const levelB = String(
                b.level || b.risk_level || b.severity || "LOW"
            ).toUpperCase();

            return (priority[levelB] || 0) - (priority[levelA] || 0);
        })[0] || null;
}, [events]);

console.log("Selected active event:", activeEvent);


    // =========================================================
    // MOST DANGEROUS SENSOR
    // =========================================================

    const mostDangerousSensor = useMemo(() => {

        if (!sensors.length) {
            return null;
        }

        return [...sensors].sort(
            (a, b) =>
                getRiskScore(b) -
                getRiskScore(a)
        )[0];

    }, [sensors]);



// =========================================================
// FINAL RISK
// =========================================================

const risk = useMemo(() => {

    const buildingRisk = {
        level: String(
            buildingSummary?.risk_level ||
            buildingSummary?.level ||
buildingSummary?.overall_risk ||
"LOW"
        ).toUpperCase(),

        score: Number(
            buildingSummary?.risk_score ??
            buildingSummary?.score ??
            0
        ),
    };

    if (activeEvent) {

        const level = String(
            activeEvent.level ||
            activeEvent.risk_level ||
            ""
        ).toUpperCase();

        const score = Number(
            activeEvent.score ??
            activeEvent.risk_score ??
            0
        );

        return {
            level: level || buildingRisk.level,
            score: activeEvent.score != null ||
                activeEvent.risk_score != null
                ? score
                : buildingRisk.score,

            recommendation:
                activeEvent.recommendation ||
                "Emergency condition detected.",
        };
    }

    return {
        level: buildingRisk.level,
        score: buildingRisk.score,

        recommendation:
            buildingRisk.level === "CRITICAL"
                ? "Immediate evacuation recommended."
                : buildingRisk.level === "HIGH"
                    ? "High-risk conditions detected."
                    : buildingRisk.level === "MODERATE"
                        ? "Elevated conditions detected."
                        : "No significant hazards detected.",
    };

}, [activeEvent, buildingSummary]);


const riskScore = latest
    ? getRiskScore(latest)
    : 0;

const riskLevel = getRiskLevel(latest);


    // =========================================================
    // BUILDING STATUS
    // =========================================================




const buildingStatus = useMemo(() => {
    if (!sensors.length) return "WAITING";

    const levels = sensors.map((sensor) => {
        const score = getRiskScore(sensor);

        if (score >= 70) return "CRITICAL";
        if (score >= 40) return "HIGH";
        if (score >= 20) return "MODERATE";

        return "LOW";
    });

    if (levels.includes("CRITICAL")) return "EVACUATE";
    if (levels.includes("HIGH") || levels.includes("MODERATE")) {
        return "WARNING";
    }

    return "NORMAL";
}, [sensors]);




    // =========================================================
    // RISK CLASS
    // =========================================================

    const getRiskClass = () => {

        if (riskLevel === "CRITICAL") {
            return "critical";
        }

        if (riskLevel === "HIGH") {
            return "high";
        }

        if (riskLevel === "MODERATE") {
            return "moderate";
```

#### Lines 441–660

```jsx
        }

        return "low";
    };


    // =========================================================
    // SENSOR STATUS
    // =========================================================

    const getTemperatureStatusForSensor = (sensor) => {

        const temperature =
            Number(sensor?.temperature || 0);


        if (temperature >= 40) {
            return "Critical";
        }

        if (temperature >= 38) {
            return "Elevated";
        }

        if (temperature >= 35) {
            return "Moderate";
        }

        return "Stable";
    };


    const getSmokeStatusForSensor = (sensor) => {

        const smoke =
            Number(sensor?.smoke || 0);


        if (smoke >= 0.9) {
            return "Critical";
        }

        if (smoke >= 0.7) {
            return "High";
        }

        if (smoke >= 0.5) {
            return "Moderate";
        }

        return "Normal";
    };


    const getOccupancyStatusForSensor = (sensor) => {

        const occupancy =
            Number(sensor?.occupancy || 0);


        if (occupancy >= 48) {
            return "Critical";
        }

        if (occupancy >= 40) {
            return "High";
        }

        if (occupancy >= 30) {
            return "Moderate";
        }

        return "Normal";
    };


    const getDLConfidence = () => {
    const sensor = mostDangerousSensor || latest;

    if (!sensor) return null;

    const confidence =
        sensor.dl_confidence ??
        sensor.confidence;

    if (
        confidence === undefined ||
        confidence === null ||
        confidence === ""
    ) {
        return null;
    }

    const numeric = Number(confidence);

    if (Number.isNaN(numeric)) return null;

    if (numeric > 0 && numeric <= 1) {
        return Math.round(numeric * 100);
    }

    return Math.round(numeric);
};

const dlConfidence = getDLConfidence();

    // =========================================================
    // EVENT HELPERS
    // =========================================================

    const getEventLevel = (event) => {

        if (!event) {
            return null;
        }


        const explicitLevel =
    event.event_type === "FIRE_ALARM_PREDICTED"
        ? event.severity || "CRITICAL"
        : event.level || event.risk_level || event.severity;


        if (explicitLevel) {

            return String(
                explicitLevel
            ).toUpperCase();
        }


        const score =
            Number(
                event.score ??
                event.risk_score ??
                0
            );


        if (score >= 75) {
            return "CRITICAL";
        }

        if (score >= 60) {
            return "HIGH";
        }

        if (score >= 35) {
            return "MODERATE";
        }

        return "LOW";
    };



const getEventMessage = (event) => {
    if (!event) {
        return "";
    }

    const level = getEventLevel(event);

    if (level === "CRITICAL") {
        return "CRITICAL EMERGENCY! Evacuate the affected area immediately using the recommended route, after confirming the exit is clear.";
    }

    if (level === "HIGH") {
        return "HIGH RISK DETECTED! Avoid the affected zone and follow the recommended evacuation route. Verify that the route is clear.";
    }

    if (level === "MODERATE") {
        return "WARNING! Elevated risk conditions detected. Stay alert and monitor the recommended evacuation route.";
    }

    return (
        event.recommendation ||
        event.message ||
        event.description ||
        "Conditions are currently normal. Continue monitoring live sensor readings."
    );
};




// =========================================================
// LIVE SENSOR EMERGENCIES
// =========================================================



const sensorEmergencies = useMemo(() => {
    return sensors
        .map((sensor) => {
            const score = getRiskScore(sensor);

            let level = "LOW";

            if (score >= 70) level = "CRITICAL";
            else if (score >= 40) level = "HIGH";
            else if (score >= 20) level = "MODERATE";

            return {
                ...sensor,
                level,
                risk_score: score,
                device_id:
                    sensor.device_id ||
                    `sensor-${getZoneId(sensor)}`,
            };
        })
        .filter((sensor) =>
            ["CRITICAL", "HIGH", "MODERATE"].includes(sensor.level)
        )
        .sort((a, b) => b.risk_score - a.risk_score);
}, [sensors]);

const sensorEmergency = sensorEmergencies[0] || null;
```

#### Lines 661–880

```jsx
const displayEvent = activeEvent || sensorEmergency || null;



const alertZoneId = String(
    displayEvent?.zone_id ??
    displayEvent?.device_id?.replace(/^sensor-/, "") ??
    latest?.zone_id ??
    latest?.device_id?.replace(/^sensor-/, "") ??
    ""
);

const alertZoneRoute = evacuationData?.zones?.find(
    (zone) => String(zone.zone_id) === alertZoneId
);

const recommendedExit =
    alertZoneRoute?.recommended_exit ??
    (!alertZoneId ? evacuationData?.recommended_exit : null);

const alternativeExits =
    alertZoneRoute?.alternative_exits ??
    (!alertZoneId ? evacuationData?.alternative_exits : []) ??
    [];




    // =========================================================
    // TIME
    // =========================================================

    const formatTime = (date) => {

        if (!date) {
            return "--:--:--";
        }


        try {

            return new Date(
                date
            ).toLocaleTimeString(
                [],
                {
                    hour: "2-digit",
                    minute: "2-digit",
                    second: "2-digit",
                }
            );

        } catch {

            return "--:--:--";
        }
    };


    // =========================================================
    // BUILDING ALPHA
    // =========================================================

    const floors = [
        {
            floor: 3,
            zones: [
                "301",
                "302",
                "303",
                "304",
            ],
        },
        {
            floor: 2,
            zones: [
                "201",
                "202",
                "203",
                "204",
            ],
        },
        {
            floor: 1,
            zones: [
                "101",
                "102",
                "103",
                "104",
            ],
        },
    ];


    // =========================================================
    // LOADING
    // =========================================================

    if (!latest && !error) {

        return (

            <div className="dashboard">

                <header className="dashboard-topbar">

                    <div>

                        <p className="dashboard-eyebrow">
                            DIGITAL TWIN / COMMAND CENTER
                        </p>

                        <h1>
                            Building Overview
                        </h1>

                    </div>


                    <div className="dashboard-live-status">

                        <span className="dashboard-live-dot"></span>

                        <strong>
                            LIVE
                        </strong>

                        <span className="dashboard-divider"></span>

                        Synchronizing sensors...

                    </div>

                </header>


                <div className="dashboard-loading">

                    <div className="loading-ring"></div>

                    <h2>
                        Connecting to Digital Twin
                    </h2>

                    <p>
                        Waiting for live virtual sensor data...
                    </p>

                </div>

            </div>
        );
    }


    // =========================================================
    // MAIN
    // =========================================================

    return (

        <div className="dashboard">


            {/* HEADER */}

            <header className="dashboard-topbar">

                <div>

                    <p className="dashboard-eyebrow">
                        DIGITAL TWIN / COMMAND CENTER
                    </p>

                    <h1>
                        Building Overview
                    </h1>

                </div>


                <div
                    className={`dashboard-live-status ${buildingStatus !== "NORMAL"
                        ? "warning-status"
                        : ""
                        }`}
                >

                    <span className="dashboard-live-dot"></span>

                    <strong>
                        LIVE
                    </strong>

                    <span className="dashboard-divider"></span>

                    {buildingStatus === "NORMAL"
                        ? "All systems normal"
                        : buildingStatus === "WARNING"
                            ? "Elevated conditions detected"
                            : buildingStatus === "EVACUATE"
                                ? "Emergency conditions"
                                : "Waiting for sensor data"
                    }

                </div>

            </header>


            {/* ERRORS */}

            {error && (
                <div className="dashboard-error">
                    ⚠️ {error}
                </div>
            )}


            {evacuationError && (
```

#### Lines 881–1100

```jsx
                <div className="dashboard-error">
                    ⚠️ {evacuationError}
                </div>
            )}


{/* SIMULATION RESULT */}
{simulationInput && (
    <section className="dashboard-panel simulation-panel">
        <div className="simulation-header">
            <div>
                <span className="panel-label">

                </span>
                <h2>Latest Simulation Result</h2>
                <p>Scenario analysis · Zone {simulationInput.zone_id}</p>
            </div>

            <span className="simulation-badge">
                {simulationInput.risk_level || "SIMULATED"}
            </span>
        </div>

        <div className="simulation-grid">
            <div className="simulation-metric">
                <span>Temperature</span>
                <strong>{simulationInput.temperature}°C</strong>
            </div>

            <div className="simulation-metric">
                <span>Smoke Level</span>
                <strong>{simulationInput.smoke}</strong>
            </div>

            <div className="simulation-metric">
                <span>Occupancy</span>
                <strong>{simulationInput.occupancy}</strong>
            </div>

            <div className="simulation-metric">
                <span>Risk Score</span>
                <strong>
                    {simulationInput.risk_score ?? "--"}
                    {simulationInput.risk_score != null ? "/100" : ""}
                </strong>
            </div>
        </div>
    </section>
)}

            {/* =================================================
                EMERGENCY ALERT
            ================================================= */}

            {displayEvent && (

                <section className="emergency-alert">




                    <div className="emergency-alert-content">

                        <span className="emergency-alert-label">

                            {activeEvent
                                ? "ACTIVE EMERGENCY EVENT"
                                : "LIVE SENSOR ALERT"
                            }

                        </span>


<h2>
    {displayEvent.event_type === "FIRE_ALARM_PREDICTED"
        ? "FIRE ALARM DETECTED"
        : `${getEventLevel(displayEvent)} RISK DETECTED`}
</h2>


                        <p>
                            {getEventMessage(
                                displayEvent
                            )}
                        </p>


                        <div className="emergency-alert-meta">

                            <span>
                                Zone:
                                <strong>
                                    {" "}
                                    {displayEvent.device_id ||
    (displayEvent.zone_id != null
        ? `sensor-${displayEvent.zone_id}`
        : latest?.device_id) ||
    "--"}
                                </strong>
                            </span>


                            <span>
                                Risk Score:
                                <strong>
                                    {" "}
                                    {displayEvent.score ??
                                        displayEvent.risk_score ??
                                        riskScore
                                    }
                                    /100
                                </strong>
                            </span>


                            <span>
                                Temperature:
                                <strong>
                                    {" "}
                                    {displayEvent.temperature ??
                                        latest?.temperature ??
                                        "--"
                                    }°C
                                </strong>
                            </span>


                            <span>
                                Smoke:
                                <strong>
                                    {" "}
                                    {displayEvent.smoke ??
                                        latest?.smoke ??
                                        "--"
                                    }
                                </strong>
                            </span>


                            <span>
                                Occupancy:
                                <strong>
                                    {" "}
                                    {displayEvent.occupancy ??
                                        latest?.occupancy ??
                                        "--"
                                    }
                                </strong>
                            </span>
</div>

{/* RECOMMENDED EVACUATION ROUTE */}
{recommendedExit && (
    <div className="recommended-route-panel">
        <h3>RECOMMENDED EVACUATION ROUTE</h3>

        <p>
            <strong>Recommended Exit: </strong>
            {recommendedExit.exit} — {recommendedExit.name}
        </p>

        <p>
            <strong>Status: </strong>
            {recommendedExit.status}
        </p>

        <p>
            <strong>Safe Route: </strong>
            {(recommendedExit.path || []).join(" → ")}
        </p>

        {alternativeExits.map((exit) => (
            <p key={exit.exit}>
                <strong>Alternative Exit: </strong>
                {exit.exit} — {exit.name}
            </p>
        ))}
    </div>
)}

</div>


<button
    className="emergency-alert-button"
    onClick={() => {
        if (alertZoneId) {
            localStorage.setItem(
                "selectedZone",
                String(alertZoneId)
            );
        }

        navigate("/evacuation");
    }}
>
    EVACUATION CONTROL →
</button>


                </section>
            )}


            {/* =================================================
                ALL AFFECTED ZONES — every HIGH / CRITICAL /
                MODERATE zone shown as a separate alert card.
            ================================================= */}

            {sensorEmergencies.length >= 1 && (
                <section className="affected-zones-section">

                    <div className="affected-zones-header">
                        <span className="affected-zones-label">AFFECTED ZONES</span>
                        <span className="affected-zones-count">
                            {sensorEmergencies.length} zone{sensorEmergencies.length !== 1 ? "s" : ""} require attention
                        </span>
                    </div>

                    <div className="affected-zones-grid">
```

#### Lines 1101–1320

```jsx
                        {sensorEmergencies.map((sensor) => {
                            const zid = getZoneId(sensor);
                            const level = getRiskLevel(sensor);
                            const score = getRiskScore(sensor);
                            const levelClass =
                                level === "CRITICAL" ? "az-critical"
                                : level === "HIGH"   ? "az-high"
                                : "az-moderate";
                            return (
                                <div
                                    key={`az-${zid}`}
                                    className={`affected-zone-card ${levelClass}`}
                                >
                                    <div className="az-card-top">
                                        <span className="az-zone-id">Zone {zid}</span>
                                        <span className={`az-badge ${levelClass}`}>{level}</span>
                                    </div>

                                    <div className="az-score-bar-wrapper">
                                        <div
                                            className={`az-score-bar-fill ${levelClass}`}
                                            style={{ width: `${score}%` }}
                                        />
                                        <span className="az-score-label">{score}/100</span>
                                    </div>

                                    <div className="az-readings">
                                        <span>🌡️ {sensor.temperature !== undefined ? `${Number(sensor.temperature).toFixed(1)}°C` : "--"}</span>
                                        <span>💨 Smoke {sensor.smoke !== undefined ? Number(sensor.smoke).toFixed(3) : "--"}</span>
                                        <span>👥 {sensor.occupancy ?? "--"} people</span>
                                    </div>

                                    {sensor.recommendation && (
                                        <p className="az-recommendation">
                                            {sensor.recommendation}
                                        </p>
                                    )}

                                    {sensor.timestamp && (
                                        <span className="az-timestamp">
                                            Updated: {formatTime(sensor.timestamp)}
                                        </span>
                                    )}

                                    {sensor.fire_alarm_dl_prediction === 1 && (
    <span className="az-fire-badge">🔥 FIRE MODEL ALERT</span>
)}
                                </div>
                            );
                        })}
                    </div>

                </section>
            )}


            {/* =================================================
                STATS
            ================================================= */}

            <section className="stats-grid">


                <div className="stat-card">

                    <span className="stat-label">
                        BUILDING STATUS
                    </span>

                    <strong
                        className={
                            buildingStatus === "NORMAL"
                                ? "safe-text"
                                : buildingStatus === "WARNING"
                                    ? "warning-text"
                                    : "critical-text"
                        }
                    >
                        {buildingStatus}
                    </strong>

                    <span className="stat-description">

                        {buildingStatus === "NORMAL"
                            ? "No active emergencies"
                            : buildingStatus === "WARNING"
                                ? "Attention required"
                                : buildingStatus === "EVACUATE"
                                    ? "Immediate action required"
                                    : "Waiting for sensor data"
                        }

                    </span>

                </div>


                <div className="stat-card">

                    <span className="stat-label">
                        TOTAL OCCUPANCY
                    </span>

                    <strong>
                        {totalOccupancy}
                    </strong>

                    <span className="stat-description">
                        People across monitored zones
                    </span>

                </div>


                <div className="stat-card">

                    <span className="stat-label">
                        ACTIVE SENSORS
                    </span>

                    <strong>
                        {sensors.length} / {
                            buildingSummary?.total_zones ??
                            floors.reduce(
                                (total, floor) =>
                                    total + floor.zones.length,
                                0
                            )
                        }
                    </strong>

                    <span className="stat-description">
                        Virtual zones responding
                    </span>

                </div>


                <div className="stat-card">
    <span className="stat-label">
        AI RISK SCORE
    </span>

    <strong
    className={
        (mostDangerousSensor?.dl_prediction || "LOW") === "LOW"
            ? "safe-text"
            : (mostDangerousSensor?.dl_prediction || "").toUpperCase() === "MODERATE"
                ? "warning-text"
                : "critical-text"
    }
>
    {mostDangerousSensor?.risk_score ?? riskScore}%
</strong>

<span className="stat-description">
    {mostDangerousSensor?.dl_prediction || "No prediction"} predicted risk
</span>
</div>

            </section>


            {/* =================================================
                MAIN GRID
            ================================================= */}

            <section className="dashboard-main-grid">


                {/* =================================================
                    BUILDING ALPHA
                ================================================= */}

                <div className="dashboard-panel twin-panel">

                    <div className="panel-header">

                        <div>

                            <span className="panel-label">
                                LIVE DIGITAL TWIN
                            </span>

                            <h2>
                                Building Alpha
                            </h2>

                        </div>


                        <button
                            className="panel-button"
                            onClick={() =>
                                navigate("/monitoring")
                            }
                        >
                            Open Monitoring →
                        </button>

                    </div>


                    <div className="building-view">

                        <div
                            className={`building ${getRiskClass()}`}
                        >

                            {floors.map(
                                ({
                                    floor,
                                    zones,
                                }) => (

                                    <div
                                        key={floor}
                                        className="building-floor"
                                    >
```

#### Lines 1321–1540

```jsx
                                        <div className="floor-label">

                                            FLOOR

                                            <b>
                                                0{floor}
                                            </b>

                                        </div>


                                        <div className="rooms">

                                            {zones.map(
                                                zoneId => {

                                                    const sensor =
                                                        getSensorForZone(
                                                            zoneId
                                                        );

                                                    const score =
                                                        getRiskScore(
                                                            sensor
                                                        );

                                                    const level =
                                                        getRiskLevel(
                                                            sensor
                                                        );


                                                    return (

                                                        <div
                                                            key={zoneId}
                                                            className={`building-room ${getRiskClassForZone(
                                                                sensor
                                                            )}`}
                                                            title={
                                                                sensor
                                                                    ? `Zone ${zoneId} • ${level} • Risk ${score}% • ${sensor.temperature ?? "--"}°C • Occupancy ${sensor.occupancy ?? "--"}`
                                                                    : `Zone ${zoneId} • No sensor data`
                                                            }
                                                        >

                                                            <div className="room-info">

                                                                <strong>
                                                                    {zoneId}
                                                                </strong>

                                                                {sensor && (
                                                                    <span>
                                                                        {score.toFixed(1)}%
                                                                    </span>
                                                                )}

                                                            </div>

                                                        </div>

                                                    );

                                                }
                                            )}

                                        </div>

                                    </div>

                                )
                            )}

                        </div>


                        <div className="building-meta">

                            <span>
                                {buildingSummary?.building_id
                                    ? String(
                                        buildingSummary.building_id
                                    ).toUpperCase()
                                    : "BUILDING ALPHA"
                                }
                            </span>

                            <span>
                                {buildingSummary?.total_zones ??
                                    floors.reduce(
                                        (total, floor) =>
                                            total + floor.zones.length,
                                        0
                                    )
                                } monitored zones
                            </span>

                        </div>


                        <div className="twin-sync">

                            <span className="sync-pulse"></span>

                            DIGITAL TWIN SYNCHRONIZED

                        </div>

                    </div>

                </div>


                {/* =================================================
                    AI RISK
                ================================================= */}

                <div className="dashboard-panel risk-panel">

                    <div className="panel-header">

                        <div>

                            <span className="panel-label">
                                AI INTELLIGENCE
                            </span>

                            <h2>
                                Risk Analysis
                            </h2>

                        </div>

                    </div>


                    <div
                        className={`risk-circle ${getRiskClass()}`}
                    >

                        <strong>
                            {riskScore}%
                        </strong>

                        <span>
                            {riskLevel}
                        </span>

                    </div>


                    <div className="risk-details">

                        <div className="risk-row">
    <span>DL prediction</span>
    <strong>
        {mostDangerousSensor?.dl_prediction || "NO PREDICTION"}
    </strong>
</div>

<div className="risk-row">
    <span>DL confidence</span>
    <strong>
        {dlConfidence !== null
            ? `${dlConfidence}%`
            : "Unavailable"}
    </strong>
</div>

                        <div className="risk-row">

                            <span>
                                Temperature
                            </span>

                            <strong>

                                {mostDangerousSensor
                                    ? getTemperatureStatusForSensor(
                                        mostDangerousSensor
                                    )
                                    : "Waiting"
                                }

                            </strong>

                        </div>


                        <div className="risk-row">

                            <span>
                                Smoke levels
                            </span>

                            <strong>

                                {mostDangerousSensor
                                    ? getSmokeStatusForSensor(
                                        mostDangerousSensor
                                    )
                                    : "Waiting"
                                }

                            </strong>

                        </div>


                        <div className="risk-row">

                            <span>
                                Occupancy load
                            </span>

                            <strong>

                                {mostDangerousSensor
                                    ? getOccupancyStatusForSensor(
```

#### Lines 1541–1760

```jsx
                                        mostDangerousSensor
                                    )
                                    : "Waiting"
                                }

                            </strong>

                        </div>

                    </div>


                    <button
                        className="ai-button"
                        onClick={() =>
                            navigate("/ai-insights")
                        }
                    >
                        View AI Insights →
                    </button>

                </div>

            </section>


            {/* =================================================
                SENSOR OVERVIEW
            ================================================= */}

            <section className="sensor-overview">

                <div className="panel-header">

                    <div>

                        <span className="panel-label">
                            ENVIRONMENT MONITOR
                        </span>

                        <h2>
                            Current Sensor Conditions
                        </h2>

                    </div>


                    <span className="sensor-device">

                        {latest?.device_id ||
                            "LIVE SENSOR NETWORK"
                        }

                    </span>

                </div>


                <div className="dashboard-sensor-grid">


                    <div className="dashboard-sensor-card">

                        <span className="dashboard-sensor-icon">
                            🌡️
                        </span>

                        <div>

                            <span>
                                TEMPERATURE
                            </span>

                            <strong>
                                {latest?.temperature ?? "--"}°C
                            </strong>

                        </div>

                    </div>


                    <div className="dashboard-sensor-card">

                        <span className="dashboard-sensor-icon">
                            💧
                        </span>

                        <div>

                            <span>
                                HUMIDITY
                            </span>

                            <strong>
                                {latest?.humidity ?? "--"}%
                            </strong>

                        </div>

                    </div>


                    <div className="dashboard-sensor-card">

                        <span className="dashboard-sensor-icon">
                            💨
                        </span>

                        <div>

                            <span>
                                SMOKE LEVEL
                            </span>

                            <strong>
                                {latest?.smoke ?? "--"}
                            </strong>

                        </div>

                    </div>


                    <div className="dashboard-sensor-card">

                        <span className="dashboard-sensor-icon">
                            👥
                        </span>

                        <div>

                            <span>
                                OCCUPANCY
                            </span>

                            <strong>
                                {latest?.occupancy ?? "--"}
                            </strong>

                        </div>

                    </div>

                </div>

            </section>


            {/* =================================================
                BOTTOM GRID
            ================================================= */}

            <section className="dashboard-bottom-grid">


                {/* EVENTS */}

                <div className="dashboard-panel activity-panel">

                    <div className="panel-header">

                        <div>

                            <span className="panel-label">
                                EVENT MONITOR
                            </span>

                            <h2>
                                Recent Activity
                            </h2>

                        </div>


                       {events.some(
    event => String(event.status || "").toUpperCase() === "ACTIVE"
) && (

                            <span className="sensor-device">

                                {events.filter(
    event => String(event.status || "").toUpperCase() === "ACTIVE"
).length} ACTIVE EVENTS

                            </span>

                        )}

                    </div>


                    {displayEvent ? (

                        <div className="event-row">

                            <span
                                className={
                                    getEventLevel(
                                        displayEvent
                                    ) === "CRITICAL"
                                        ? "event-indicator danger"
                                        : "event-indicator warning"
                                }
                            ></span>


                            <div>

                                <strong>
                                    {getEventLevel(
                                        displayEvent
                                    )} risk event
                                </strong>

                                <span>
                                    {getEventMessage(
                                        displayEvent
                                    )}
                                </span>
```

#### Lines 1761–1922

```jsx

                            </div>


                            <time>
                                {formatTime(
                                    displayEvent.timestamp
                                )}
                            </time>

                        </div>

                    ) : !events.some(
    event => String(event.status || "").toUpperCase() === "ACTIVE"
) ? (

                        <div className="no-events">

                            <span className="no-events-icon">
                                ✓
                            </span>

                            <div>

                                <strong>
                                    No active emergency events
                                </strong>

                                <span>
                                    Building systems are being monitored
                                </span>

                            </div>

                        </div>

                    ) : (

    events
        .filter(
            event =>
                String(event.status || "").toUpperCase() === "ACTIVE"
        )
        .slice(0, 4)
        .map((event, index) => {

            const level = getEventLevel(event);

            return (
                <div
                    className="event-row"
                    key={`${event.timestamp || "event"}-${index}`}
                >
                    <span
                        className={
                            level === "CRITICAL"
                                ? "event-indicator danger"
                                : level === "HIGH" ||
                                  level === "MODERATE"
                                    ? "event-indicator warning"
                                    : "event-indicator safe"
                        }
                    ></span>

                    <div>
                        <strong>{level} risk event</strong>
                        <span>{getEventMessage(event)}</span>
                    </div>

                    <time>{formatTime(event.timestamp)}</time>
                </div>
            );
        })

)}

                </div>


                {/* QUICK ACTION */}

                <div className="dashboard-panel quick-panel">

                    <span className="panel-label">
                        QUICK ACTION
                    </span>


                    <h2>
                        Emergency Simulation
                    </h2>


                    <p>
                        Test how the Digital Twin
                        responds to different
                        emergency scenarios.
                    </p>


                    <button
                        className="simulation-button"
                        onClick={() =>
                            navigate("/simulation")
                        }
                    >
                        Start Simulation →
                    </button>


                    <button
                        className="evacuation-button"
                        onClick={() =>
                            navigate("/evacuation")
                        }
                    >
                        Open Evacuation Control →
                    </button>

                </div>

            </section>


            {/* FOOTER */}

            <footer className="dashboard-footer">

                <span>

                    <i className="footer-live-dot"></i>

                    Digital Twin synchronized

                </span>


                <span>

                    Last sync:

                    {" "}

                    <strong>
                        {formatTime(lastSync)}
                    </strong>

                </span>


                <span>
                    Auto-refresh: 3 seconds
                </span>

            </footer>

        </div>
    );
}


export default Dashboard;
```

### frontend/src/pages/Monitoring.jsx

Frontend source reference — not executable in this Python notebook.

#### Lines 1–220

```jsx
import { useEffect, useMemo, useState } from "react";

import {
    CartesianGrid,
    Line,
    LineChart,
    ResponsiveContainer,
    Tooltip,
    XAxis,
    YAxis,
} from "recharts";

import {
    getSensorHistory,
    getSensors,
} from "../services/api";

import {
    calculateRisk,
    getRiskBackground,
    getRiskColor,
} from "../utils/riskEngine";

import "./Monitoring.css";


function Monitoring() {

    const [sensors, setSensors] = useState([]);
    const [history, setHistory] = useState([]);

    const [selectedZone, setSelectedZone] = useState("");
    const [dropdownOpen, setDropdownOpen] = useState(false);

    const [error, setError] = useState("");
    const [loading, setLoading] = useState(true);


    /* =========================================================
       GET ZONE ID
    ========================================================= */

    function getZoneId(sensor) {

        if (!sensor) {
            return "";
        }

        if (
            sensor.zone_id !== undefined &&
            sensor.zone_id !== null
        ) {
            return String(sensor.zone_id);
        }

        if (sensor.device_id) {
            return String(sensor.device_id).replace(
                /^sensor-/,
                ""
            );
        }

        return "";
    }


    /* =========================================================
       LOAD LIVE SENSOR DATA
       Polls independently every 3 seconds
    ========================================================= */

    useEffect(() => {

        let mounted = true;

        const loadLiveSensors = async () => {

            try {

                const latestData = await getSensors();

                if (!mounted) {
                    return;
                }

                const latestList =
                    Array.isArray(latestData)
                        ? latestData
                        : [];

                setSensors(latestList);
                setError("");
                setLoading(false);

                /*
                 * Select the first available zone only
                 * when no zone is currently selected.
                 */
                setSelectedZone(previousZone => {

                    if (
                        previousZone ||
                        latestList.length === 0
                    ) {
                        return previousZone;
                    }

                    return getZoneId(
                        latestList[0]
                    );
                });

            } catch (err) {

                console.error(
                    "Monitoring sensor error:",
                    err
                );

                if (mounted) {

                    setError(
                        "Unable to connect to EVAC-TWIN backend."
                    );

                    setLoading(false);
                }
            }
        };


        loadLiveSensors();


        const interval = setInterval(
            loadLiveSensors,
            3000
        );


        return () => {

            mounted = false;

            clearInterval(interval);

        };

    }, []);


    /* =========================================================
       LOAD HISTORY FOR SELECTED ZONE
    ========================================================= */

    useEffect(() => {

        if (!selectedZone) {
            return;
        }

        let mounted = true;

        const loadHistory = async () => {

            try {

                const historyData =
                    await getSensorHistory(
                        selectedZone,
                        100
                    );

                if (!mounted) {
                    return;
                }

                setHistory(
                    Array.isArray(historyData)
                        ? historyData
                        : []
                );

            } catch (err) {

                console.error(
                    "Sensor history error:",
                    err
                );

            }
        };


        loadHistory();


        const interval = setInterval(
            loadHistory,
            3000
        );


        return () => {

            mounted = false;

            clearInterval(interval);

        };

    }, [selectedZone]);


    /* =========================================================
       SORT ZONES
    ========================================================= */

    const zoneSensors = useMemo(() => {
```

#### Lines 221–440

```jsx
        return [...sensors].sort(
            (a, b) =>
                Number(getZoneId(a)) -
                Number(getZoneId(b))
        );

    }, [sensors]);


    /* =========================================================
       SELECTED SENSOR
    ========================================================= */

    const latest = useMemo(() => {

        if (!zoneSensors.length) {
            return null;
        }

        const found =
            zoneSensors.find(
                sensor =>
                    getZoneId(sensor) ===
                    String(selectedZone)
            );

        return found || zoneSensors[0];

    }, [
        zoneSensors,
        selectedZone,
    ]);


    /* =========================================================
       CURRENT ZONE
    ========================================================= */

    const currentZone =
        latest
            ? getZoneId(latest)
            : selectedZone;


    /* =========================================================
       CENTRALIZED RISK ENGINE
    ========================================================= */

    const risk = useMemo(() => {

        return calculateRisk(latest);

    }, [latest]);


    /* =========================================================
       HISTORY FOR SELECTED ZONE
    ========================================================= */

    const selectedZoneHistory = useMemo(() => {

        return history
            .filter(
                sensor =>
                    getZoneId(sensor) ===
                    String(currentZone)
            )
            .sort(
                (a, b) =>
                    new Date(
                        a.timestamp || 0
                    ) -
                    new Date(
                        b.timestamp || 0
                    )
            );

    }, [
        history,
        currentZone,
    ]);


    /* =========================================================
       CHART DATA
    ========================================================= */

    const chartData = useMemo(() => {

        return selectedZoneHistory
            .slice(-20)
            .map(sensor => ({

                time:
                    sensor.timestamp
                        ? new Date(
                            sensor.timestamp
                        ).toLocaleTimeString()
                        : "--",

                temperature:
                    Number(
                        sensor.temperature || 0
                    ),

                humidity:
                    Number(
                        sensor.humidity || 0
                    ),

                smoke:
                    Number(
                        sensor.smoke || 0
                    ),

                occupancy:
                    Number(
                        sensor.occupancy || 0
                    ),

            }));

    }, [
        selectedZoneHistory,
    ]);


    /* =========================================================
       SENSOR STATUS
    ========================================================= */

    const temperatureStatus =
        !latest
            ? "WAITING"
            : Number(latest.temperature) >= 40
                ? "CRITICAL"
                : Number(latest.temperature) >= 38
                    ? "HIGH"
                    : Number(latest.temperature) >= 35
                        ? "MODERATE"
                        : "NORMAL";


    const smokeStatus =
        !latest
            ? "WAITING"
            : Number(latest.smoke) >= 0.8
                ? "CRITICAL"
                : Number(latest.smoke) >= 0.5
                    ? "HIGH"
                    : Number(latest.smoke) >= 0.3
                        ? "MODERATE"
                        : "NORMAL";


    const occupancyStatus =
        !latest
            ? "WAITING"
            : Number(latest.occupancy) >= 45
                ? "CRITICAL"
                : Number(latest.occupancy) >= 30
                    ? "HIGH"
                    : Number(latest.occupancy) >= 15
                        ? "MODERATE"
                        : "NORMAL";


    /* =========================================================
       LOADING
    ========================================================= */

    if (loading) {

        return (

            <div className="monitoring-page">

                <div className="monitoring-loading">

                    <div className="loading-spinner" />

                    <h2>
                        Connecting to EVAC-TWIN
                    </h2>

                    <p>
                        Waiting for live sensor data...
                    </p>

                </div>

            </div>

        );
    }


    /* =========================================================
       MAIN RENDER
    ========================================================= */

    return (

        <div className="monitoring-page">


            {/* =================================================
               HEADER
            ================================================= */}

            <header className="monitoring-header">

                <div>

                    <span className="monitoring-eyebrow">
                        COMMAND CENTER / LIVE MONITORING
                    </span>

                    <h1>
                        Live Monitoring
```

#### Lines 441–660

```jsx
                    </h1>

                    <p>
                        Real-time virtual IoT sensor monitoring
                    </p>

                </div>


                <div
                    className="monitoring-status"
                    style={{
                        color:
                            getRiskColor(
                                risk.level
                            ),

                        background:
                            getRiskBackground(
                                risk.level
                            ),
                    }}
                >

                    <span>●</span>

                    {risk.level}

                </div>

            </header>


            {/* =================================================
               ERROR
            ================================================= */}

            {error && (

                <div className="monitoring-error">

                    ⚠️ {error}

                </div>

            )}


            {latest && (

                <>


                    {/* =================================================
                       ZONE SELECTOR
                    ================================================= */}

                    <section className="zone-selector-card">

                        <div>

                            <span className="section-label">
                                BUILDING ALPHA
                            </span>

                            <h2>
                                Monitoring Zones
                            </h2>

                            <p className="zone-count">
                                {zoneSensors.length} / 12 zones online
                            </p>

                        </div>


                        <div className="zone-dropdown-wrapper">


                            <button
                                className="zone-dropdown-button"
                                onClick={() =>
                                    setDropdownOpen(
                                        previous =>
                                            !previous
                                    )
                                }
                            >

                                <span className="selected-zone-content">

                                    <strong>
                                        {currentZone}
                                    </strong>

                                    <span
                                        className="zone-risk-dot"
                                        style={{
                                            background:
                                                getRiskColor(
                                                    risk.level
                                                ),
                                        }}
                                    />

                                    <span
                                        style={{
                                            color:
                                                getRiskColor(
                                                    risk.level
                                                ),
                                        }}
                                    >
                                        {risk.level}
                                    </span>

                                </span>


                                <span
                                    className={`dropdown-arrow ${dropdownOpen
                                        ? "open"
                                        : ""
                                        }`}
                                >
                                    ▼
                                </span>

                            </button>


                            {dropdownOpen && (

                                <div className="zone-dropdown-menu">

                                    {zoneSensors.map(
                                        sensor => {

                                            const zone =
                                                getZoneId(
                                                    sensor
                                                );

                                            const zoneRisk =
                                                calculateRisk(
                                                    sensor
                                                );

                                            const selected =
                                                zone ===
                                                String(
                                                    currentZone
                                                );


                                            return (

                                                <button
                                                    key={zone}
                                                    className={`zone-option ${selected
                                                        ? "selected"
                                                        : ""
                                                        }`}
                                                    onClick={() => {

                                                        setSelectedZone(
                                                            zone
                                                        );

                                                        setDropdownOpen(
                                                            false
                                                        );

                                                    }}
                                                >

                                                    <span>
                                                        Zone {zone}
                                                    </span>


                                                    <span className="zone-option-risk">

                                                        <i
                                                            style={{
                                                                background:
                                                                    getRiskColor(
                                                                        zoneRisk.level
                                                                    ),
                                                            }}
                                                        />

                                                        <span
                                                            style={{
                                                                color:
                                                                    getRiskColor(
                                                                        zoneRisk.level
                                                                    ),
                                                            }}
                                                        >
                                                            {zoneRisk.level}
                                                        </span>

                                                    </span>

                                                </button>

                                            );

                                        }
                                    )}

                                </div>

                            )}

                        </div>

                    </section>
```

#### Lines 661–880

```jsx

                    {/* =================================================
                       CONNECTION
                    ================================================= */}

                    <div className="monitoring-info-bar">

                        <span>

                            <b className="online-dot">
                                ●
                            </b>

                            Sensor connection active

                        </span>


                        <span>
                            🆔 {latest.device_id || "--"}
                        </span>


                        <span>
                            📍 Zone {currentZone || "--"}
                        </span>


                        <span>

                            🕐{" "}

                            {latest.timestamp
                                ? new Date(
                                    latest.timestamp
                                ).toLocaleString()
                                : "--"
                            }

                        </span>

                    </div>


                    {/* =================================================
                       RISK
                    ================================================= */}

                    <section
                        className="zone-risk-summary"
                        style={{
                            borderColor:
                                getRiskColor(
                                    risk.level
                                ),
                        }}
                    >

                        <div>

                            <span className="section-label">
                                ZONE RISK
                            </span>

                            <h2
                                style={{
                                    color:
                                        getRiskColor(
                                            risk.level
                                        ),
                                }}
                            >
                                {risk.level}
                            </h2>

                            <p>

                                {risk.level === "LOW"

                                    ? "Environmental conditions are stable. Continue normal monitoring."

                                    : risk.level === "MODERATE"

                                        ? "Elevated environmental conditions detected. Continue close monitoring."

                                        : risk.level === "HIGH"

                                            ? "High-risk conditions detected. Prepare for evacuation."

                                            : risk.level === "CRITICAL"

                                                ? "Critical conditions detected. Immediate evacuation may be required."

                                                : "Waiting for sensor data."

                                }

                            </p>

                        </div>


                        <div className="risk-score-display">

                            <strong>
                                {risk.score}
                            </strong>

                            <span>
                                / 100
                            </span>

                            <small>
                                RISK SCORE
                            </small>

                        </div>

                    </section>


                    {/* =================================================
                       SENSOR CARDS
                    ================================================= */}

                    <section className="sensor-section">

                        <div className="section-heading">

                            <div>

                                <span className="section-label">
                                    ENVIRONMENT MONITOR
                                </span>

                                <h2>
                                    Current Conditions
                                </h2>

                            </div>

                            <span className="live-label">
                                ● LIVE
                            </span>

                        </div>


                        <div className="monitoring-sensor-grid">


                            <SensorCard
                                icon="🌡️"
                                label="Temperature"
                                value={`${latest.temperature ?? "--"} °C`}
                                status={
                                    temperatureStatus
                                }
                                color={
                                    getRiskColor(
                                        temperatureStatus ===
                                            "NORMAL"
                                            ? "LOW"
                                            : temperatureStatus
                                    )
                                }
                            />


                            <SensorCard
                                icon="💧"
                                label="Humidity"
                                value={`${latest.humidity ?? "--"} %`}
                                status="NORMAL"
                                color="#35c98a"
                            />


                            <SensorCard
                                icon="💨"
                                label="Smoke Level"
                                value={
                                    latest.smoke ?? "--"
                                }
                                status={
                                    smokeStatus
                                }
                                color={
                                    getRiskColor(
                                        smokeStatus ===
                                            "NORMAL"
                                            ? "LOW"
                                            : smokeStatus
                                    )
                                }
                            />


                            <SensorCard
                                icon="👥"
                                label="Occupancy"
                                value={
                                    latest.occupancy ?? "--"
                                }
                                status={
                                    occupancyStatus
                                }
                                color={
                                    getRiskColor(
                                        occupancyStatus ===
                                            "NORMAL"
                                            ? "LOW"
                                            : occupancyStatus
                                    )
                                }
                            />

                        </div>

                    </section>
```

#### Lines 881–1100

```jsx


                    {/* =================================================
                       TEMPERATURE CHART
                    ================================================= */}

                    <section className="chart-card">

                        <div className="chart-heading">

                            <h2>
                                🌡️ Temperature Trend
                            </h2>

                            <span>
                                Zone {currentZone}
                            </span>

                        </div>


                        <ResponsiveContainer
                            width="100%"
                            height={280}
                        >

                            <LineChart
                                data={chartData}
                            >

                                <CartesianGrid
                                    strokeDasharray="3 3"
                                    stroke="#26364d"
                                />

                                <XAxis
                                    dataKey="time"
                                    stroke="#64748b"
                                />

                                <YAxis
                                    stroke="#64748b"
                                />

                                <Tooltip
                                    contentStyle={{
                                        background:
                                            "#111827",

                                        border:
                                            "1px solid #334155",

                                        color:
                                            "#f8fafc",
                                    }}
                                />

                                <Line
                                    type="monotone"
                                    dataKey="temperature"
                                    stroke="#f97316"
                                    strokeWidth={3}
                                    dot={false}
                                    isAnimationActive={false}
                                />

                            </LineChart>

                        </ResponsiveContainer>

                    </section>


                    {/* =================================================
                       SMOKE + OCCUPANCY
                    ================================================= */}

                    <div className="monitoring-chart-grid">


                        <section className="chart-card">

                            <h2>
                                💨 Smoke Level
                            </h2>

                            <ResponsiveContainer
                                width="100%"
                                height={250}
                            >

                                <LineChart
                                    data={chartData}
                                >

                                    <CartesianGrid
                                        strokeDasharray="3 3"
                                        stroke="#26364d"
                                    />

                                    <XAxis
                                        dataKey="time"
                                        stroke="#64748b"
                                    />

                                    <YAxis
                                        domain={[0, 1]}
                                        stroke="#64748b"
                                    />

                                    <Tooltip
                                        contentStyle={{
                                            background:
                                                "#111827",

                                            border:
                                                "1px solid #334155",

                                            color:
                                                "#f8fafc",
                                        }}
                                    />

                                    <Line
                                        type="monotone"
                                        dataKey="smoke"
                                        stroke="#ef4444"
                                        strokeWidth={3}
                                        dot={false}
                                        isAnimationActive={false}
                                    />

                                </LineChart>

                            </ResponsiveContainer>

                        </section>


                        <section className="chart-card">

                            <h2>
                                👥 Occupancy
                            </h2>

                            <ResponsiveContainer
                                width="100%"
                                height={250}
                            >

                                <LineChart
                                    data={chartData}
                                >

                                    <CartesianGrid
                                        strokeDasharray="3 3"
                                        stroke="#26364d"
                                    />

                                    <XAxis
                                        dataKey="time"
                                        stroke="#64748b"
                                    />

                                    <YAxis
                                        stroke="#64748b"
                                    />

                                    <Tooltip
                                        contentStyle={{
                                            background:
                                                "#111827",

                                            border:
                                                "1px solid #334155",

                                            color:
                                                "#f8fafc",
                                        }}
                                    />

                                    <Line
                                        type="monotone"
                                        dataKey="occupancy"
                                        stroke="#38bdf8"
                                        strokeWidth={3}
                                        dot={false}
                                        isAnimationActive={false}
                                    />

                                </LineChart>

                            </ResponsiveContainer>

                        </section>

                    </div>


                    {/* =================================================
                       HISTORY
                    ================================================= */}

                    <section className="history-card">

                        <div className="section-heading">

                            <div>

                                <span className="section-label">
                                    SENSOR HISTORY
                                </span>

                                <h2>
                                    Recent Readings — Zone{" "}
                                    {currentZone}
                                </h2>

                            </div>
```

#### Lines 1101–1312

```jsx
                            <span className="live-label">
                                ● LIVE
                            </span>

                        </div>


                        <div className="history-table">

                            <div className="history-header">

                                <span>
                                    Time
                                </span>

                                <span>
                                    Temperature
                                </span>

                                <span>
                                    Humidity
                                </span>

                                <span>
                                    Smoke
                                </span>

                                <span>
                                    Occupancy
                                </span>

                            </div>


                            {selectedZoneHistory.length === 0 ? (

                                <div className="history-row">

                                    <span>
                                        No history data
                                    </span>

                                    <span>
                                        --
                                    </span>

                                    <span>
                                        --
                                    </span>

                                    <span>
                                        --
                                    </span>

                                    <span>
                                        --
                                    </span>

                                </div>

                            ) : (

                                selectedZoneHistory
                                    .slice()
                                    .reverse()
                                    .slice(0, 10)
                                    .map(
                                        (
                                            sensor,
                                            index
                                        ) => (

                                            <div
                                                className="history-row"
                                                key={`${sensor.timestamp}-${index}`}
                                            >

                                                <span>

                                                    {sensor.timestamp
                                                        ? new Date(
                                                            sensor.timestamp
                                                        ).toLocaleTimeString()
                                                        : "--"
                                                    }

                                                </span>


                                                <span>

                                                    {
                                                        sensor.temperature ??
                                                        "--"
                                                    } °C

                                                </span>


                                                <span>

                                                    {
                                                        sensor.humidity ??
                                                        "--"
                                                    } %

                                                </span>


                                                <span>

                                                    {
                                                        sensor.smoke ??
                                                        "--"
                                                    }

                                                </span>


                                                <span>

                                                    {
                                                        sensor.occupancy ??
                                                        "--"
                                                    }

                                                </span>

                                            </div>

                                        )
                                    )

                            )}

                        </div>

                    </section>


                    {/* =================================================
                       FOOTER
                    ================================================= */}

                    <footer className="monitoring-footer">

                        Automatically updating every 3 seconds
                        {" • "}
                        Monitoring sensor-{currentZone}

                    </footer>

                </>

            )}

        </div>

    );
}


/* =============================================================
   SENSOR CARD
============================================================= */

function SensorCard({
    icon,
    label,
    value,
    status,
    color,
}) {

    return (

        <div className="monitoring-sensor-card">

            <div className="sensor-card-top">

                <span className="sensor-icon">
                    {icon}
                </span>

                <span
                    className="sensor-status"
                    style={{
                        color,
                    }}
                >
                    ● {status}
                </span>

            </div>


            <span className="sensor-card-label">
                {label}
            </span>


            <strong className="sensor-card-value">
                {value}
            </strong>

        </div>

    );
}


export default Monitoring;
```

### frontend/src/pages/Evacuation.jsx

Frontend source reference — not executable in this Python notebook.

#### Lines 1–220

```jsx
import { useEffect, useState } from "react";
import {
    getEvacuationRoutes,
    getEvents,
    getSensors,
    getAlgorithmComparison,
} from "../services/api";
import { calculateRisk } from "../utils/riskEngine";
import "./Evacuation.css";

function Evacuation() {
    const urlZone = new URLSearchParams(window.location.search).get("zone");
    const [sensors, setSensors] = useState([]);
    const [selectedDevice, setSelectedDevice] = useState(
    () => `sensor-${urlZone || localStorage.getItem("selectedZone") || "101"}`
);
useEffect(() => {
    if (urlZone) {
        localStorage.setItem("selectedZone", urlZone);
        setSelectedDevice(`sensor-${urlZone}`);
    }
}, [urlZone]);
    const [events, setEvents] = useState([]);
    const [evacuationRoutes, setEvacuationRoutes] = useState(null);
    const [error, setError] = useState("");
    const [algorithmComparison, setAlgorithmComparison] = useState([]);
    useEffect(() => {
    const savedComparison = localStorage.getItem(
        "algorithmComparison"
    );

    if (savedComparison) {
        setAlgorithmComparison(
            JSON.parse(savedComparison)
        );
    }
}, []);

    useEffect(() => {
        const loadData = async () => {
            try {
                const [sensorData, eventData] =
    await Promise.all([
        getSensors(),
        getEvents(),
    ]);

                setSensors(sensorData || []);
                setEvents(eventData || []);
                setError("");
            } catch (err) {
                console.error(err);
                setError(
                    "Unable to connect to EVAC-TWIN backend."
                );
            }
        };

        loadData();

        const interval = setInterval(
            loadData,
            3000
        );

        return () =>
            clearInterval(interval);
    }, []);

    /* =====================================================
       LATEST SENSOR
    ===================================================== */



const liveLatest =
    sensors.find(
        (sensor) => sensor.device_id === selectedDevice
    ) || null;


// =====================================================
// LATEST LIVE SENSOR
// =====================================================

const latest = liveLatest;

useEffect(() => {
    if (!latest?.zone_id) {
        return;
    }

    // Mee existing useEffect code ikkada continue avvali

    const loadRoutes = async () => {
        try {
            const routeData = await getEvacuationRoutes(latest.zone_id);
            setEvacuationRoutes(routeData || null);
        } catch (err) {
            console.error("Evacuation routes failed:", err);
        }
    };

    loadRoutes();

}, [
    latest?.zone_id,
    latest?.temperature,
    latest?.smoke,
    latest?.occupancy
]);
    useEffect(() => {
    if (!latest?.device_id) {
        return;
    }

    const loadComparison = async () => {
        try {
            const data =
    await getAlgorithmComparison(
        latest.device_id,
        latest.temperature,
        latest.smoke,
        latest.occupancy
    );

           setAlgorithmComparison(
    Array.isArray(data) ? data : data?.comparison || []
);
        } catch (err) {
            console.error(
                "Algorithm comparison failed:",
                err
            );
        }
    };

    loadComparison();
}, [latest?.device_id, latest?.temperature, latest?.smoke, latest?.occupancy]);

    /* =====================================================
       ACTIVE EVENT
    ===================================================== */

   const activeEvent = events.find(
    (event) =>
        event.status === "ACTIVE" &&
        String(event.zone_id) === String(latest?.zone_id)
);

    /* =====================================================
       RISK
    ===================================================== */

    const calculatedRisk = latest
    ? calculateRisk(latest)
    : {
        level: "WAITING",
        score: 0,
        recommendation:
            "Waiting for live sensor information.",
        factors: {
            temperature: 0,
            smoke: 0,
            occupancy: 0,
        },
    };

const risk = latest
    ? {
        ...calculatedRisk,
        level:
            latest.dl_prediction ||
            latest.risk_level ||
            calculatedRisk.level,
        score:
            latest.risk_score ??
            calculatedRisk.score,
        recommendation:
            latest.recommendation ||
            calculatedRisk.recommendation,
    }
    : calculatedRisk;
console.log("Selected Zone:", latest?.zone_id);
console.log("Calculated Risk:", calculatedRisk);
console.log("Active Event:", activeEvent);

    const riskLevel = risk.level;

const riskScore = Number(risk.score ?? 0);

    /* =====================================================
       RISK THEME
    ===================================================== */

    const getRiskTheme = () => {
        switch (riskLevel) {
            case "CRITICAL":
                return {
                    color: "#ff5c5c",
                    soft: "#321b22",
                    border: "#71323c",
                    icon: "!",
                    title:
                        "EVACUATE IMMEDIATELY",
                };

            case "HIGH":
                return {
                    color: "#f59e0b",
                    soft: "#332714",
                    border: "#70531b",
                    icon: "!",
                    title:
                        "PREPARE FOR EVACUATION",
                };

            case "MODERATE":
                return {
                    color: "#38a0ca",
```

#### Lines 221–440

```jsx
                    soft: "#122d3d",
                    border: "#24536b",
                    icon: "•",
                    title:
                        "MONITOR CONDITIONS",
                };

            default:
                return {
                    color: "#35c98a",
                    soft: "#12352b",
                    border: "#24654c",
                    icon: "✓",
                    title: "SAFE",
                };
        }
    };

    const theme =
        getRiskTheme();

    /* =====================================================
       EVACUATION STATUS
    ===================================================== */
const getEvacuationStatus = () => {
    if (!latest)
        return "WAITING FOR DATA";

    if (riskLevel === "CRITICAL") {
        return "EVACUATE IMMEDIATELY";
    }

    if (riskLevel === "HIGH") {
        return "PREPARE FOR EVACUATION";
    }

    if (riskLevel === "MODERATE") {
        return "MONITOR CONDITIONS";
    }

    if (evacuationRoutes?.route_status) {
        return evacuationRoutes.route_status;
    }

    return "SAFE";
};

    const evacuationStatus =
        getEvacuationStatus();

    /* =====================================================
       SENSOR STATUS
    ===================================================== */

    const getTemperatureStatus =
        () => {
            if (!latest)
                return "WAITING";

            if (
                latest.temperature >=
                40
            )
                return "CRITICAL";

            if (
                latest.temperature >=
                38
            )
                return "ELEVATED";

            return "NORMAL";
        };

    const getSmokeStatus = () => {
        if (!latest)
            return "WAITING";

        if (
            latest.smoke >=
            0.9
        )
            return "CRITICAL";

        if (
            latest.smoke >=
            0.7
        )
            return "HIGH";

        if (
            latest.smoke >=
            0.5
        )
            return "MODERATE";

        return "NORMAL";
    };

    const getOccupancyStatus =
        () => {
            if (!latest)
                return "WAITING";

            if (
                latest.occupancy >=
                48
            )
                return "CRITICAL";

            if (
                latest.occupancy >=
                40
            )
                return "HIGH";

            if (
                latest.occupancy >=
                30
            )
                return "MODERATE";

            return "NORMAL";
        };

    const getSensorColor =
        (status) => {
            if (
                status ===
                "CRITICAL"
            )
                return "#ff5c5c";

            if (
                status === "HIGH" ||
                status === "ELEVATED"
            )
                return "#f59e0b";

            if (
                status ===
                "MODERATE"
            )
                return "#38a0ca";

            return "#35c98a";
        };

    /* =====================================================
       ACTION
    ===================================================== */

    const getAction = () => {
        if (
            riskLevel ===
            "CRITICAL"
        ) {
            return {
                title:
                    "Evacuate Immediately",
                text:
                    "Emergency conditions require immediate evacuation. Follow the dynamically recommended route and avoid blocked exits.",
            };
        }

        if (
            riskLevel ===
            "HIGH"
        ) {
            return {
                title:
                    "Prepare for Evacuation",
                text:
                    "Conditions are deteriorating. Prepare occupants and keep the safest emergency route ready.",
            };
        }

        if (
            riskLevel ===
            "MODERATE"
        ) {
            return {
                title:
                    "Continue Monitoring",
                text:
                    "Elevated conditions detected. Continue monitoring live sensor data for further changes.",
            };
        }

        return {
            title:
                "No Evacuation Required",
            text:
                "Environmental conditions are currently within the safe operating range.",
        };
    };

    const action =
        getAction();

    /* =====================================================
       ROUTE COLOR
    ===================================================== */

    const getRouteColor =
        (status) => {
            switch (status) {
                case "AVAILABLE":
                    return "#35c98a";

                case "CAUTION":
                    return "#f59e0b";

                case "BLOCKED":
                    return "#ff5c5c";

                default:
                    return "#64748b";
            }
        };
```

#### Lines 441–660

```jsx

  const simulationRoute = (() => {
    try {
        const data = localStorage.getItem("simulationRoute");
        return data ? JSON.parse(data) : null;
    } catch {
        return null;
    }
})();


const recommendedExit =
    evacuationRoutes?.selected_zone_recommendation ||
    evacuationRoutes?.recommended_exit;

const routeList =
    evacuationRoutes?.selected_zone_routes ||
    evacuationRoutes?.all_routes ||
    [];

const hasSafeRoute = routeList.some(
    (route) => route.status === "AVAILABLE"
);
    /* =====================================================
       RENDER
    ===================================================== */


return (
    <div className="evacuation-page">

        <div className="sensor-select">
            <label>Select Starting Zone: </label>

            <select
                value={selectedDevice}
                onChange={(e) => {
                    const deviceId = e.target.value;

                    setSelectedDevice(deviceId);

                    localStorage.setItem(
                        "selectedZone",
                        deviceId.replace(/^sensor-/, "")
                    );
                }}
            >
                {sensors.map((sensor) => (
                    <option
                        key={sensor.device_id}
                        value={sensor.device_id}
                    >
                        {sensor.zone_id}
                    </option>
                ))}
            </select>
        </div>

        {/* =================================================
           HEADER
        ================================================= */}




            <header className="evac-header">

                <div>
                    <div className="evac-breadcrumb">
                        COMMAND CENTER
                        <span>/</span>
                        EVACUATION
                    </div>

                    <h1>
                        Evacuation Management
                    </h1>

                    <p>
                        Real-time emergency
                        response and intelligent
                        route management
                    </p>
                </div>

                <div className="evac-header-status">

                    <div className="system-online">
                        <span>●</span>
                        SYSTEM ONLINE
                    </div>

                    <div className="live-badge">
                        LIVE
                    </div>

                </div>

            </header>

            {/* =================================================
               ERROR
            ================================================= */}

            {error && (
                <div className="error-box">

                    <div className="error-icon">
                        !
                    </div>

                    <div>
                        <strong>
                            Backend Connection Error
                        </strong>

                        <p>
                            {error}
                        </p>
                    </div>

                </div>
            )}

            {/* =================================================
               LOADING
            ================================================= */}

            {!latest &&
                !error && (
                    <div className="loading-state">

                        <div className="loading-spinner" />

                        <h3>
                            Synchronizing
                            Digital Twin
                        </h3>

                        <p>
                            Waiting for live
                            sensor data...
                        </p>

                    </div>
                )}

            {latest && (
                <>


                    {/* =================================================
                       COMMAND STATUS
                    ================================================= */}

                    <section
                        className="command-status"
                        style={{
                            "--status-color":
                                theme.color,

                            "--status-soft":
                                theme.soft,

                            "--status-border":
                                theme.border,
                        }}
                    >

                        <div className="command-main">

                            <div className="status-icon">
                                {theme.icon}
                            </div>

                            <div className="status-text">

                                <span className="eyebrow">
                                    CURRENT EVACUATION STATUS
                                </span>

                                <h2>
                                    {evacuationStatus}
                                </h2>

                                <p>
                                    {
                                        risk.recommendation
                                    }
                                </p>

                            </div>

                        </div>

                        <div className="risk-score-block">

                            <div className="risk-number">
                                {riskScore}
                            </div>

                            <div className="risk-score-info">

                                <strong>
                                    RISK SCORE
                                </strong>

                                <span>
                                    OUT OF 100
                                </span>

                            </div>

                        </div>

                        <div className="command-route">

                            <span className="eyebrow">
                                ROUTE DECISION
                            </span>
```

#### Lines 661–880

```jsx

                            <strong>
                                {recommendedExit
                                    ? recommendedExit.exit
                                    : "NO SAFE EXIT"}
                            </strong>

                            <span>
                                {recommendedExit
                                    ? recommendedExit.name
                                    : "No currently safe evacuation route"}
                            </span>

                        </div>

                    </section>

                    {/* =================================================
                       RISK METER
                    ================================================= */}

                    <section className="risk-meter-card">

                        <div className="risk-meter-header">

                            <div>

                                <span className="section-label">
                                    THREAT ASSESSMENT
                                </span>

                                <h2>
                                    Current Risk Level
                                </h2>

                            </div>

                            <span
                                className="risk-level-badge"
                                style={{
                                    color:
                                        theme.color,

                                    background:
                                        theme.soft,

                                    borderColor:
                                        theme.border,
                                }}
                            >
                                {riskLevel}
                            </span>

                        </div>

                        <div className="risk-meter">

                            <div
                                className="risk-meter-fill"
                                style={{
                                    width: `${Math.min(
                                        riskScore,
                                        100
                                    )}%`,

                                    background:
                                        theme.color,
                                }}
                            />

                        </div>

                        <div className="risk-scale">

                            <span>
                                SAFE
                            </span>

                            <span>
                                MODERATE
                            </span>

                            <span>
                                HIGH
                            </span>

                            <span>
                                CRITICAL
                            </span>

                        </div>

                    </section>

                    {/* =================================================
                       ACTIVE EVENT
                    ================================================= */}

                    {activeEvent && (
                        <section
                            className="active-event"
                            style={{
                                borderColor:
                                    theme.border,
                            }}
                        >

                            <div
                                className="event-icon"
                                style={{
                                    color:
                                        theme.color,

                                    background:
                                        theme.soft,
                                }}
                            >
                                !
                            </div>

                            <div className="event-content">

                                <span
                                    style={{
                                        color:
                                            theme.color,
                                    }}
                                >
                                    ACTIVE EMERGENCY EVENT
                                </span>

                                <strong>
                                    {
                                        activeEvent.level
                                    }{" "}
                                    RISK DETECTED
                                </strong>

                                <p>
                                    {
                                        activeEvent.recommendation ||
                                        "Emergency condition detected."
                                    }
                                </p>

                            </div>

                            <div className="event-score">

                                <span>
                                    SCORE
                                </span>

                                <strong>
                                    {
                                        activeEvent.score
                                    }
                                </strong>

                            </div>

                        </section>
                    )}

                    {/* =================================================
                       SENSOR MONITOR
                    ================================================= */}

                    <section className="section-block">

                        <div className="section-header">

                            <div>

                                <span className="section-label">
                                    ENVIRONMENT MONITOR
                                </span>

                                <h2>
                                    Live Risk Factors
                                </h2>

                            </div>

                            <div className="device-badge">

                                DEVICE

                                <strong>
                                    {
                                        latest.device_id
                                    }
                                </strong>

                            </div>

                        </div>

                        <div className="sensor-grid">


<SensorCard
    icon="°"
    label="TEMPERATURE"
    value={`${latest.temperature}°C`}
    status={getTemperatureStatus()}
    color={getSensorColor(
        getTemperatureStatus()
    )}
    description={`Indicator value: ${risk.factors.temperature}`}
/>


<SensorCard
    icon="◈"
    label="SMOKE LEVEL"
    value={latest.smoke}
    status={getSmokeStatus()}
    color={getSensorColor(
        getSmokeStatus()
```

#### Lines 881–1100

```jsx
    )}
    description={`Indicator value: ${risk.factors.smoke}`}
/>



<SensorCard
    icon="♙"
    label="OCCUPANCY"
    value={latest.occupancy}
    status={getOccupancyStatus()}
    color={getSensorColor(
        getOccupancyStatus()
    )}
    description={`Indicator value: ${risk.factors.occupancy}`}
/>


                            <SensorCard
                                icon="◌"
                                label="HUMIDITY"
                                value={`${latest.humidity}%`}
                                status="NORMAL"
                                color="#35c98a"
                                description="Environmental data"
                            />

                        </div>

                    </section>

                    {/* =================================================
                       RESPONSE ENGINE
                    ================================================= */}

                    <section className="response-card">

                        <div className="section-header">

                            <div>

                                <span className="section-label">
                                    RESPONSE ENGINE
                                </span>

                                <h2>
                                    Recommended Action
                                </h2>

                            </div>

                            <span
                                className="risk-level-badge"
                                style={{
                                    color:
                                        theme.color,

                                    background:
                                        theme.soft,

                                    borderColor:
                                        theme.border,
                                }}
                            >
                                {riskLevel}
                            </span>

                        </div>

                        <div
                            className="action-panel"
                            style={{
                                background:
                                    theme.soft,

                                borderColor:
                                    theme.border,
                            }}
                        >

                            <div
                                className="action-icon"
                                style={{
                                    color:
                                        theme.color,
                                }}
                            >
                                {theme.icon}
                            </div>

                            <div className="action-content">

                                <h3
                                    style={{
                                        color:
                                            theme.color,
                                    }}
                                >
                                    {action.title}
                                </h3>

                                <p>
                                    {action.text}
                                </p>

                            </div>

                            <div className="automated">

                                <span>
                                    DECISION
                                </span>

                                <strong>
                                    AUTOMATED
                                </strong>

                            </div>

                        </div>

                    </section>

                    {/* =================================================
                       ROUTES
                    ================================================= */}

                    <section className="route-card">

                        <div className="route-header">

                            <div>

                                <span className="section-label">
                                    DIGITAL TWIN ROUTING
                                </span>

                                <h2>
                                    Evacuation Routes
                                </h2>

                                <p>
                                    Routes dynamically
                                    evaluated from
                                    current environmental
                                    and occupancy
                                    conditions.
                                </p>

                            </div>

                            <div className="routes-live">

                                <span>
                                    ●
                                </span>

                                ROUTES LIVE

                            </div>

                        </div>


<div className="route-list">
    {routeList.map((exit, index) => (
        <Route
            key={exit.exit}
            number={String(index + 1).padStart(2, "0")}
            title={exit.name}
            code={exit.exit}
            description={exit.reason}
            status={exit.status}
            color={getRouteColor(exit.status)}
            riskScore={exit.risk_score}
            connectedZones={exit.connected_zones}
            path={exit.path}
            distance={exit.distance}
            isRecommended={recommendedExit?.exit === exit.exit}
        />
    ))}

    {!evacuationRoutes && (
        <div style={{ padding: "30px", textAlign: "center", color: "#64748b" }}>
            Calculating evacuation routes...
        </div>
    )}

    {evacuationRoutes && routeList.length === 0 && (
        <div style={{ padding: "30px", textAlign: "center", color: "#ff5c5c" }}>
            No evacuation routes returned by the backend.
        </div>
    )}
</div>

                        {/* =================================================
                           ROUTE STATS
                        ================================================= */}

                        <div className="route-stats">

                            <Stat
                                label="RISK LEVEL"
                                value={
                                    riskLevel
                                }
                                color={
                                    theme.color
                                }
                            />

                            <Stat
                                label="RISK SCORE"
                                value={`${riskScore}/100`}
                            />

                            <Stat
                                label="OCCUPANTS"
                                value={
                                    latest.occupancy
```

#### Lines 1101–1320

```jsx
                                }
                            />




<Stat
    label="ROUTE STATUS"
    value={
        riskLevel === "CRITICAL"
            ? routeList.some(
                route => route.status === "AVAILABLE"
            )
                ? "EVACUATION REQUIRED"
                : routeList.some(
                    route => route.status === "CAUTION"
                )
                    ? "CAUTION ROUTE — EVACUATE"
                    : "NO AVAILABLE ROUTE"
            : riskLevel === "HIGH"
                ? "PREPARE TO EVACUATE"
                : routeList.some(
                    route => route.status === "AVAILABLE"
                )
                    ? "SAFE ROUTE"
                    : routeList.some(
                        route => route.status === "CAUTION"
                    )
                        ? "CAUTION ROUTE"
                        : "NO AVAILABLE ROUTE"
    }
    color={
        riskLevel === "CRITICAL"
            ? "#ff5c5c"
            : riskLevel === "HIGH"
                ? "#f59e0b"
                : routeList.some(
                    route => route.status === "AVAILABLE"
                )
                    ? "#35c98a"
                    : routeList.some(
                        route => route.status === "CAUTION"
                    )
                        ? "#f59e0b"
                        : "#ff5c5c"
    }
/>






                        </div>

                    </section>


{/* =================================================
   ALGORITHM COMPARISON
================================================= */}

<section className="system-card">

    <div className="system-header">
        <div>
            <span className="section-label">
                ROUTING ALGORITHM ANALYSIS
            </span>

            <h2>Algorithm Comparison</h2>

            <small>
                Comparing route search methods for the selected zone.
            </small>
        </div>

        <span className="connected">
            ● LIVE
        </span>
    </div>

    <div className="algorithm-grid">

        {algorithmComparison
            .filter((item) => {
                const recommended =
                    evacuationRoutes?.selected_zone_recommendation?.exit ||
                    evacuationRoutes?.recommended_exit?.exit;

                if (!recommended) return true;

                const normalize = (value) =>
                    String(value)
                        .toLowerCase()
                        .replace(/[^a-z0-9]/g, "");

                return (
                    normalize(item.exit).includes(normalize(recommended)) ||
                    normalize(recommended).includes(normalize(item.exit))
                );
            })
            .map((item) => (

                <div className="algorithm-card" key={item.exit}>

                    <strong>{item.exit}</strong>

                    <div>
                        <span>BFS</span>

                        <small>
                            Finds a path with the minimum number of edges.
                        </small>

                        <small>
                            {item.bfs?.path?.join(" → ") || "No path available"}
                        </small>

                        <small>
                            Edges: {item.bfs?.edges ?? "N/A"}
                        </small>


                    </div>

                    <div>
                        <span>DFS</span>

                        <small>
                            Explores the graph depth-first; it does not guarantee
                            the shortest path.
                        </small>

                        <small>
                            {item.dfs?.path?.join(" → ") || "No path available"}
                        </small>

                        <small>
                            Edges: {item.dfs?.edges ?? "N/A"}
                        </small>


                    </div>

                    <div className="recommended-algorithm">

                        <span>
                            Dijkstra <b>PRIMARY</b>
                        </span>

                        <small>
                            Finds the minimum accumulated cost using
                            non-negative risk-aware edge weights.
                        </small>

                        <small>
                            {item.dijkstra?.path?.join(" → ") || "No path available"}
                        </small>

                        <small>
                            Cost: {item.dijkstra?.cost ?? "N/A"}
                            {" | "}
                            Edges: {item.dijkstra?.edges ?? "N/A"}
                        </small>



                        <small className="why-dijkstra">
                            Primary algorithm because EVAC-TWIN uses
                            risk-aware weighted route costs.
                        </small>

                    </div>

                    <div>
                        <span>A*</span>

                        <small>
                            Uses weighted route cost and a destination-guided
                            heuristic to search for a route.
                        </small>

                        <small>
                            {item.a_star?.path?.join(" → ") || "No path available"}
                        </small>

                        <small>
                            Cost: {item.a_star?.cost ?? "N/A"}
                            {" | "}
                            Edges: {item.a_star?.edges ?? "N/A"}
                        </small>



                    </div>

                </div>
            ))}

    </div>

</section>


                    {/* =================================================
                       SYSTEM STATUS
                    ================================================= */}

                    <section className="system-card">

                        <div className="system-header">

                            <div>

                                <span className="section-label">
                                    SYSTEM STATUS
                                </span>

                                <h2>
```

#### Lines 1321–1540

```jsx
                                    Digital Twin Connection
                                </h2>

                            </div>

                            <span className="connected">
                                ● CONNECTED
                            </span>

                        </div>

                        <div className="system-grid">

                            <Info
                                label="DEVICE"
                                value={
                                    latest.device_id
                                }
                            />

                            <Info
                                label="LAST UPDATED"
                                value={
                                    latest.timestamp
                                        ? new Date(
                                            latest.timestamp
                                        ).toLocaleTimeString()
                                        : "—"
                                }
                            />

                            <Info
                                label="UPDATE FREQUENCY"
                                value="3 seconds"
                            />

                            <Info
                                label="BACKEND"
                                value="ONLINE"
                                green
                            />

                        </div>

                    </section>

                    {/* =================================================
                       FOOTER
                    ================================================= */}

                    <footer className="evac-footer">

                        <span>
                            <b>●</b>{" "}
                            EVAC-TWIN DIGITAL
                            TWIN SYNCHRONIZED
                        </span>

                        <span>
                            Real-time monitoring
                            · 3 second refresh
                        </span>

                    </footer>

                </>
            )}

        </div>
    );
}


/* =========================================================
   SENSOR CARD
========================================================= */

function SensorCard({
    icon,
    label,
    value,
    status,
    color,
    description,
}) {
    return (
        <div
            className="sensor-card"
            style={{
                "--sensor-color":
                    color,
            }}
        >

            <div className="sensor-top">

                <div
                    className="sensor-icon"
                    style={{
                        color,
                        background:
                            `${color}12`,
                    }}
                >
                    {icon}
                </div>

                <span
                    className="sensor-status"
                    style={{
                        color,
                    }}
                >
                    ● {status}
                </span>

            </div>

            <span className="sensor-label">
                {label}
            </span>

            <strong className="sensor-value">
                {value}
            </strong>

            <div className="sensor-bottom">

                <span>
                    {description}
                </span>

                <span
                    style={{
                        color,
                    }}
                >
                    ●
                </span>

            </div>

        </div>
    );
}


/* =========================================================
   ROUTE
========================================================= */

function Route({
    number,
    title,
    code,
    description,
    status,
    color,
    riskScore,
    connectedZones,
    path,
    distance,
    isRecommended,
}) {
    return (
        <div
            className="route"
            style={{
                "--route-color":
                    color,
            }}
        >

            {/* NUMBER */}

            <div className="route-number">
                {number}
            </div>

            {/* MARKER */}

            <div className="route-marker">

                <div
                    className="marker-dot"
                    style={{
                        background:
                            color,

                        boxShadow:
                            `0 0 0 5px ${color}20`,
                    }}
                />

            </div>

            {/* CONTENT */}

            <div className="route-content">

                <div className="route-title-row">

                    <strong>
                        {title}
                    </strong>

                    <span className="route-code">
                        {code}
                    </span>

                    {isRecommended && (
                        <span
                            style={{
                                padding:
                                    "3px 7px",
                                borderRadius:
                                    "999px",
                                background:
                                    "#12352b",
                                border:
```

#### Lines 1541–1760

```jsx
                                    "1px solid #24654c",
                                color:
                                    "#35c98a",
                                fontSize:
                                    "7px",
                                fontWeight:
                                    "900",
                                letterSpacing:
                                    "0.6px",
                            }}
                        >
                            RECOMMENDED
                        </span>
                    )}

                </div>

                <p>
                    {description}
                </p>

                {/* =================================================
                   PHYSICAL PATH
                ================================================= */}

                {path?.length > 0 && (
                    <div
                        className="route-path"
                        title={path.join(
                            " → "
                        )}
                    >

                        <span className="route-path-label">
                            PATH
                        </span>

                        <div className="route-path-nodes">

                            {path.map(
                                (
                                    node,
                                    index
                                ) => (

                                    <div
                                        className="route-path-node-wrapper"
                                        key={`${node}-${index}`}
                                    >

                                        <span
                                            className={`route-path-node ${node.startsWith(
                                                "EXIT_"
                                            )
                                                ? "exit-node"
                                                : node.startsWith(
                                                    "STAIR_"
                                                )
                                                    ? "stair-node"
                                                    : "room-node"
                                                }`}
                                        >
                                            {formatNodeName(
                                                node
                                            )}
                                        </span>

                                        {index <
                                            path.length -
                                            1 && (
                                                <span className="route-path-arrow">
                                                    →
                                                </span>
                                            )}

                                    </div>

                                )
                            )}

                        </div>

                    </div>
                )}

                {distance !==
                    null &&
                    distance !==
                    undefined && (
                        <div className="route-distance">
    ROUTE EDGES{" "}
    <strong>{distance}</strong>
</div>
                    )}

                {connectedZones?.length >
                    0 && (
                        <div className="connected-zones">
                            Zones:{" "}
                            {connectedZones.join(
                                ", "
                            )}
                        </div>
                    )}

            </div>

            {/* RIGHT */}

            <div className="route-right">

                <span
                    className="route-status"
                    style={{
                        color,
                        background:
                            `${color}12`,
                    }}
                >
                    {status}
                </span>

                <span className="route-risk">
                    Risk {riskScore}
                </span>

            </div>

        </div>
    );
}


/* =========================================================
   FORMAT TOPOLOGY NODE
========================================================= */

function formatNodeName(
    node
) {
    if (
        node === "EXIT_A"
    )
        return "EXIT A";

    if (
        node === "EXIT_B"
    )
        return "EXIT B";

    if (
        node === "EXIT_C"
    )
        return "EXIT C";

    if (
        node === "STAIR_A"
    )
        return "STAIR A";

    if (
        node === "STAIR_B"
    )
        return "STAIR B";

    return node;
}


/* =========================================================
   STAT
========================================================= */

function Stat({
    label,
    value,
    color,
}) {
    return (
        <div className="stat">

            <span>
                {label}
            </span>

            <strong
                style={{
                    color:
                        color ||
                        "#f8fafc",
                }}
            >
                {value}
            </strong>

        </div>
    );
}


/* =========================================================
   INFO
========================================================= */

function Info({
    label,
    value,
    green,
}) {
    return (
        <div className="info">

            <span>
                {label}
            </span>

            <strong
                style={{
                    color: green
                        ? "#35c98a"
```

#### Lines 1761–1776

```jsx
                        : "#d8e0ea",
                }}
            >
                {green &&
                    "● "}

                {value}

            </strong>

        </div>
    );
}


export default Evacuation;
```

### frontend/src/pages/EvacuationMap.jsx

Frontend source reference — not executable in this Python notebook.

#### Lines 1–220

```jsx

import React from "react";

const floors = [3, 2, 1];

const roomNumbers = {
    1: ["101", "102", "103", "104"],
    2: ["201", "202", "203", "204"],
    3: ["301", "302", "303", "304"]
};

const corridorNames = [
    "LEFT",
    "CENTER",
    "RIGHT"
];

const stairs = ["A", "B", "C"];

const positions = {};

floors.forEach((floor, index) => {
    const y = 55 + index * 190;

    roomNumbers[floor].forEach((room, i) => {
        positions[room] = {
            x: 65 + i * 125,
            y: y
        };
    });

    corridorNames.forEach((name, i) => {
        positions[`CORRIDOR_${floor}_${name}`] = {
            x: 127 + i * 125,
            y: y + 65
        };
    });

    stairs.forEach((stair, i) => {
        positions[`STAIR_${stair}_${floor}`] = {
            x: 127 + i * 125,
            y: y + 125
        };
    });
});

positions.EXIT_A_NODE = { x: 127, y: 625 };
positions.EXIT_B_NODE = { x: 252, y: 625 };
positions.EXIT_C_NODE = { x: 377, y: 625 };

const edges = [];

floors.forEach(floor => {
    const rooms = roomNumbers[floor];

    edges.push(
        [rooms[0], `CORRIDOR_${floor}_LEFT`],
        [rooms[1], `CORRIDOR_${floor}_LEFT`],
        [rooms[1], `CORRIDOR_${floor}_CENTER`],
        [rooms[2], `CORRIDOR_${floor}_CENTER`],
        [rooms[2], `CORRIDOR_${floor}_RIGHT`],
        [rooms[3], `CORRIDOR_${floor}_RIGHT`],

        [`CORRIDOR_${floor}_LEFT`, `CORRIDOR_${floor}_CENTER`],
        [`CORRIDOR_${floor}_CENTER`, `CORRIDOR_${floor}_RIGHT`],

        [`CORRIDOR_${floor}_LEFT`, `STAIR_A_${floor}`],
        [`CORRIDOR_${floor}_CENTER`, `STAIR_B_${floor}`],
        [`CORRIDOR_${floor}_RIGHT`, `STAIR_C_${floor}`]
    );
});

stairs.forEach(stair => {
    edges.push(
        [`STAIR_${stair}_1`, `STAIR_${stair}_2`],
        [`STAIR_${stair}_2`, `STAIR_${stair}_3`]
    );
});

edges.push(
    ["STAIR_A_1", "EXIT_A_NODE"],
    ["STAIR_B_1", "EXIT_B_NODE"],
    ["STAIR_C_1", "EXIT_C_NODE"]
);

export default function EvacuationMap({
    zone,
    path = [],
    exit
}) {
    const selectedZone = String(zone || "");

    const activePath = path.map(node => String(node));

    const pathEdges = new Set();

    for (let i = 0; i < activePath.length - 1; i++) {
        pathEdges.add(
            `${activePath[i]}|${activePath[i + 1]}`
        );

        pathEdges.add(
            `${activePath[i + 1]}|${activePath[i]}`
        );
    }

    const isPathEdge = (a, b) =>
        pathEdges.has(`${a}|${b}`);

    const isPathNode = node =>
        activePath.includes(node);

    const nodeColor = node => {
        if (node === selectedZone) return "#ef4444";
        if (isPathNode(node)) return "#f59e0b";
        return "#15803d";
    };

    const renderNode = (node, label, width = 75) => {
        const point = positions[node];

        if (!point) return null;

        const isSelected = node === selectedZone;

        return (
            <g key={node}>
                <rect
                    x={point.x - width / 2}
                    y={point.y - 17}
                    width={width}
                    height="34"
                    rx="6"
                    fill={nodeColor(node)}
                    stroke={isSelected ? "#ffffff" : "#94a3b8"}
                    strokeWidth={isSelected ? "3" : "1"}
                />

                <text
                    x={point.x}
                    y={point.y + 5}
                    fill="white"
                    textAnchor="middle"
                    fontSize="12"
                    fontWeight="bold"
                >
                    {label}
                </text>
            </g>
        );
    };

    const renderEdge = ([a, b], index) => {
        const start = positions[a];
        const end = positions[b];

        if (!start || !end) return null;

        const highlighted = isPathEdge(a, b);

        return (
            <line
                key={`${a}-${b}-${index}`}
                x1={start.x}
                y1={start.y}
                x2={end.x}
                y2={end.y}
                stroke={highlighted ? "#f59e0b" : "#64748b"}
                strokeWidth={highlighted ? "5" : "2"}
                strokeLinecap="round"
            />
        );
    };

    return (
        <section style={{
            background: "#172334",
            padding: "18px",
            borderRadius: "12px",
            marginBottom: "20px",
            color: "white",
            overflow: "hidden"
        }}>
            <h2>Building Alpha - Evacuation Map</h2>

            <p>
                Selected Emergency Zone: <b>{selectedZone || "None"}</b>
            </p>

            <div style={{
                display: "flex",
                gap: "14px",
                flexWrap: "wrap",
                marginBottom: "15px",
                fontSize: "12px"
            }}>
                <span>🔴 Selected Zone</span>
                <span>🟠 Recommended Route</span>
                <span>🟢 Other Rooms</span>
                <span>⚪ Normal Connections</span>
            </div>

            <div style={{ overflowX: "auto" }}>
                <svg
                    viewBox="0 0 500 680"
                    width="100%"
                    style={{ minWidth: "340px" }}
                >
                    {floors.map((floor, index) => (
                        <g key={floor}>
                            <rect
                                x="5"
                                y={30 + index * 190}
                                width="490"
                                height="175"
                                rx="10"
                                fill="#26374b"
                                stroke="#475569"
                            />
```

#### Lines 221–288

```jsx
                            <text
                                x="18"
                                y={48 + index * 190}
                                fill="white"
                                fontSize="13"
                                fontWeight="bold"
                            >
                                FLOOR {floor}
                            </text>
                        </g>
                    ))}

                    {edges.map(renderEdge)}

                    {floors.map(floor => (
                        <g key={`rooms-${floor}`}>
                            {roomNumbers[floor].map(room =>
                                renderNode(room, room, 75)
                            )}

                            {corridorNames.map(name => {
                                const node = `CORRIDOR_${floor}_${name}`;

                                return renderNode(
                                    node,
                                    name,
                                    80
                                );
                            })}

                            {stairs.map(stair => {
                                const node = `STAIR_${stair}_${floor}`;

                                return renderNode(
                                    node,
                                    `STAIR ${stair}`,
                                    85
                                );
                            })}
                        </g>
                    ))}

                    {["A", "B", "C"].map(stair =>
                        renderNode(
                            `EXIT_${stair}_NODE`,
                            `EXIT ${stair}`,
                            85
                        )
                    )}
                </svg>
            </div>

            <p>
                <b>Recommended Exit:</b> {exit || "Not available"}
            </p>

            <p style={{
                overflowWrap: "anywhere",
                fontSize: "13px"
            }}>
                <b>Dijkstra Route:</b>{" "}
                {activePath.length
                    ? activePath.join(" → ")
                    : "No route available"}
            </p>
        </section>
    );
}
```

### frontend/src/pages/Simulation.jsx

Frontend source reference — not executable in this Python notebook.

#### Lines 1–220

```jsx
import { useEffect, useMemo, useState } from "react";

import {
    calculateRisk,
    getRiskBackground,
    getRiskColor,
} from "../utils/riskEngine";

import {
    getSensors,
    getZoneEvacuationRoute,
    getAlgorithmComparison,
    sendSimulationAlert,
    predictRiskDL,
} from "../services/api";

import "./Simulation.css";

function Simulation() {
    const [scenario, setScenario] =
        useState("Fire Emergency");

    const [temperature, setTemperature] =
        useState(38);

    const [smoke, setSmoke] =
        useState(0.7);

    const [algorithmComparison, setAlgorithmComparison] =
        useState([]);

    const [occupancy, setOccupancy] =
        useState(30);

    const [result, setResult] =
        useState(null);

    const [loadingSensors, setLoadingSensors] =
        useState(true);

    const [simulationLoading, setSimulationLoading] =
        useState(false);

    const [sensorError, setSensorError] =
        useState("");

    const [liveSensor, setLiveSensor] =
        useState(null);

    const [sensors, setSensors] =
        useState([]);

    const [selectedZone, setSelectedZone] =
        useState("");

    const [route, setRoute] =
        useState(null);

    // =========================================================
    // LOAD LIVE SENSOR DATA
    // =========================================================

    useEffect(() => {
        const loadSensors = async () => {
            try {
                setSensorError("");

                const sensorData = await getSensors();

                if (
                    Array.isArray(sensorData) &&
                    sensorData.length > 0
                ) {
                    setSensors(sensorData);

                    setSelectedZone((currentZone) =>
                        currentZone ||
                        String(sensorData[0].zone_id)
                    );
                }
            } catch (error) {
                console.error(
                    "Failed to load live sensors:",
                    error
                );

                setSensorError(
                    "Unable to connect to live sensor data."
                );
            } finally {
                setLoadingSensors(false);
            }
        };

        loadSensors();

        const interval = setInterval(
            loadSensors,
            5000
        );

        return () => {
            clearInterval(interval);
        };
    }, []);

    // =========================================================
    // SELECTED LIVE SENSOR
    // =========================================================

    useEffect(() => {
        if (!selectedZone || sensors.length === 0) {
            return;
        }

        const sensor = sensors.find(
            (item) =>
                String(item.zone_id) ===
                String(selectedZone)
        );

        if (sensor) {
            setLiveSensor(sensor);
        }
    }, [selectedZone, sensors]);

    // =========================================================
    // SCENARIO CONFIGURATION
    // =========================================================

    const scenarioConfig = {
        "Fire Emergency": {
            description:
                "Simulate a developing fire inside the building.",
            temperatureMultiplier: 1.2,
            smokeMultiplier: 1.4,
            occupancyMultiplier: 0.8,
        },

        "High Temperature": {
            description:
                "Evaluate the building under extreme heat conditions.",
            temperatureMultiplier: 1.6,
            smokeMultiplier: 0.5,
            occupancyMultiplier: 0.6,
        },

        "Smoke Emergency": {
            description:
                "Simulate increasing smoke concentration.",
            temperatureMultiplier: 0.7,
            smokeMultiplier: 1.7,
            occupancyMultiplier: 0.7,
        },

        "Overcrowding": {
            description:
                "Evaluate evacuation pressure from high occupancy.",
            temperatureMultiplier: 0.5,
            smokeMultiplier: 0.6,
            occupancyMultiplier: 1.8,
        },

        "Combined Emergency": {
            description:
                "Simulate simultaneous environmental hazards.",
            temperatureMultiplier: 1.3,
            smokeMultiplier: 1.3,
            occupancyMultiplier: 1.3,
        },
    };

    // =========================================================
    // LIVE PREVIEW - FRONTEND RISK CALCULATION
    // =========================================================

    const calculateScenarioRisk = () => {
        const sensor = {
            temperature: Number(temperature),
            smoke: Number(smoke),
            occupancy: Number(occupancy),
            humidity: 50,
            device_id: "simulation",
            timestamp: new Date().toISOString(),
        };

        const baseRisk = calculateRisk(sensor);

        const config = scenarioConfig[scenario];

        if (!config) {
            return baseRisk;
        }

        const adjustedTemperature =
            baseRisk.factors.temperature *
            config.temperatureMultiplier;

        const adjustedSmoke =
            baseRisk.factors.smoke *
            config.smokeMultiplier;

        const adjustedOccupancy =
            baseRisk.factors.occupancy *
            config.occupancyMultiplier;

        const score = Math.min(
            Math.round(
                adjustedTemperature +
                adjustedSmoke +
                adjustedOccupancy
            ),
            100
        );

        let level = "LOW";

        let recommendation =
            "Environmental conditions are stable. Continue normal monitoring.";
```

#### Lines 221–440

```jsx
        if (score >= 70) {
            level = "CRITICAL";

            recommendation =
                "Immediate evacuation recommended. Activate emergency procedures.";
        } else if (score >= 40) {
            level = "HIGH";

            recommendation =
                "High-risk conditions detected. Prepare evacuation routes and alert occupants.";
        } else if (score >= 20) {
            level = "MODERATE";

            recommendation =
                "Elevated conditions detected. Continue monitoring sensor values.";
        }

        return {
            score,
            level,
            recommendation,

            factors: {
                temperature: Math.round(adjustedTemperature),
                smoke: Math.round(adjustedSmoke),
                occupancy: Math.round(adjustedOccupancy),
            },
        };
    };

    // =========================================================
    // LIVE RISK PREVIEW
    // =========================================================

    const liveRisk = useMemo(
        () => calculateScenarioRisk(),
        [
            scenario,
            temperature,
            smoke,
            occupancy,
        ]
    );

    const activeRisk = result || liveRisk;

    const riskColor = getRiskColor(
        activeRisk.level
    );

    const riskBackground = getRiskBackground(
        activeRisk.level
    );

    // =========================================================
    // RUN SIMULATION - PYTORCH MLP
    // =========================================================

    const runSimulation = async () => {
        try {
            if (!selectedZone) {
                alert("Please select a zone.");
                return;
            }

            const selectedSensor = sensors.find(
                (sensor) =>
                    String(sensor.zone_id) ===
                    String(selectedZone)
            );

            if (!selectedSensor) {
                alert("No sensor found for the selected zone.");
                return;
            }

            setSimulationLoading(true);

            const simulationInput = {
                temperature: Number(temperature),
                humidity: Number(
                    liveSensor?.humidity ?? 50
                ),
                smoke: Number(smoke),
                occupancy: Number(occupancy),
            };

            console.log(
                "SIMULATION INPUT:",
                {
                    zone: selectedZone,
                    scenario,
                    ...simulationInput,
                }
            );

            // Get prediction from backend PyTorch MLP
            const dlResult = await predictRiskDL(
                simulationInput.temperature,
                simulationInput.humidity,
                simulationInput.smoke,
                simulationInput.occupancy
            );

            console.log(
                "DEEP LEARNING PREDICTION:",
                dlResult
            );

            const predictionLevel = String(
                dlResult.prediction || "LOW"
            ).toUpperCase();

            const simulationResult = {
                score: Math.max(
                    0,
                    Math.min(
                        100,
                        Number(dlResult.risk_score ?? 0)
                    )
                ),

                level: predictionLevel,

                confidence: Number(
                    dlResult.confidence ?? 0
                ),

                probabilities:
                    dlResult.probabilities || {},

                recommendation:
                    dlResult.recommendation ||
                    "Continue monitoring the building conditions.",

                factors: {
                    temperature:
                        simulationInput.temperature,

                    smoke:
                        simulationInput.smoke,

                    occupancy:
                        simulationInput.occupancy,
                },
            };

            // Algorithm comparison
            const comparisonData =
                await getAlgorithmComparison(
                    selectedSensor.device_id,
                    simulationInput.temperature,
                    simulationInput.smoke,
                    simulationInput.occupancy
                );

            setAlgorithmComparison(
                comparisonData?.comparison || []
            );

            // Dynamic evacuation route
            const routeResult =
                await getZoneEvacuationRoute(
                    selectedSensor.device_id,
                    simulationInput.temperature,
                    simulationInput.smoke,
                    simulationInput.occupancy
                );

            setRoute(routeResult);

            // Emergency SMS
            let smsResult = {
                sms_sent: false,
                message: "No emergency SMS required.",
            };

            if (
                predictionLevel === "HIGH" ||
                predictionLevel === "CRITICAL"
            ) {
                smsResult = await sendSimulationAlert(
                    selectedSensor.device_id,
                    simulationInput.temperature,
                    simulationInput.smoke,
                    simulationInput.occupancy,
                    scenario
                );
            }

            console.log(
                "SIMULATION ROUTE:",
                routeResult
            );

            console.log(
                "SIMULATION SMS RESULT:",
                smsResult
            );

            if (smsResult.sms_sent) {
                alert("Emergency SMS sent successfully!");
            } else {
                alert(smsResult.message);
            }

            // Save simulation data
            localStorage.setItem(
                "selectedZone",
                selectedZone
            );

            localStorage.setItem(
                "algorithmComparison",
                JSON.stringify(
                    comparisonData?.comparison || []
                )
            );

            localStorage.setItem(
```

#### Lines 441–660

```jsx
                "simulationInput",
                JSON.stringify({
                    zone_id: selectedZone,
                    scenario,
                    ...simulationInput,
                    risk_score: simulationResult.score,
                    risk_level: simulationResult.level,
                    dl_confidence: simulationResult.confidence,
                })
            );

            localStorage.setItem(
                "simulationRoute",
                JSON.stringify(routeResult)
            );

            setResult(simulationResult);

        } catch (error) {
            console.error(
                "Simulation failed:",
                error
            );

            alert(
                `Simulation failed: ${error.message}`
            );
        } finally {
            setSimulationLoading(false);
        }
    };

    // =========================================================
    // RESET
    // =========================================================

    const resetSimulation = () => {
        setScenario("Fire Emergency");

        if (liveSensor) {
            setTemperature(
                Number(liveSensor.temperature ?? 38)
            );

            setSmoke(
                Number(liveSensor.smoke ?? 0.7)
            );

            setOccupancy(
                Number(liveSensor.occupancy ?? 30)
            );
        } else {
            setTemperature(38);
            setSmoke(0.7);
            setOccupancy(30);
        }

        setResult(null);
        setRoute(null);
        setAlgorithmComparison([]);
    };

    // =========================================================
    // SCENARIO DESCRIPTION
    // =========================================================

    const getScenarioDescription = () => {
        return (
            scenarioConfig[scenario]?.description ||
            "Configure emergency conditions."
        );
    };

    // =========================================================
    // FLOOR STATUS
    // =========================================================

    const getFloorClass = (floor) => {
    return "simulation-floor";
};
    // =========================================================
    // ROOM STATUS
    // =========================================================

    const getRoomClass = (floor, room) => {
    const zoneId = `${floor}0${room}`;

    const roomSensor = sensors.find(
        (sensor) => String(sensor.zone_id) === zoneId
    );

    if (!roomSensor) return "normal-room";

    const risk =
        String(selectedZone) === zoneId
            ? activeRisk
            : calculateRisk(roomSensor);

    if (risk.level === "CRITICAL") return "critical-room";

    if (risk.level === "HIGH" || risk.level === "MODERATE") {
        return "warning-room";
    }

    return "normal-room";
};
    // =========================================================
    // SCENARIO IMPACT
    // =========================================================

    const scenarioImpact = {
        "Fire Emergency":
            "Fire conditions are primarily influenced by heat and smoke.",

        "High Temperature":
            "Extreme temperature is the dominant risk factor.",

        "Smoke Emergency":
            "Smoke concentration is the dominant emergency factor.",

        "Overcrowding":
            "High occupancy increases evacuation pressure across the building.",

        "Combined Emergency":
            "Multiple environmental hazards are contributing simultaneously.",
    };

    // =========================================================
    // RENDER
    // =========================================================

    return (
        <div
            className="simulation-page"
            style={{
                "--simulation-risk-color": riskColor,
                "--simulation-risk-background": riskBackground,
            }}
        >
            {/* HEADER */}

            <header className="simulation-header">
                <div>
                    <p className="simulation-eyebrow">
                        DIGITAL TWIN / WHAT-IF ENGINE
                    </p>

                    <h1>
                        Emergency Simulation
                    </h1>

                    <p className="simulation-subtitle">
                        Test emergency conditions and evaluate how
                        Building Alpha responds.
                    </p>
                </div>

                <div className="simulation-mode">
                    <span className="simulation-mode-dot"></span>
                    SIMULATION MODE
                </div>
            </header>

            {/* MAIN GRID */}

            <section className="simulation-grid">

                {/* CONTROLS */}

                <div className="simulation-panel controls-panel">

                    <div className="simulation-panel-header">
                        <div>
                            <span>
                                SCENARIO CONFIGURATION
                            </span>

                            <h2>
                                Emergency Conditions
                            </h2>
                        </div>

                        <div className="simulation-number">
                            01
                        </div>
                    </div>

                    {/* LIVE SENSOR STATUS */}

                    <div className="simulation-live-status">
                        <div>
                            <span className="live-status-dot"></span>

                            <span>
                                {loadingSensors
                                    ? "CONNECTING TO LIVE SENSORS"
                                    : sensorError
                                        ? "LIVE SENSOR CONNECTION FAILED"
                                        : "LIVE SENSOR DATA CONNECTED"}
                            </span>
                        </div>

                        {liveSensor && (
                            <small>
                                {liveSensor.zone_id ||
                                    liveSensor.device_id ||
                                    "Building Alpha"}
                            </small>
                        )}
                    </div>

                    {/* ZONE */}

                    <label className="simulation-label">
                        Sensor / Zone
                    </label>

                    <select
                        className="simulation-select"
                        value={selectedZone}
```

#### Lines 661–880

```jsx
                        onChange={(event) => {
                            const zone = event.target.value;

                            setSelectedZone(zone);

                            localStorage.setItem(
                                "selectedZone",
                                zone
                            );

                            setResult(null);
                        }}
                    >
                        {sensors.map((sensor) => (
                            <option
                                key={sensor.zone_id}
                                value={sensor.zone_id}
                            >
                                Zone {sensor.zone_id}
                            </option>
                        ))}
                    </select>

                    {/* SCENARIO */}

                    <label className="simulation-label">
                        Emergency Scenario
                    </label>

                    <select
                        className="simulation-select"
                        value={scenario}
                        onChange={(event) => {
                            setScenario(event.target.value);
                            setResult(null);
                        }}
                    >
                        <option>Fire Emergency</option>
                        <option>High Temperature</option>
                        <option>Smoke Emergency</option>
                        <option>Overcrowding</option>
                        <option>Combined Emergency</option>
                    </select>

                    <p className="scenario-description">
                        {getScenarioDescription()}
                    </p>

                    {/* SCENARIO IMPACT */}

                    <div className="scenario-impact">
                        <span>
                            SCENARIO IMPACT
                        </span>

                        <p>
                            {scenarioImpact[scenario]}
                        </p>
                    </div>

                    {/* TEMPERATURE */}

                    <div className="simulation-control">
                        <div className="control-heading">
                            <div>
                                <span className="control-icon temperature">
                                    🌡
                                </span>

                                <span>
                                    Temperature
                                </span>
                            </div>

                            <strong>
                                {temperature}°C
                            </strong>
                        </div>

                        <input
                            className="simulation-range temperature-range"
                            type="range"
                            min="20"
                            max="50"
                            value={temperature}
                            onChange={(event) => {
                                setTemperature(
                                    Number(event.target.value)
                                );

                                setResult(null);
                            }}
                        />

                        <div className="range-labels">
                            <span>20°C</span>
                            <span>Normal</span>
                            <span>50°C</span>
                        </div>
                    </div>

                    {/* SMOKE */}

                    <div className="simulation-control">
                        <div className="control-heading">
                            <div>
                                <span className="control-icon smoke">
                                    ◉
                                </span>

                                <span>
                                    Smoke Level
                                </span>
                            </div>

                            <strong>
                                {Number(smoke).toFixed(2)}
                            </strong>
                        </div>

                        <input
                            className="simulation-range smoke-range"
                            type="range"
                            min="0"
                            max="1"
                            step="0.01"
                            value={smoke}
                            onChange={(event) => {
                                setSmoke(
                                    Number(event.target.value)
                                );

                                setResult(null);
                            }}
                        />

                        <div className="range-labels">
                            <span>0.00</span>
                            <span>Safe</span>
                            <span>1.00</span>
                        </div>
                    </div>

                    {/* OCCUPANCY */}

                    <div className="simulation-control">
                        <div className="control-heading">
                            <div>
                                <span className="control-icon occupancy">
                                    👥
                                </span>

                                <span>
                                    Occupancy
                                </span>
                            </div>

                            <strong>
                                {occupancy}
                            </strong>
                        </div>

                        <input
                            className="simulation-range occupancy-range"
                            type="range"
                            min="0"
                            max="60"
                            value={occupancy}
                            onChange={(event) => {
                                setOccupancy(
                                    Number(event.target.value)
                                );

                                setResult(null);
                            }}
                        />

                        <div className="range-labels">
                            <span>0</span>
                            <span>Capacity</span>
                            <span>60</span>
                        </div>
                    </div>

                    {/* LIVE RISK */}

                    <div className="simulation-live-summary">
                        <div>
                            <span>
                                LIVE RISK PREVIEW
                            </span>

                            <strong
                                style={{
                                    color: getRiskColor(liveRisk.level),
                                }}
                            >
                                {liveRisk.score}
                            </strong>
                        </div>

                        <div>
                            <span>
                                LEVEL
                            </span>

                            <strong
                                style={{
                                    color: getRiskColor(liveRisk.level),
                                }}
                            >
                                {liveRisk.level}
                            </strong>
                        </div>
                    </div>

                    {/* ACTIONS */}

                    <div className="simulation-actions">
                        <button
```

#### Lines 881–1100

```jsx
                            className="run-simulation"
                            onClick={runSimulation}
                            disabled={simulationLoading || loadingSensors}
                        >
                            <span>▶</span>

                            {simulationLoading
                                ? "Running DL Model..."
                                : "Run Simulation"}
                        </button>

                        <button
                            className="reset-simulation"
                            onClick={resetSimulation}
                            disabled={simulationLoading}
                        >
                            Reset
                        </button>
                    </div>
                </div>

                {/* DIGITAL TWIN */}

                <div className="simulation-panel building-panel">

                    <div className="simulation-panel-header">
                        <div>
                            <span>
                                DIGITAL TWIN PREVIEW
                            </span>

                            <h2>
                                Building Alpha
                            </h2>
                        </div>

                        <div className="building-live">
                            <span></span>
                            SIMULATION MODEL
                        </div>
                    </div>

                    <div className="simulation-building-area">
                        <div className="simulation-building">

                            {/* FLOOR 03 */}

                            <div className={getFloorClass(3)}>
                                <div className="floor-number">
                                    <small>03</small>
                                    <span>FLOOR</span>
                                </div>

                                <div className="simulation-rooms">
                                    <i className={getRoomClass(3, 1)}></i>
                                    <i className={getRoomClass(3, 2)}></i>
                                    <i className={getRoomClass(3, 3)}></i>
                                    <i className={getRoomClass(3, 4)}></i>
                                </div>
                            </div>

                            {/* FLOOR 02 */}

                            <div className={getFloorClass(2)}>
                                <div className="floor-number">
                                    <small>02</small>
                                    <span>FLOOR</span>
                                </div>

                                <div className="simulation-rooms">
                                    <i className={getRoomClass(2, 1)}></i>
                                    <i className={getRoomClass(2, 2)}></i>
                                    <i className={getRoomClass(2, 3)}></i>
                                    <i className={getRoomClass(2, 4)}></i>
                                </div>
                            </div>

                            {/* FLOOR 01 */}

                            <div className={getFloorClass(1)}>
                                <div className="floor-number">
                                    <small>01</small>
                                    <span>FLOOR</span>
                                </div>

                                <div className="simulation-rooms">
                                    <i className={getRoomClass(1, 1)}></i>
                                    <i className={getRoomClass(1, 2)}></i>
                                    <i className={getRoomClass(1, 3)}></i>
                                    <i className={getRoomClass(1, 4)}></i>
                                </div>
                            </div>
                        </div>

                        <div className="building-ground">
                            <span></span>
                            <span></span>
                            <span></span>
                        </div>
                    </div>

                    {/* LEGEND */}

                    <div className="building-legend">
                        <div>
                            <span className="legend-dot safe"></span>
                            Normal
                        </div>

                        <div>
                            <span className="legend-dot warning"></span>
                            Warning
                        </div>

                        <div>
                            <span className="legend-dot danger"></span>
                            Critical
                        </div>
                    </div>
                </div>
            </section>

            {/* SIMULATION RESULT */}

            <section
                className={`simulation-result ${
                    result ? "result-visible" : ""
                }`}
                style={{
                    "--risk-color": riskColor,
                }}
            >
                <div className="result-top">
                    <div>
                        <span className="result-eyebrow">
                            {result
                                ? "SIMULATION COMPLETE"
                                : "LIVE RISK PREVIEW"}
                        </span>

                        <h2>
                            {result
                                ? "Simulation Result"
                                : "Current Scenario Risk"}
                        </h2>
                    </div>

                    <div
                        className="risk-score"
                        style={{
                            borderColor: riskColor,
                            color: riskColor,
                        }}
                    >
                        <strong>
                            {activeRisk.score}
                        </strong>

                        <span>/100</span>
                    </div>
                </div>

                {/* RISK SUMMARY */}

                <div className="risk-summary">
                    <div className="risk-level-box">
                        <span>RISK LEVEL</span>

                        <strong
                            style={{
                                color: riskColor,
                            }}
                        >
                            {activeRisk.level}
                        </strong>
                    </div>

                    <div className="risk-bar-container">
                        <div className="risk-bar">
                            <div
                                className="risk-bar-fill"
                                style={{
                                    width: `${activeRisk.score}%`,
                                    background: riskColor,
                                }}
                            ></div>
                        </div>

                        <div className="risk-scale">
                            <span>LOW</span>
                            <span>MODERATE</span>
                            <span>HIGH</span>
                            <span>CRITICAL</span>
                        </div>
                    </div>

                    <div className="risk-scenario">
                        <span>SCENARIO</span>

                        <strong>
                            {scenario}
                        </strong>
                    </div>
                </div>

                {/* DL CONFIDENCE */}

                {result && (
                    <div className="simulation-dl-confidence">
                        <span>
                            PYTORCH MLP CONFIDENCE
                        </span>

                        <strong>
                            {result.confidence.toFixed(2)}%
                        </strong>
                    </div>
                )}

                {/* FACTORS */}
```

#### Lines 1101–1180

```jsx

                <div className="simulation-factors">
                    <div>
                        <span>TEMPERATURE</span>

                        <strong>
                            {activeRisk.factors.temperature}
                        </strong>
                    </div>

                    <div>
                        <span>SMOKE LEVEL</span>

                        <strong>
                            {activeRisk.factors.smoke}
                        </strong>
                    </div>

                    <div>
                        <span>OCCUPANCY</span>

                        <strong>
                            {activeRisk.factors.occupancy}
                        </strong>
                    </div>
                </div>

                {/* RECOMMENDATION */}

                <div className="risk-recommendation">
                    <div className="recommendation-icon">
                        {activeRisk.level === "CRITICAL"
                            ? "!"
                            : activeRisk.level === "HIGH"
                                ? "!"
                                : activeRisk.level === "MODERATE"
                                    ? "•"
                                    : "✓"}
                    </div>

                    <div>
                        <span>
                            RECOMMENDED ACTION
                        </span>

                        <p>
                            {activeRisk.recommendation}
                        </p>
                    </div>
                </div>

                {!result && (
                    <p className="preview-note">
                        Adjust the conditions above, then run the
                        simulation to generate a final Deep Learning
                        assessment.
                    </p>
                )}
            </section>

            {/* FOOTER */}

            <footer className="simulation-footer">
                <span>
                    ● Deep Learning risk engine
                </span>

                <span>
                    Digital Twin simulation environment
                </span>

                <span>
                    Scenario: {scenario}
                </span>
            </footer>
        </div>
    );
}

export default Simulation;
```

### frontend/src/pages/AIInsights.jsx

Frontend source reference — not executable in this Python notebook.

#### Lines 1–220

```jsx
import { useEffect, useState } from "react";
import { getSensors } from "../services/api";
import {
    calculateRisk,
    getRiskBackground,
    getRiskColor,
} from "../utils/riskEngine";
import "./AIInsights.css";

function AIInsights() {
    const [sensors, setSensors] = useState([]);
    const [error, setError] = useState("");

    useEffect(() => {
        const loadSensors = async () => {
            try {
                const data = await getSensors();
                setSensors(data);
                setError("");
            } catch (err) {
                setError("Unable to connect to EVAC-TWIN backend.");
            }
        };

        loadSensors();

        const interval = setInterval(loadSensors, 3000);

        return () => clearInterval(interval);
    }, []);

    const latest = sensors[0];

const calculatedRisk = latest
    ? calculateRisk(latest)
    : {
        level: "WAITING",
        score: 0,
        recommendation: "Waiting for live sensor information.",
        factors: {
            temperature: 0,
            smoke: 0,
            occupancy: 0,
        },
    };

const risk = latest
    ? {
        ...calculatedRisk,
        level:
            latest.dl_prediction ||
            latest.risk_level ||
            calculatedRisk.level,
        score:
            latest.risk_score ??
            calculatedRisk.score,
        recommendation:
            latest.recommendation ||
            calculatedRisk.recommendation,
    }
    : calculatedRisk;

    const getInsight = () => {
        if (!latest) {
            return {
                title: "Waiting for sensor data",
                text: "AI analysis will begin when live sensor readings are available.",
            };
        }

        if (risk.level === "CRITICAL") {
            return {
                title: "Critical emergency conditions",
                text: "Multiple environmental indicators are showing dangerous values. Immediate evacuation procedures should be considered.",
            };
        }

        if (risk.level === "HIGH") {
            return {
                title: "High-risk conditions detected",
                text: "Sensor readings indicate a significant increase in emergency risk. Monitor affected zones and prepare evacuation routes.",
            };
        }

        if (risk.level === "MODERATE") {
            return {
                title: "Elevated risk detected",
                text: "Some environmental conditions are above normal levels. Continue monitoring for changes.",
            };
        }

        return {
            title: "Building conditions are stable",
            text: "Current environmental and occupancy readings remain within acceptable ranges.",
        };
    };

    const insight = getInsight();
    const dlConfidence = Number(latest?.dl_confidence ?? latest?.confidence ?? 0);

    return (
        <div className="ai-page">

            {/* HEADER */}
            <header className="ai-header">
                <div>
                    <p className="ai-eyebrow">
                        AI INTELLIGENCE / DECISION SUPPORT
                    </p>

                    <h1>AI Insights</h1>

                    <p className="ai-subtitle">
                        Automated emergency risk analysis from live Digital Twin data.
                    </p>
                </div>

                <div className="ai-live">
                    <span></span>
                    LIVE ANALYSIS
                </div>
            </header>

            {/* ERROR */}
            {error && (
                <div className="ai-error">
                    ⚠️ {error}
                </div>
            )}

            {/* LOADING */}
            {!latest && !error && (
                <div className="ai-loading">
                    <div className="ai-loader">◌</div>

                    <h2>Waiting for Digital Twin data</h2>

                    <p>
                        AI analysis will update automatically.
                    </p>
                </div>
            )}

            {latest && (
                <>
                    {/* AI OVERVIEW */}
                    <section className="ai-overview">

                        <div className="ai-risk-card">
                            <div className="ai-dl-confidence">
    <span>DEEP LEARNING CONFIDENCE</span>
    <strong>{dlConfidence.toFixed(2)}%</strong>
</div>

                            <div>
                                <span className="ai-label">
                                    CURRENT AI RISK SCORE
                                </span>

                                <div
                                    className="ai-risk-number"
                                    style={{
                                        color: getRiskColor(risk.level),
                                    }}
                                >
                                    {risk.score}%
                                </div>

                                <strong
                                    className="ai-risk-level"
                                    style={{
                                        color: getRiskColor(risk.level),
                                    }}
                                >
                                    {risk.level} RISK
                                </strong>
                            </div>

                            <div
                                className="ai-risk-ring"
                                style={{
                                    borderColor: getRiskColor(risk.level),
                                }}
                            >
                                <span>{risk.score}</span>
                                <small>/100</small>
                            </div>

                        </div>

                        {/* RECOMMENDATION */}
                        <div
                            className="ai-recommendation"
                            style={{
                                background: getRiskBackground(risk.level),
                                borderColor: getRiskColor(risk.level),
                            }}
                        >
                            <span className="ai-label">
                                AI RECOMMENDATION
                            </span>

                            <h2>{insight.title}</h2>

                            <p>{insight.text}</p>

                            <div className="recommendation-action">
                                <strong>Recommended Action</strong>

                                <span>
                                    {risk.recommendation}
                                </span>
                            </div>
                        </div>

                    </section>

                    {/* SENSOR INTELLIGENCE */}
                    <section className="ai-panel">
```

#### Lines 221–440

```jsx
                        <div className="ai-panel-header">
                            <div>
                                <span className="ai-label">
                                    SENSOR INTELLIGENCE
                                </span>

                                <h2>Risk Factors</h2>
                            </div>

                            <span className="ai-device">
                                {latest.device_id}
                            </span>
                        </div>

                        <div className="factor-grid">

                            {/* TEMPERATURE */}
                            <div className="factor-card">

                                <div className="factor-top">
                                    <span>🌡️</span>
                                    <strong>Temperature</strong>
                                </div>

                                <div className="factor-value">
                                    {latest.temperature}°C
                                </div>

                                <div className="factor-bar">
                                    <div
                                        style={{
                                            width: `${Math.min(
                                                (latest.temperature / 50) * 100,
                                                100
                                            )
                                                }% `,
                                        }}
                                    ></div>
                                </div>

                                <p>
                                    {latest.temperature >= 40
                                        ? "Critical temperature detected"
                                        : latest.temperature >= 38
                                            ? "High temperature detected"
                                            : latest.temperature >= 35
                                                ? "Elevated temperature detected"
                                                : "Temperature remains stable"}
                                </p>

                            </div>

                            {/* SMOKE */}
                            <div className="factor-card">

                                <div className="factor-top">
                                    <span>💨</span>
                                    <strong>Smoke</strong>
                                </div>

                                <div className="factor-value">
                                    {latest.smoke}
                                </div>

                                <div className="factor-bar">
                                    <div
                                        style={{
                                            width: `${latest.smoke * 100}% `,
                                        }}
                                    ></div>
                                </div>

                                <p>
                                    {latest.smoke >= 0.8
                                        ? "High smoke concentration"
                                        : latest.smoke >= 0.5
                                            ? "Elevated smoke level"
                                            : latest.smoke >= 0.3
                                                ? "Low smoke detected"
                                                : "Smoke level normal"}
                                </p>

                            </div>

                            {/* OCCUPANCY */}
                            <div className="factor-card">

                                <div className="factor-top">
                                    <span>👥</span>
                                    <strong>Occupancy</strong>
                                </div>

                                <div className="factor-value">
                                    {latest.occupancy}
                                </div>

                                <div className="factor-bar">
                                    <div
                                        style={{
                                            width: `${Math.min(
                                                (latest.occupancy / 50) * 100,
                                                100
                                            )
                                                }% `,
                                        }}
                                    ></div>
                                </div>

                                <p>
                                    {latest.occupancy >= 45
                                        ? "Critical occupancy load"
                                        : latest.occupancy >= 30
                                            ? "Moderate occupancy"
                                            : latest.occupancy >= 15
                                                ? "Elevated occupancy"
                                                : "Occupancy load normal"}
                                </p>

                            </div>

                            {/* HUMIDITY */}
                            <div className="factor-card">

                                <div className="factor-top">
                                    <span>💧</span>
                                    <strong>Humidity</strong>
                                </div>

                                <div className="factor-value">
                                    {latest.humidity}%
                                </div>

                                <div className="factor-bar">
                                    <div
                                        style={{
                                            width: `${Math.min(
                                                latest.humidity,
                                                100
                                            )
                                                }% `,
                                        }}
                                    ></div>
                                </div>

                                <p>
                                    Environmental humidity monitoring active
                                </p>

                            </div>

                        </div>

                    </section>

                    {/* DECISION ENGINE */}
                    <section className="ai-panel">

                        <div className="ai-panel-header">

                            <div>
                                <span className="ai-label">
                                    DECISION ENGINE
                                </span>

                                <h2>Risk Assessment</h2>
                            </div>

                        </div>

                        <div className="assessment-list">

                            <div className="assessment-row">
                                <span>
                                    Temperature contribution
                                </span>

                                <strong>
                                    {risk.factors.temperature > 0
                                        ? `+ ${risk.factors.temperature} `
                                        : "0"}
                                </strong>
                            </div>

                            <div className="assessment-row">
                                <span>
                                    Smoke contribution
                                </span>

                                <strong>
                                    {risk.factors.smoke > 0
                                        ? `+ ${risk.factors.smoke} `
                                        : "0"}
                                </strong>
                            </div>

                            <div className="assessment-row">
                                <span>
                                    Occupancy contribution
                                </span>

                                <strong>
                                    {risk.factors.occupancy > 0
                                        ? `+ ${risk.factors.occupancy} `
                                        : "0"}
                                </strong>
                            </div>

                            <div className="assessment-total">

                                <span>
                                    Total AI Risk Score
                                </span>

                                <strong
                                    style={{
                                        color: getRiskColor(risk.level),
                                    }}
                                >
                                    {risk.score} / 100
                                </strong>
```

#### Lines 441–459

```jsx

                            </div>

                        </div>

                    </section>

                    {/* FOOTER */}
                    <div className="ai-footer">
                        🟢 AI engine synchronized · Analysis updates every 3 seconds
                    </div>
                </>
            )}

        </div>
    );
}

export default AIInsights;
```

### frontend/src/pages/Analytics.jsx

Frontend source reference — not executable in this Python notebook.

#### Lines 1–220

```jsx
import { useEffect, useState } from "react";
import {
    Bar,
    BarChart,
    CartesianGrid,
    Line,
    LineChart,
    ResponsiveContainer,
    Tooltip,
    XAxis,
    YAxis,
} from "recharts";
import { getSensors } from "../services/api";

function Analytics() {
    const [sensors, setSensors] = useState([]);
    const [error, setError] = useState("");

    useEffect(() => {
        const loadSensors = async () => {
            try {
                const data = await getSensors();
                setSensors(data);
                setError("");
            } catch (err) {
                setError("Unable to load analytics data.");
            }
        };

        loadSensors();

        const interval = setInterval(loadSensors, 5000);

        return () => clearInterval(interval);
    }, []);

    if (error) {
        return (
            <div style={styles.page}>
                <h1>Analytics</h1>
                <div style={styles.error}>{error}</div>
            </div>
        );
    }

    if (sensors.length === 0) {
        return (
            <div style={styles.page}>
                <h1>Analytics</h1>
                <div style={styles.loading}>
                    Waiting for sensor data...
                </div>
            </div>
        );
    }

    // Remove duplicate readings
    const uniqueSensors = sensors.filter(
        (sensor, index, self) =>
            index ===
            self.findIndex(
                (item) =>
                    item.timestamp === sensor.timestamp
            )
    );

    const temperatures = uniqueSensors.map(
        (sensor) => sensor.temperature
    );

    const humidities = uniqueSensors.map(
        (sensor) => sensor.humidity
    );

    const smokeValues = uniqueSensors.map(
        (sensor) => sensor.smoke
    );

    const occupancies = uniqueSensors.map(
        (sensor) => sensor.occupancy
    );

    const average = (values) =>
        values.reduce((sum, value) => sum + value, 0) /
        values.length;

    const max = (values) => Math.max(...values);

    const min = (values) => Math.min(...values);

    const warningCount = uniqueSensors.filter(
        (sensor) =>
            sensor.smoke >= 0.8 ||
            sensor.temperature >= 38 ||
            sensor.occupancy >= 45
    ).length;

    // Chart data
    const chartData = [...uniqueSensors]
        .reverse()
        .map((sensor) => ({
            time: new Date(
                sensor.timestamp
            ).toLocaleTimeString([], {
                hour: "2-digit",
                minute: "2-digit",
                second: "2-digit",
            }),
            temperature: sensor.temperature,
            humidity: sensor.humidity,
            smoke: sensor.smoke,
            occupancy: sensor.occupancy,
        }));

    const warningPercentage =
        (warningCount / uniqueSensors.length) * 100;

    return (
        <div style={styles.page}>

            {/* HEADER */}
            <div style={styles.header}>
                <div>
                    <h1 style={styles.title}>
                        Analytics
                    </h1>

                    <p style={styles.subtitle}>
                        Historical emergency and sensor analytics
                    </p>
                </div>

                <div style={styles.liveBadge}>
                    <span style={styles.dot}>●</span>
                    LIVE DATA
                </div>
            </div>

            {/* SUMMARY CARDS */}
            <div style={styles.statsGrid}>

                <div style={styles.statCard}>
                    <div style={styles.statIcon}>🌡️</div>
                    <p style={styles.statLabel}>
                        Avg Temperature
                    </p>

                    <h2 style={styles.statValue}>
                        {average(temperatures).toFixed(2)}°C
                    </h2>

                    <small style={styles.statSub}>
                        Range {min(temperatures).toFixed(1)}° -
                        {max(temperatures).toFixed(1)}°
                    </small>
                </div>

                <div style={styles.statCard}>
                    <div style={styles.statIcon}>💧</div>
                    <p style={styles.statLabel}>
                        Avg Humidity
                    </p>

                    <h2 style={styles.statValue}>
                        {average(humidities).toFixed(2)}%
                    </h2>

                    <small style={styles.statSub}>
                        Environmental condition
                    </small>
                </div>

                <div style={styles.statCard}>
                    <div style={styles.statIcon}>💨</div>
                    <p style={styles.statLabel}>
                        Avg Smoke
                    </p>

                    <h2 style={styles.statValue}>
                        {average(smokeValues).toFixed(2)}
                    </h2>

                    <small style={styles.statSub}>
                        0 - 1 scale
                    </small>
                </div>

                <div style={styles.statCard}>
                    <div style={styles.statIcon}>👥</div>
                    <p style={styles.statLabel}>
                        Avg Occupancy
                    </p>

                    <h2 style={styles.statValue}>
                        {average(occupancies).toFixed(1)}
                    </h2>

                    <small style={styles.statSub}>
                        People detected
                    </small>
                </div>

            </div>

            {/* EMERGENCY SUMMARY */}
            <div style={styles.emergencyCard}>

                <div>
                    <p style={styles.emergencyLabel}>
                        Emergency Analysis
                    </p>

                    <h2 style={styles.warningNumber}>
                        {warningCount}
                    </h2>

                    <p style={styles.emergencyText}>
                        Warning readings detected
                    </p>
                </div>
```

#### Lines 221–440

```jsx

                <div style={styles.emergencyRight}>
                    <div style={styles.progressBackground}>
                        <div
                            style={{
                                ...styles.progress,
                                width: `${Math.min(
                                    warningPercentage,
                                    100
                                )}%`,
                            }}
                        />
                    </div>

                    <span>
                        {warningPercentage.toFixed(1)}% of readings
                    </span>
                </div>

            </div>

            {/* TEMPERATURE CHART */}
            <div style={styles.chartCard}>

                <div style={styles.chartHeader}>
                    <div>
                        <h2 style={styles.chartTitle}>
                            🌡️ Temperature Trend
                        </h2>

                        <p style={styles.chartSubtitle}>
                            Temperature variation over recent readings
                        </p>
                    </div>

                    <span style={styles.chartUnit}>
                        °C
                    </span>
                </div>

                <ResponsiveContainer width="100%" height={300}>
                    <LineChart data={chartData}>
                        <CartesianGrid
                            strokeDasharray="3 3"
                            stroke="#334155"
                        />

                        <XAxis
                            dataKey="time"
                            stroke="#94a3b8"
                        />

                        <YAxis
                            stroke="#94a3b8"
                        />

                        <Tooltip
                            contentStyle={{
                                background: "#0f172a",
                                border: "1px solid #334155",
                                borderRadius: "10px",
                                color: "#fff",
                            }}
                        />

                        <Line
                            type="monotone"
                            dataKey="temperature"
                            stroke="#f97316"
                            strokeWidth={3}
                            dot={false}
                            activeDot={{ r: 6 }}
                        />
                    </LineChart>
                </ResponsiveContainer>

            </div>

            {/* SMOKE + OCCUPANCY */}
            <div style={styles.twoColumn}>

                <div style={styles.chartCard}>

                    <div style={styles.chartHeader}>
                        <div>
                            <h2 style={styles.chartTitle}>
                                💨 Smoke Levels
                            </h2>

                            <p style={styles.chartSubtitle}>
                                Fire/smoke risk indicator
                            </p>
                        </div>
                    </div>

                    <ResponsiveContainer width="100%" height={280}>
                        <LineChart data={chartData}>
                            <CartesianGrid
                                strokeDasharray="3 3"
                                stroke="#334155"
                            />

                            <XAxis
                                dataKey="time"
                                stroke="#94a3b8"
                            />

                            <YAxis
                                domain={[0, 1]}
                                stroke="#94a3b8"
                            />

                            <Tooltip
                                contentStyle={{
                                    background: "#0f172a",
                                    border: "1px solid #334155",
                                    borderRadius: "10px",
                                }}
                            />

                            <Line
                                type="monotone"
                                dataKey="smoke"
                                stroke="#ef4444"
                                strokeWidth={3}
                                dot={false}
                            />
                        </LineChart>
                    </ResponsiveContainer>

                </div>

                <div style={styles.chartCard}>

                    <div style={styles.chartHeader}>
                        <div>
                            <h2 style={styles.chartTitle}>
                                👥 Occupancy Trend
                            </h2>

                            <p style={styles.chartSubtitle}>
                                People detected over time
                            </p>
                        </div>
                    </div>

                    <ResponsiveContainer width="100%" height={280}>
                        <BarChart data={chartData}>
                            <CartesianGrid
                                strokeDasharray="3 3"
                                stroke="#334155"
                            />

                            <XAxis
                                dataKey="time"
                                stroke="#94a3b8"
                            />

                            <YAxis
                                stroke="#94a3b8"
                            />

                            <Tooltip
                                contentStyle={{
                                    background: "#0f172a",
                                    border: "1px solid #334155",
                                    borderRadius: "10px",
                                }}
                            />

                            <Bar
                                dataKey="occupancy"
                                fill="#38bdf8"
                                radius={[5, 5, 0, 0]}
                            />
                        </BarChart>
                    </ResponsiveContainer>

                </div>

            </div>

            {/* RECENT HISTORY */}
            <div style={styles.historyCard}>

                <h2 style={styles.chartTitle}>
                    📊 Recent Sensor History
                </h2>

                <div style={styles.table}>

                    <div style={styles.tableHeader}>
                        <span>Time</span>
                        <span>Temperature</span>
                        <span>Humidity</span>
                        <span>Smoke</span>
                        <span>Occupancy</span>
                    </div>

                    {uniqueSensors
                        .slice(0, 10)
                        .map((sensor) => {

                            const warning =
                                sensor.smoke >= 0.8 ||
                                sensor.temperature >= 38 ||
                                sensor.occupancy >= 45;

                            return (
                                <div
                                    key={sensor.timestamp}
                                    style={{
                                        ...styles.row,
                                        background: warning
                                            ? "#29161a"
                                            : "transparent",
                                    }}
                                >
                                    <span>
                                        {new Date(
```

#### Lines 441–660

```jsx
                                            sensor.timestamp
                                        ).toLocaleTimeString()}
                                    </span>

                                    <span>
                                        {sensor.temperature} °C
                                    </span>

                                    <span>
                                        {sensor.humidity} %
                                    </span>

                                    <span
                                        style={{
                                            color:
                                                sensor.smoke >= 0.8
                                                    ? "#f87171"
                                                    : "#e2e8f0",
                                        }}
                                    >
                                        {sensor.smoke}
                                    </span>

                                    <span
                                        style={{
                                            color:
                                                sensor.occupancy >= 45
                                                    ? "#f87171"
                                                    : "#e2e8f0",
                                        }}
                                    >
                                        {sensor.occupancy}
                                    </span>
                                </div>
                            );
                        })}

                </div>

            </div>

            <p style={styles.refresh}>
                Automatically updating every 5 seconds •{" "}
                {uniqueSensors.length} unique readings
            </p>

        </div>
    );
}

const styles = {

    page: {
        minHeight: "100vh",
        padding: "30px",
        background: "#0f172a",
        color: "#f8fafc",
        boxSizing: "border-box",
    },

    header: {
        display: "flex",
        justifyContent: "space-between",
        alignItems: "center",
        marginBottom: "30px",
    },

    title: {
        margin: 0,
        fontSize: "34px",
        fontWeight: "700",
    },

    subtitle: {
        marginTop: "8px",
        color: "#94a3b8",
    },

    liveBadge: {
        background: "#103b2a",
        color: "#4ade80",
        padding: "10px 18px",
        borderRadius: "20px",
        fontWeight: "700",
        fontSize: "13px",
    },

    dot: {
        marginRight: "7px",
    },

    statsGrid: {
        display: "grid",
        gridTemplateColumns:
            "repeat(auto-fit, minmax(220px, 1fr))",
        gap: "20px",
        marginBottom: "25px",
    },

    statCard: {
        background: "#1e293b",
        border: "1px solid #334155",
        borderRadius: "16px",
        padding: "22px",
    },

    statIcon: {
        fontSize: "28px",
        marginBottom: "12px",
    },

    statLabel: {
        margin: 0,
        color: "#94a3b8",
        fontSize: "14px",
    },

    statValue: {
        margin: "8px 0",
        fontSize: "30px",
    },

    statSub: {
        color: "#64748b",
    },

    emergencyCard: {
        display: "flex",
        justifyContent: "space-between",
        alignItems: "center",
        gap: "30px",
        background: "#29161a",
        border: "1px solid #7f1d1d",
        borderRadius: "16px",
        padding: "25px",
        marginBottom: "25px",
    },

    emergencyLabel: {
        margin: 0,
        color: "#fca5a5",
        fontSize: "14px",
    },

    warningNumber: {
        margin: "5px 0",
        fontSize: "36px",
        color: "#f87171",
    },

    emergencyText: {
        margin: 0,
        color: "#cbd5e1",
    },

    emergencyRight: {
        minWidth: "250px",
        color: "#fca5a5",
        fontSize: "13px",
    },

    progressBackground: {
        height: "8px",
        background: "#451a1a",
        borderRadius: "10px",
        overflow: "hidden",
        marginBottom: "8px",
    },

    progress: {
        height: "100%",
        background: "#ef4444",
        borderRadius: "10px",
    },

    chartCard: {
        background: "#1e293b",
        border: "1px solid #334155",
        borderRadius: "16px",
        padding: "25px",
        marginBottom: "25px",
    },

    chartHeader: {
        display: "flex",
        justifyContent: "space-between",
        alignItems: "center",
        marginBottom: "15px",
    },

    chartTitle: {
        margin: 0,
        fontSize: "20px",
    },

    chartSubtitle: {
        marginTop: "6px",
        color: "#64748b",
        fontSize: "13px",
    },

    chartUnit: {
        color: "#94a3b8",
        fontSize: "13px",
    },

    twoColumn: {
        display: "grid",
        gridTemplateColumns:
            "repeat(auto-fit, minmax(400px, 1fr))",
        gap: "25px",
    },

    historyCard: {
        background: "#1e293b",
        border: "1px solid #334155",
        borderRadius: "16px",
        padding: "25px",
        marginTop: "0",
    },
```

#### Lines 661–713

```jsx

    table: {
        width: "100%",
        overflowX: "auto",
        marginTop: "20px",
    },

    tableHeader: {
        display: "grid",
        gridTemplateColumns:
            "1.3fr 1fr 1fr 1fr 1fr",
        minWidth: "650px",
        gap: "15px",
        padding: "14px 10px",
        color: "#94a3b8",
        fontSize: "13px",
        borderBottom: "1px solid #334155",
    },

    row: {
        display: "grid",
        gridTemplateColumns:
            "1.3fr 1fr 1fr 1fr 1fr",
        minWidth: "650px",
        gap: "15px",
        padding: "14px 10px",
        borderBottom: "1px solid #273449",
        color: "#e2e8f0",
        fontSize: "14px",
    },

    error: {
        background: "#3b1111",
        color: "#ff8080",
        padding: "15px",
        borderRadius: "10px",
    },

    loading: {
        padding: "40px",
        textAlign: "center",
        color: "#94a3b8",
    },

    refresh: {
        textAlign: "right",
        color: "#64748b",
        fontSize: "12px",
        marginTop: "15px",
    },
};

export default Analytics;
```

### frontend/src/App.css

Frontend source reference — not executable in this Python notebook.

#### Lines 1–220

```css
/* =========================================================
   EVAC-TWIN — APP / SIDEBAR
   ========================================================= */

* {
  box-sizing: border-box;
}

html,
body,
#root {
  margin: 0;
  min-height: 100%;
  width: 100%;
}

body {
  font-family: Inter, "Segoe UI", Arial, sans-serif;
  background: #0f172a;
  color: #f8fafc;
}

.app-layout {
  min-height: 100vh;
  display: flex;
  background: #0f172a;
}

/* =========================================================
   SIDEBAR
   ========================================================= */

.sidebar {
  width: 260px;
  height: 100vh;

  position: fixed;
  left: 0;
  top: 0;

  display: flex;
  flex-direction: column;

  padding: 24px 16px 18px;

  background: #111827;
  border-right: 1px solid #263244;

  z-index: 100;

  overflow-y: auto;
  overflow-x: hidden;

  scrollbar-width: thin;
  scrollbar-color: #26344d transparent;
}

/* Chrome / Edge scrollbar */

.sidebar::-webkit-scrollbar {
  width: 5px;
}

.sidebar::-webkit-scrollbar-track {
  background: transparent;
}

.sidebar::-webkit-scrollbar-thumb {
  background: #26344d;
  border-radius: 10px;
}

.sidebar::-webkit-scrollbar-thumb:hover {
  background: #33445f;
}

/* =========================================================
   BRAND
   ========================================================= */

.brand {
  width: 100%;

  display: flex;
  align-items: center;

  gap: 11px;

  padding: 4px 8px 24px;

  border-bottom: 1px solid #263244;

  flex-shrink: 0;
}

.brand-icon {
  width: 42px;
  height: 42px;

  flex: 0 0 42px;

  display: flex;
  align-items: center;
  justify-content: center;

  border-radius: 10px;

  background: #1d4ed8;
  color: #ffffff;

  font-size: 21px;
  font-weight: 800;

  box-shadow: 0 0 20px rgba(29, 78, 216, 0.22);
}

.brand>div:last-child {
  min-width: 0;
}

.brand h1 {
  margin: 0;

  color: #f8fafc;

  font-size: 17px;
  line-height: 1.2;
  font-weight: 750;

  letter-spacing: 0.4px;

  white-space: nowrap;
}

.brand span {
  display: block;

  margin-top: 5px;

  color: #64748b;

  font-size: 9px;
  line-height: 1.3;

  white-space: nowrap;
}

/* =========================================================
   NAVIGATION
   ========================================================= */

.navigation {
  width: 100%;

  display: flex;
  flex-direction: column;

  margin-top: 25px;

  gap: 5px;

  flex-shrink: 0;
}

.nav-section {
  margin: 5px 10px 9px;

  color: #64748b;

  font-size: 9px;
  font-weight: 750;

  letter-spacing: 1.5px;

  line-height: 1.3;

  white-space: nowrap;
}

.nav-section.intelligence {
  margin-top: 25px;
}

/* =========================================================
   NAV ITEMS
   ========================================================= */

.nav-item {
  width: 100%;
  min-height: 43px;

  display: flex;
  align-items: center;

  gap: 12px;

  padding: 10px 12px;

  border-radius: 9px;

  color: #94a3b8;

  text-decoration: none;

  font-size: 13px;
  font-weight: 550;

  line-height: 1.2;

  transition:
    background 0.2s ease,
    color 0.2s ease,
    transform 0.2s ease;

  white-space: nowrap;
}

.nav-item span {
  width: 22px;
  min-width: 22px;
```

#### Lines 221–440

```css

  display: flex;
  align-items: center;
  justify-content: center;

  color: #718096;

  text-align: center;

  font-size: 17px;
  line-height: 1;

  transition: color 0.2s ease;
}

.nav-item:hover {
  background: #1a2537;
  color: #e2e8f0;
}

.nav-item:hover span {
  color: #93c5fd;
}

.nav-item.active {
  background: #172554;

  color: #60a5fa;

  box-shadow:
    inset 3px 0 0 #3b82f6,
    0 4px 14px rgba(15, 23, 42, 0.2);
}

.nav-item.active span {
  color: #60a5fa;
}

/* =========================================================
   SYSTEM CARD
   ========================================================= */

.system-card {
  width: 100%;

  margin-top: auto;

  display: flex;
  align-items: center;

  gap: 10px;

  padding: 13px;

  background: #172235;

  border: 1px solid #263244;

  border-radius: 11px;

  flex-shrink: 0;
}

.system-status-dot {
  width: 8px;
  height: 8px;

  flex: 0 0 8px;

  border-radius: 50%;

  background: #22c55e;

  box-shadow:
    0 0 0 4px rgba(34, 197, 94, 0.08),
    0 0 10px rgba(34, 197, 94, 0.45);
}

.system-card>div:last-child {
  min-width: 0;
}

.system-card strong {
  display: block;

  color: #e2e8f0;

  font-size: 11px;
  font-weight: 700;

  line-height: 1.25;

  white-space: nowrap;
}

.system-card span {
  display: block;

  margin-top: 4px;

  color: #64748b;

  font-size: 9px;

  line-height: 1.2;

  white-space: nowrap;
}

/* =========================================================
   MAIN CONTENT
   ========================================================= */

.main-content {
  width: calc(100% - 260px);

  min-height: 100vh;

  margin-left: 260px;

  background: #0f172a;

  overflow-x: hidden;
}

/* =========================================================
   SHORT SCREEN FIX
   ========================================================= */

@media (max-height: 650px) {

  .sidebar {
    padding-top: 16px;
    padding-bottom: 12px;
  }

  .brand {
    padding-bottom: 16px;
  }

  .navigation {
    margin-top: 16px;
    gap: 3px;
  }

  .nav-item {
    min-height: 38px;
    padding: 8px 12px;
  }

  .nav-section.intelligence {
    margin-top: 15px;
  }

  .system-card {
    margin-top: 15px;
  }
}

/* =========================================================
   TABLET
   ========================================================= */

@media (max-width: 1000px) {

  .sidebar {
    width: 235px;
  }

  .main-content {
    width: calc(100% - 235px);
    margin-left: 235px;
  }

  .brand h1 {
    font-size: 16px;
  }

  .nav-item {
    font-size: 12px;
  }
}

/* =========================================================
   SMALL SCREEN
   ========================================================= */

@media (max-width: 700px) {

  .sidebar {
    width: 215px;
    padding-left: 12px;
    padding-right: 12px;
  }

  .main-content {
    width: calc(100% - 215px);
    margin-left: 215px;
  }

  .brand {
    padding-left: 6px;
    padding-right: 6px;
  }

  .brand-icon {
    width: 38px;
    height: 38px;
    flex-basis: 38px;

    font-size: 19px;
  }

  .brand h1 {
    font-size: 14px;
  }

  .brand span {
    font-size: 8px;
  }
```

#### Lines 441–521

```css

  .nav-item {
    min-height: 40px;

    padding: 9px 10px;

    gap: 9px;

    font-size: 11px;
  }

  .nav-item span {
    width: 20px;
    min-width: 20px;

    font-size: 15px;
  }

  .nav-section {
    font-size: 8px;
  }

  .system-card {
    padding: 11px 10px;
  }

  .system-card strong {
    font-size: 10px;
  }

  .system-card span {
    font-size: 8px;
  }
}

/* =========================================================
   VERY SMALL WIDTH
   ========================================================= */

@media (max-width: 520px) {

  .sidebar {
    width: 190px;
  }

  .main-content {
    width: calc(100% - 190px);
    margin-left: 190px;
  }

  .brand {
    gap: 8px;
  }

  .brand-icon {
    width: 34px;
    height: 34px;
    flex-basis: 34px;

    font-size: 17px;
  }

  .brand h1 {
    font-size: 12px;
  }

  .brand span {
    font-size: 7px;
  }

  .nav-item {
    font-size: 10px;
    padding: 9px 8px;
  }

  .nav-item span {
    width: 18px;
    min-width: 18px;
    font-size: 14px;
  }
}
```

### frontend/src/index.css

Frontend source reference — not executable in this Python notebook.

#### Lines 1–220

```css
@import url("https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700;800&display=swap");

* {
  box-sizing: border-box;
  margin: 0;
  padding: 0;
}

:root {
  font-family: "Inter", sans-serif;
  color: #e8edf5;
  background: #070b12;
  font-synthesis: none;
}

body {
  min-width: 320px;
  min-height: 100vh;
  background:
    radial-gradient(circle at 75% 15%, rgba(44, 76, 120, 0.12), transparent 30%),
    #070b12;
}

button {
  font-family: inherit;
}

.app {
  min-height: 100vh;
  display: flex;
}

/* SIDEBAR */

.sidebar {
  width: 250px;
  min-height: 100vh;
  padding: 26px 16px;
  border-right: 1px solid rgba(255, 255, 255, 0.07);
  background: rgba(9, 13, 21, 0.92);
  display: flex;
  flex-direction: column;
}

.brand {
  display: flex;
  align-items: center;
  gap: 12px;
  padding: 0 10px 30px;
}

.brand-icon {
  width: 38px;
  height: 38px;
  border: 1px solid rgba(95, 170, 255, 0.5);
  border-radius: 10px;
  display: grid;
  place-items: center;
  font-weight: 800;
  color: #75b8ff;
  background: rgba(61, 133, 210, 0.12);
}

.brand h1 {
  font-size: 15px;
  letter-spacing: 1.5px;
}

.brand span {
  display: block;
  color: #687487;
  font-size: 9px;
  margin-top: 4px;
}

.navigation {
  display: flex;
  flex-direction: column;
  gap: 5px;
}

.nav-section {
  color: #4d5869;
  font-size: 9px;
  font-weight: 700;
  letter-spacing: 1.5px;
  padding: 20px 12px 8px;
}

.nav-item {
  border: none;
  background: transparent;
  color: #7f8b9e;
  padding: 11px 12px;
  border-radius: 8px;
  text-align: left;
  display: flex;
  gap: 12px;
  align-items: center;
  cursor: pointer;
  font-size: 12px;
  transition: 0.2s;
}

.nav-item:hover,
.nav-item.active {
  background: rgba(82, 145, 215, 0.1);
  color: #dcecff;
}

.nav-item.active {
  border: 1px solid rgba(82, 145, 215, 0.15);
}

.nav-item span {
  width: 18px;
  text-align: center;
  color: #659ed5;
}

.system-card {
  margin-top: auto;
  padding: 13px;
  border: 1px solid rgba(75, 180, 120, 0.14);
  background: rgba(49, 147, 92, 0.05);
  border-radius: 10px;
  display: flex;
  align-items: center;
  gap: 10px;
}

.system-card strong {
  display: block;
  font-size: 10px;
}

.system-card span:last-child {
  display: block;
  color: #657184;
  font-size: 8px;
  margin-top: 3px;
}

.status-dot,
.live-dot,
.pulse {
  width: 7px;
  height: 7px;
  border-radius: 50%;
  background: #48c98a;
  box-shadow: 0 0 12px rgba(72, 201, 138, 0.7);
}

/* MAIN */

.main-content {
  flex: 1;
  padding: 30px 36px;
  overflow: auto;
}

.topbar {
  display: flex;
  align-items: center;
  justify-content: space-between;
  margin-bottom: 28px;
}

.eyebrow,
.panel-label,
.stat-label {
  font-size: 9px;
  letter-spacing: 1.5px;
  color: #647084;
  font-weight: 700;
}

.topbar h2 {
  font-size: 24px;
  margin-top: 7px;
}

.top-status {
  display: flex;
  align-items: center;
  gap: 8px;
  font-size: 10px;
  color: #8995a7;
}

.divider {
  height: 14px;
  width: 1px;
  background: #29303b;
  margin: 0 5px;
}

/* CARDS */

.stats-grid {
  display: grid;
  grid-template-columns: repeat(4, 1fr);
  gap: 12px;
  margin-bottom: 14px;
}

.stat-card,
.twin-panel,
.insights-panel,
.event-panel,
.quick-panel {
  border: 1px solid rgba(255, 255, 255, 0.07);
  background: rgba(14, 19, 29, 0.72);
  border-radius: 12px;
}

.stat-card {
  padding: 18px;
}
```

#### Lines 221–440

```css
.stat-card strong {
  display: block;
  font-size: 24px;
  margin: 12px 0 5px;
}

.safe-text {
  color: #52c98b;
}

.stat-description {
  color: #657184;
  font-size: 9px;
}

/* DASHBOARD */

.dashboard-grid {
  display: grid;
  grid-template-columns: 2fr 1fr;
  gap: 14px;
}

.twin-panel,
.insights-panel,
.event-panel,
.quick-panel {
  padding: 20px;
}

.panel-header {
  display: flex;
  justify-content: space-between;
  align-items: center;
}

.panel-header h3,
.quick-panel h3 {
  font-size: 14px;
  margin-top: 6px;
}

.view-button {
  background: transparent;
  color: #72b2f5;
  border: 1px solid rgba(82, 145, 215, 0.25);
  border-radius: 7px;
  padding: 8px 11px;
  font-size: 9px;
  cursor: pointer;
}

/* DIGITAL TWIN PLACEHOLDER */

.twin-placeholder {
  height: 340px;
  margin-top: 20px;
  border-radius: 9px;
  position: relative;
  overflow: hidden;
  background:
    linear-gradient(rgba(54, 105, 155, 0.05) 1px, transparent 1px),
    linear-gradient(90deg, rgba(54, 105, 155, 0.05) 1px, transparent 1px),
    #090e17;
  background-size: 25px 25px;
  display: grid;
  place-items: center;
}

.building-grid {
  width: 270px;
  transform: perspective(700px) rotateX(45deg) rotateZ(-8deg);
}

.building-floor {
  height: 75px;
  margin-bottom: 4px;
  border: 1px solid rgba(83, 163, 230, 0.35);
  background: rgba(41, 91, 136, 0.12);
  padding: 8px;
}

.building-floor>span {
  font-size: 7px;
  color: #6885a4;
}

.rooms {
  display: grid;
  grid-template-columns: repeat(4, 1fr);
  gap: 4px;
  margin-top: 8px;
}

.rooms i {
  height: 40px;
  border: 1px solid rgba(83, 163, 230, 0.25);
  background: rgba(63, 136, 197, 0.1);
}

.twin-overlay {
  position: absolute;
  bottom: 15px;
  left: 15px;
  display: flex;
  align-items: center;
  gap: 8px;
  color: #6e829b;
  font-size: 8px;
  letter-spacing: 1px;
}

/* AI */

.risk-score {
  display: grid;
  place-items: center;
  padding: 32px 0;
}

.score-circle {
  width: 130px;
  height: 130px;
  border-radius: 50%;
  border: 7px solid rgba(73, 192, 132, 0.16);
  box-shadow: inset 0 0 30px rgba(73, 192, 132, 0.04);
  display: grid;
  place-items: center;
  align-content: center;
}

.score-circle strong {
  font-size: 27px;
}

.score-circle span {
  color: #51c58b;
  font-size: 9px;
  letter-spacing: 2px;
  margin-top: 5px;
}

.insight-item {
  display: flex;
  justify-content: space-between;
  border-top: 1px solid rgba(255, 255, 255, 0.05);
  padding: 13px 0;
  font-size: 10px;
}

.insight-item span {
  color: #657184;
}

.insight-item strong {
  font-weight: 500;
  color: #b8c4d5;
}

/* BOTTOM */

.bottom-grid {
  display: grid;
  grid-template-columns: 2fr 1fr;
  gap: 14px;
  margin-top: 14px;
}

.event-row {
  display: flex;
  align-items: center;
  gap: 12px;
  padding: 14px 0;
  border-bottom: 1px solid rgba(255, 255, 255, 0.05);
}

.event-indicator {
  width: 7px;
  height: 7px;
  border-radius: 50%;
  background: #638fc1;
}

.event-indicator.safe {
  background: #4ac98b;
  box-shadow: 0 0 10px rgba(74, 201, 139, 0.5);
}

.event-row div {
  flex: 1;
}

.event-row strong {
  display: block;
  font-size: 10px;
}

.event-row div span {
  color: #5f6b7c;
  font-size: 8px;
  display: block;
  margin-top: 4px;
}

.event-row time {
  color: #566173;
  font-size: 8px;
}

.quick-panel p {
  color: #687486;
  font-size: 10px;
  line-height: 1.7;
  margin: 15px 0 22px;
}

.simulation-button {
  width: 100%;
  border: 1px solid rgba(81, 155, 222, 0.3);
  background: rgba(64, 133, 195, 0.1);
```

#### Lines 441–483

```css
  color: #82bdf1;
  padding: 11px;
  border-radius: 7px;
  cursor: pointer;
  font-size: 10px;
}

/* RESPONSIVE */

@media (max-width: 1000px) {
  .sidebar {
    width: 210px;
  }

  .stats-grid {
    grid-template-columns: repeat(2, 1fr);
  }

  .dashboard-grid,
  .bottom-grid {
    grid-template-columns: 1fr;
  }
}

@media (max-width: 700px) {
  .sidebar {
    display: none;
  }

  .main-content {
    padding: 20px;
  }

  .stats-grid {
    grid-template-columns: 1fr;
  }

  .topbar {
    align-items: flex-start;
    gap: 15px;
    flex-direction: column;
  }
}
```

## 19. Project file inventory

This section lists source/configuration files discovered during generation. Dataset files, model binaries, virtual-environment files, `node_modules`, caches and Git internals are intentionally excluded from the source listing because they can be large or machine-specific.

```text
.gitignore
.ipynb_checkpoints/EVAC_TWIN_Clean-checkpoint.ipynb
.ipynb_checkpoints/EVAC_TWIN_Complete_Project-checkpoint.ipynb
.ipynb_checkpoints/EVAC_TWIN_Connectivity_Demo-checkpoint.ipynb
.vscode/settings.json
backend/.env
backend/0.3
backend/0.5
backend/0.7
backend/0.9
backend/15
backend/20
backend/30
backend/35
backend/38
backend/40
backend/48
backend/app/__init__.py
backend/app/building_layout.py
backend/app/building_structure.py
backend/app/database.py
backend/app/iot_simulator.py
backend/app/iot_simulator_backup.py
backend/app/main.py
backend/app/ml/__init__.py
backend/app/ml/compare_dl_models.py
backend/app/ml/compare_models.py
backend/app/ml/fire_predict.py
backend/app/ml/models/fire_alarm_config.json
backend/app/ml/models/risk_config.json
backend/app/ml/predict.py
backend/app/ml/results/calibration_curve.png
backend/app/ml/results/classification_report.txt
backend/app/ml/results/confusion_matrix.png
backend/app/ml/results/dataset_distribution.png
backend/app/ml/results/precision_recall_curve.png
backend/app/ml/results/risk_classification_report.txt
backend/app/ml/results/risk_confusion_matrix.png
backend/app/ml/results/risk_dataset_distribution.png
backend/app/ml/results/risk_precision_recall_curve.png
backend/app/ml/results/risk_roc_curve.png
backend/app/ml/results/risk_training_loss.png
backend/app/ml/results/risk_training_results.txt
backend/app/ml/results/risk_training_validation_accuracy.png
backend/app/ml/results/roc_curve.png
backend/app/ml/results/training_loss.png
backend/app/ml/results/training_results.txt
backend/app/ml/results/training_validation_accuracy.png
backend/app/ml/train_fire_model.py
backend/app/ml/train_model.py
backend/app/ml/train_risk_model.py
backend/app/ml/zone_risk.py
backend/app/mqtt_subscriber.py
backend/app/risk_engine.py
backend/app/routing/__init__.py
backend/app/routing/evacuation_router.py
backend/app/services/__init__.py
backend/app/services/alert_state.py
backend/app/services/routing_service.py
backend/app/services/sensor_service.py
backend/app/services/sms_service.py
backend/backend_structure.txt
backend/cd
backend/cls
backend/curl
backend/EXIT-A
backend/EXIT-B
backend/EXIT-C
backend/python
backend/requirements.txt
backend/test_routing.py
backend/test_sms.py
backend/type
backend.zip
backup_20261013_105316/Dashboard.jsx.bak
backup_20261013_105316/iot_simulator.py.bak
backup_20261013_105316/mqtt_subscriber.py.bak
backup_20261013_105316/routing_service.py.bak
backup_20261013_105316/run_evactwin.bat.bak
CommandsList.txt
create_evactwin_flow_notebook.py
create_evactwin_notebook.py
create_notebook.py
curl
EVAC_TWIN_Clean.ipynb
EVAC_TWIN_Complete.ipynb
EVAC_TWIN_Complete_Project.ipynb
EVAC_TWIN_Connectivity_Demo.ipynb
frontend/.gitignore
frontend/.oxlintrc.json
frontend/index.html
frontend/package-lock.json
frontend/package.json
frontend/public/favicon.svg
frontend/public/icons.svg
frontend/README.md
frontend/src/App.css
frontend/src/App.jsx
frontend/src/assets/hero.png
frontend/src/assets/react.svg
frontend/src/assets/vite.svg
frontend/src/index.css
frontend/src/main.jsx
frontend/src/pages/AIInsights.css
frontend/src/pages/AIInsights.jsx
frontend/src/pages/Analytics.jsx
frontend/src/pages/Dashboard.css
frontend/src/pages/Dashboard.jsx
frontend/src/pages/Evacuation.css
frontend/src/pages/Evacuation.jsx
frontend/src/pages/EvacuationMap.jsx
frontend/src/pages/Monitoring.css
frontend/src/pages/Monitoring.jsx
frontend/src/pages/Simulation.css
frontend/src/pages/Simulation.jsx
frontend/src/services/api.js
frontend/src/utils/riskEngine.js
frontend/vite.config.js
run_evactwin.bat
```

## 20. Notebook generation record

- Generated from: `C:\EVAC-TWIN`
- Generated at: `2026-11-20T12:58:39`
- Output: `EVAC_TWIN_Execution_Flow.ipynb`
- Original project files: read-only; not modified by this generator.
- Missing source files: marked in their corresponding sections.
- Rebuild after changing source code by running `python create_evactwin_flow_notebook.py` again.

**End of EVAC-TWIN execution-flow notebook.**